# 任务4：L2软边带 + soft-clDice，单项结构损失消融

**这次只新增一组五折训练，不重跑L0/L1/L2。** 对照是你已完成的L2_soft；新增组从同一官方ImageNet初始化重新训练，沿用相同80轮、512输入、模型、EMA、RGB采样/增强计划和原五折，仅增加结构损失。

上轮结果：L2相对同几何L0，在固定0.5的干净Dice提高0.009804；分别校准后仍提高0.007059，但最差折略退。本轮回答：在软边带监督之上，结构损失是否还有独立增量，尤其能否减少细线遗漏，而不牺牲困难折和像素精度。它不是已验证的改进，也不保证私榜涨分。

## 准备

1. 上传本Notebook，挂载原任务4数据，启用GPU。
2. 把**上一轮标签消融Notebook的完整输出**作为Input挂载（可用Kaggle Save Version保留后选择其Output）。需要`ablation_config.json`、`execution_environment.json`及`L2_soft/fold0..4`下的`completed.json`、`history.csv`、`validation_probabilities.npz`。旧模型权重不是本轮对照计算的必需输入。
3. 默认自动查找；有多个副本时设置`BASELINE_RUN`，指向直接含`L2_soft`和`ablation_config.json`的目录。也支持根目录结构相同的完整备份ZIP。
4. **你刚下载的`ablation_analysis_results (1).zip`是轻量分析包，不含概率，不能代替完整输出。** 新模型所需官方初始权重默认从旧输出的`pretrained_cache`复用；缓存缺失时开Internet，或填写`WEIGHTS_PATH`。

## 唯一训练变化

`L = L2_reference + 0.1 × soft-clDice(main prediction, C)`

L2_reference仍是原来的软边带BCE＋干净C Dice，辅头权重不变；新增项只加在主头，系数加在原损失整体归一化之后。骨架监督使用干净核心C，不把JPEG边带当骨架。固定10次软骨架迭代、smooth=1、逐图平均、float32运算，本轮不搜索系数或迭代次数。

软骨架采用[clDice论文](https://arxiv.org/abs/2003.07311)和[作者PyTorch实现](https://github.com/jocpae/clDice/tree/master/cldice_loss/pytorch)中的可微软腐蚀/开运算；这里明确采用逐图平均，以对应原有损失的归约方式。训练的soft-clDice与评价时二值骨架clDice分别报告，二者不是同一个数。

## 比较与限制

- 校验旧OOF概率文件哈希，重算基线指标并核对你已返回的240行L2记录；新模型每折初始权重和每轮RGB计划必须与旧L2一致。
- 固定最终80轮EMA，不按外层验证挑最佳轮次。先看D4、阈值0.5，同时保留无TTA和0.1/0.4；OOF校准网格保持不变，仍只是开发估计。
- 同时报告C/P目标、逐图/逐折、最差折、精确率/召回、细中心线召回和clDice，并保存全部20图OOF误差图。中心线召回不等于已证明连通性改善。
- 不读取公开分数、不自动选赢家、不预测测试集、不生成比赛CSV。旧原尺寸候选继续保留；不能仅因新组clDice上升就替换它。

计算量是一组5×80轮，约为上轮三组训练量的三分之一，另有soft-clDice、推理和保存开销。80轮全部重训，不从旧L2末期权重微调。


## 1. 路径与运行开关


In [ ]:
from pathlib import Path
DATA_DIR = None             # 原任务4比赛数据；自动查找segmentation目录
BASELINE_RUN = None         # 自动查找上一轮完整标签消融输出；也可填目录/完整备份ZIP
OUTPUT_DIR = Path('/kaggle/working/task4_cldice_ablation') if Path('/kaggle/working').exists() else Path.cwd() / 'task4_cldice_results'
RESUME_FROM = None          # 续接本轮：完整structure输出目录/备份；新会话输出须为空
WEIGHTS_PATH = None         # 默认复用旧输出pretrained_cache的官方ImageNet权重，否则下载
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
INSTALL_MISSING_DEPS = True # 仅缺少scikit-image时安装，需要Internet
RUN_TRAINING = True         # False只做预检查；默认执行新增的一组五折训练


## 2. 加载内嵌实现


In [ ]:
import importlib.util
import sys
import subprocess
if importlib.util.find_spec('skimage') is None:
    if not INSTALL_MISSING_DEPS:
        raise RuntimeError('Install scikit-image or enable INSTALL_MISSING_DEPS.')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-image'])
RUNTIME_DIR = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()) / 'task4_structure_runtime'
RUNTIME_DIR.mkdir(parents=True,exist_ok=True)
SOURCES = {'task4_pipeline.py': '"""Task 4: small-data retinal vessel segmentation, audit through submission.\n\nOnly the supplied training images/labels are used. No downloaded weights.\nThe original segmentation_to_csv.py is kept unchanged and checked on export.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\n\n@dataclass\nclass Config:\n    seed: int = 42\n    epochs: int = 80\n    size: int = 512\n    patch: int = 256\n    patches_per_epoch: int = 128\n    batch_size: int = 8\n    base_channels: int = 24\n    lr: float = 0.001\n    weight_decay: float = 0.0001\n    folds: int = 5\n    fold: int = 0\n    threads: int = 4\n    amp: bool = True\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef file_sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef find_data(data_dir=None):\n    if data_dir:\n        candidates = [Path(data_dir)]\n    else:\n        base = Path(__file__).resolve().parent if \'__file__\' in globals() else Path.cwd()\n        candidates = [base / \'2026task-4/segmentation\', Path.cwd() / \'2026task-4/segmentation\',\n                      Path.cwd() / \'task4/2026task-4/segmentation\']\n        if Path(\'/kaggle/input\').exists():\n            candidates += [p.parent for p in Path(\'/kaggle/input\').rglob(\'sample_submission.csv\')]\n    found = {p.resolve() for p in candidates if (p / \'train/label\').is_dir()\n             and (p / \'test/image\').is_dir() and (p / \'sample_submission.csv\').is_file()}\n    if len(found) != 1:\n        raise ValueError(f\'Set data_dir to the segmentation folder. Found: {sorted(map(str, found))}\')\n    return found.pop()\n\n\ndef list_images(directory):\n    files = sorted(Path(directory).glob(\'*.jpg\'), key=lambda p: int(p.stem))\n    if not files:\n        raise ValueError(f\'No jpg images: {directory}\')\n    return files\n\n\ndef strict_label(path, size=512):\n    # JPEG labels contain ringing; >0 would thicken the foreground substantially.\n    mask = (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.uint8)\n    return cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)\n\n\ndef converter_label(path):\n    # Exact target IF hidden ground truth was processed by the supplied converter.\n    # This is a diagnostic proxy, not an assertion about the private scoring code.\n    return np.asarray(Image.open(path).convert(\'L\').resize((512, 512), Image.Resampling.BILINEAR)) > 0\n\n\ndef make_features(path, size=512):\n    rgb = np.asarray(Image.open(path).convert(\'RGB\'))\n    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)\n    green = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(rgb[:, :, 1])\n    return np.concatenate([rgb, green[:, :, None]], axis=2)\n\n\ndef audit(root, output, cfg):\n    output.mkdir(parents=True, exist_ok=True)\n    train, test = list_images(root / \'train/image\'), list_images(root / \'test/image\')\n    labels = {p.stem: p for p in list_images(root / \'train/label\')}\n    if len(train) != 20 or len(test) != 20 or set(labels) != {p.stem for p in train}:\n        raise ValueError(\'Expected 20 train image/label pairs and 20 test images.\')\n    with (root / \'sample_submission.csv\').open(newline=\'\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != [\'Id\', \'Predicted\']:\n            raise ValueError(f\'Unexpected template columns: {reader.fieldnames}\')\n        ids = [r[\'Id\'] for r in reader]\n    if len(ids) != len(set(ids)) or set(ids) != {p.stem for p in test}:\n        raise ValueError(\'Template IDs do not match test images.\')\n    rows, groups, test_hashes = [], {}, set()\n    for split, paths in [(\'train\', train), (\'test\', test)]:\n        for p in paths:\n            im = Image.open(p).convert(\'RGB\')\n            digest = hashlib.sha256(np.asarray(im).tobytes()).hexdigest()\n            row = dict(split=split, id=p.stem, width=im.width, height=im.height,\n                       image_sha256=file_sha(p), pixel_sha256=digest)\n            if split == \'train\':\n                if Image.open(labels[p.stem]).size != im.size:\n                    raise ValueError(f\'Image/label size mismatch: {p.stem}\')\n                row.update(label_sha256=file_sha(labels[p.stem]),\n                           vessel_fraction=float(strict_label(labels[p.stem]).mean()),\n                           converter_fraction=float(converter_label(labels[p.stem]).mean()))\n                groups.setdefault(digest, []).append(p.stem)\n            else:\n                test_hashes.add(digest)\n            rows.append(row)\n    if set(groups) & test_hashes:\n        raise ValueError(\'Exact decoded train/test duplicate detected; inspect before training.\')\n    ordered = sorted(groups)\n    random.Random(cfg.seed).shuffle(ordered)\n    assignments = {}\n    sizes = [0] * cfg.folds\n    for digest in ordered:\n        fold = min(range(cfg.folds), key=lambda k: sizes[k])\n        for image_id in groups[digest]:\n            assignments[image_id] = fold\n        sizes[fold] += len(groups[digest])\n    for row in rows:\n        row[\'fold\'] = assignments.get(row[\'id\'], -1) if row[\'split\'] == \'train\' else -1\n    fields = sorted({key for row in rows for key in row})\n    with (output / \'manifest.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    manifest_fingerprint = hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n    summary = dict(train_images=len(train), test_images=len(test), unique_train_groups=len(groups),\n                   folds=sizes, validation_ids=[p.stem for p in train if assignments[p.stem] == cfg.fold],\n                   decoded_train_test_overlap=0, patient_ids_available=False,\n                   data_fingerprint=manifest_fingerprint,\n                   observed_label_polarity=\'white vessels / black background\',\n                   screenshot_label_polarity=\'black vessels / white background\',\n                   converter=dict(size=[512, 512], resize=\'PIL bilinear\', foreground=\'nonzero\',\n                                  flatten=\'row-major C order\', start_index=1,\n                                  source_sha256=file_sha(root / \'segmentation_to_csv.py\')),\n                   strict_label=\'native grayscale >127, then nearest resize\',\n                   mean_clean_foreground=float(np.mean([r[\'vessel_fraction\'] for r in rows if r[\'split\']==\'train\'])),\n                   mean_converter_foreground=float(np.mean([r[\'converter_fraction\'] for r in rows if r[\'split\']==\'train\'])))\n    save_json(output / \'audit_summary.json\', summary)\n    return train, test, labels, assignments, ids, summary\n\n\ndef paired_d4(image, mask, rotation, flip):\n    image, mask = np.rot90(image, rotation), np.rot90(mask, rotation)\n    if flip:\n        image, mask = np.flip(image, 1), np.flip(mask, 1)\n    return image.copy(), mask.copy()\n\n\nclass PatchDataset(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.cfg = cfg\n        self.images = [make_features(p, cfg.size) for p in paths]\n        self.masks = [strict_label(labels[p.stem], cfg.size).astype(np.float32) for p in paths]\n        self.positive = [np.argwhere(m > 0) for m in self.masks]\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        idx = random.randrange(len(self.images))\n        image, mask = self.images[idx], self.masks[idx]\n        size, patch = self.cfg.size, self.cfg.patch\n        crop = random.randint(patch, min(size, int(patch * 1.25)))\n        if random.random() < 0.3 and len(self.positive[idx]):\n            y, x = self.positive[idx][random.randrange(len(self.positive[idx]))]\n            top, left = int(np.clip(y - crop//2, 0, size-crop)), int(np.clip(x-crop//2, 0, size-crop))\n        else:\n            top, left = random.randint(0, size-crop), random.randint(0, size-crop)\n        image = cv2.resize(image[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = paired_d4(image, mask, random.randrange(4), random.random() < 0.5)\n        image = image.astype(np.float32) / 255.0\n        image = np.clip(image * random.uniform(0.9, 1.1) + random.uniform(-0.04, 0.04), 0, 1)\n        image = image ** random.uniform(0.85, 1.15)\n        return torch.from_numpy(image.transpose(2, 0, 1).copy()) * 2 - 1, torch.from_numpy(mask[None].copy())\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),\n                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())\n\n\nclass VesselUNet(nn.Module):\n    def __init__(self, base=24):\n        super().__init__()\n        ch = [base * (2**i) for i in range(5)]\n        self.enc = nn.ModuleList([ConvBlock(4 if i == 0 else ch[i-1], ch[i]) for i in range(5)])\n        self.dec = nn.ModuleList([ConvBlock(ch[i+1]+ch[i], ch[i]) for i in range(3, -1, -1)])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(x if i == 0 else F.max_pool2d(x, 2))\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = F.interpolate(x, size=skip.shape[-2:], mode=\'bilinear\', align_corners=False)\n            x = block(torch.cat([x, skip], 1))\n        return self.head(x)\n\n\ndef segmentation_loss(logits, target):\n    bce = F.binary_cross_entropy_with_logits(logits.float(), target.float())\n    prob = logits.float().sigmoid()\n    dice = (2*(prob*target).sum((1,2,3))+1) / (prob.sum((1,2,3))+target.sum((1,2,3))+1)\n    return 0.5*bce + 0.5*(1-dice.mean())\n\n\ndef dice_score(pred, target):\n    pred, target = np.asarray(pred, bool), np.asarray(target, bool)\n    return float((2*np.logical_and(pred, target).sum()+1e-7) / (pred.sum()+target.sum()+1e-7))\n\n\n@torch.inference_mode()\ndef predict(model, path, cfg, device):\n    model.eval()\n    image = make_features(path, cfg.size)\n    step = cfg.patch // 2\n    starts = list(range(0, cfg.size-cfg.patch+1, step))\n    if starts[-1] != cfg.size-cfg.patch:\n        starts.append(cfg.size-cfg.patch)\n    coords = [(y,x) for y in starts for x in starts]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch), np.hanning(cfg.patch)), 0.05).astype(np.float32)\n    sums, weights = np.zeros((cfg.size, cfg.size), np.float32), np.zeros((cfg.size, cfg.size), np.float32)\n    for start in range(0, len(coords), cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        patches = np.stack([image[y:y+cfg.patch, x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(patches.copy()).to(device).float()/127.5-1\n        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n            probs = model(tensor).float().sigmoid()[:,0].cpu().numpy()\n        for (y,x), prob in zip(positions, probs):\n            sums[y:y+cfg.patch,x:x+cfg.patch] += prob * weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    return sums/weights\n\n\ndef atomic_checkpoint(path, value):\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(value, tmp)\n    os.replace(tmp, path)\n\n\ndef rng_state():\n    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'].cpu())\n    if torch.cuda.is_available() and state[\'cuda\'] is not None:\n        torch.cuda.set_rng_state_all([s.cpu() for s in state[\'cuda\']])\n\n\ndef train_stage(paths, val_paths, labels, cfg, folder, device, epochs):\n    folder.mkdir(parents=True, exist_ok=True)\n    seed_all(cfg.seed)\n    dataset = PatchDataset(paths, labels, cfg)\n    loader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=0, shuffle=False,\n                        pin_memory=device.type==\'cuda\')\n    model = VesselUNet(cfg.base_channels).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n    # Full retraining retains the original 80-epoch cosine horizon.\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr*0.02)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg.amp and device.type==\'cuda\')\n    best, history, first = -1.0, [], 1\n    last = folder / \'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last, map_location=\'cpu\', weights_only=False)\n        if ckpt[\'train_ids\'] != [p.stem for p in paths] or ckpt[\'target_epochs\'] != epochs:\n            raise ValueError(\'Stage mismatch. Use a new output directory.\')\n        model.load_state_dict(ckpt[\'model\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\'])\n        scheduler.load_state_dict(ckpt[\'scheduler\'])\n        scaler.load_state_dict(ckpt[\'scaler\'])\n        first, best, history = ckpt[\'epoch\']+1, ckpt[\'best\'], ckpt[\'history\']\n        restore_rng(ckpt[\'rng\'])\n        print(f\'{folder.name}: resume at epoch {first}/{epochs}\', flush=True)\n    targets = {p.stem: strict_label(labels[p.stem], cfg.size) for p in val_paths}\n    for epoch in range(first, epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        lr = optimizer.param_groups[0][\'lr\']\n        for image, mask in loader:\n            image, mask = image.to(device, non_blocking=True), mask.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n                loss = segmentation_loss(model(image), mask)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(optimizer)\n            scaler.update()\n            losses.append(float(loss.detach()))\n        scheduler.step()\n        score = float(np.mean([dice_score(predict(model,p,cfg,device) >= 0.5, targets[p.stem]) for p in val_paths])) if val_paths else None\n        improved = score is not None and score > best\n        if improved:\n            best = score\n            atomic_checkpoint(folder/\'best.pt\', dict(model=model.state_dict(), epoch=epoch, dice_05=score))\n        row = dict(epoch=epoch, loss=float(np.mean(losses)), dice_05=score, lr=lr,\n                   seconds=round(time.perf_counter()-begin, 3))\n        history.append(row)\n        atomic_checkpoint(last, dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                          scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch,\n                          best=best, history=history, rng=rng_state(), train_ids=[p.stem for p in paths], target_epochs=epochs))\n        with (folder/\'history.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n            writer = csv.DictWriter(f, list(row))\n            writer.writeheader()\n            writer.writerows(history)\n        print(f\'{folder.name} {epoch:03d}/{epochs} loss={row["loss"]:.4f}\' +\n              (f\' val_dice@0.5={score:.5f} best={best:.5f}\' if score is not None else \'\') +\n              f\' seconds={row["seconds"]}\', flush=True)\n    selected = torch.load(folder/(\'best.pt\' if val_paths else \'last.pt\'), map_location=\'cpu\', weights_only=False)\n    model.load_state_dict(selected[\'model\'])\n    return model, int(selected[\'epoch\'])\n\n\ndef rle_encode(mask):\n    flat = np.asarray(mask, dtype=np.uint8).ravel(order=\'C\')\n    changes = np.flatnonzero(np.diff(np.r_[np.uint8(0), flat, np.uint8(0)])) + 1\n    changes[1::2] -= changes[::2]\n    return \' \'.join(map(str, changes))\n\n\ndef rle_decode(encoded, shape=(512,512)):\n    values = np.array([int(s) for s in encoded.split()], dtype=np.int64)\n    if len(values)%2:\n        raise ValueError(\'Odd RLE length\')\n    mask = np.zeros(int(np.prod(shape)), dtype=bool)\n    previous_end = 0\n    for start, length in values.reshape(-1,2):\n        if start < 1 or length < 1 or start <= previous_end or start+length-1 > mask.size:\n            raise ValueError(\'RLE bounds/order/overlap violation\')\n        mask[start-1:start-1+length] = True\n        previous_end = start+length-1\n    return mask.reshape(shape)\n\n\ndef load_converter(root):\n    spec = importlib.util.spec_from_file_location(\'task4_official_converter\', root/\'segmentation_to_csv.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef preview(paths, probs, labels, threshold, output):\n    tile = 256\n    canvas = Image.new(\'RGB\', (tile*3, (tile+24)*min(4,len(paths))), \'white\')\n    draw = ImageDraw.Draw(canvas)\n    for row,p in enumerate(paths[:4]):\n        rgb = Image.open(p).convert(\'RGB\').resize((tile,tile))\n        target = strict_label(labels[p.stem],tile) if labels else None\n        pred = cv2.resize(probs[p.stem],(tile,tile),interpolation=cv2.INTER_LINEAR) >= threshold\n        y = row*(tile+24)\n        canvas.paste(rgb,(0,y+24))\n        draw.text((4,y+4), f\'Image {p.stem}\',fill=\'black\')\n        if target is not None:\n            canvas.paste(Image.fromarray((target*255).astype(np.uint8)).convert(\'RGB\'),(tile,y+24))\n            draw.text((tile+4,y+4),\'Label: white vessels\',fill=\'black\')\n        else:\n            overlay = np.asarray(rgb).copy()\n            overlay[pred] = (0.4*overlay[pred]+0.6*np.array([0,255,100])).astype(np.uint8)\n            canvas.paste(Image.fromarray(overlay),(tile,y+24))\n            draw.text((tile+4,y+4),\'Vessel overlay\',fill=\'black\')\n        canvas.paste(Image.fromarray((pred*255).astype(np.uint8)).convert(\'RGB\'),(tile*2,y+24))\n        draw.text((tile*2+4,y+4),f\'Prediction t={threshold:.3f}\',fill=\'black\')\n    canvas.save(output)\n\n\ndef validation_report(model, paths, labels, cfg, device, output, epoch):\n    probs = {p.stem:predict(model,p,cfg,device) for p in paths}\n    np.savez_compressed(output/\'validation_probabilities.npz\', **probs)\n    thresholds = [round(float(t),3) for t in np.arange(0.2,0.801,0.025)]\n    rows = []\n    for threshold in thresholds:\n        strict = [dice_score(probs[p.stem]>=threshold, strict_label(labels[p.stem],cfg.size)) for p in paths]\n        proxy = [dice_score(probs[p.stem]>=threshold, converter_label(labels[p.stem])) for p in paths]\n        rows.append(dict(threshold=threshold, mean_vessel_dice=float(np.mean(strict)),\n                         converter_target_proxy_dice=float(np.mean(proxy))))\n    best = max(rows,key=lambda r:(r[\'mean_vessel_dice\'],-abs(r[\'threshold\']-0.5)))\n    threshold = best[\'threshold\']\n    with (output/\'threshold_sweep.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    per_image=[]\n    for p in paths:\n        pred=probs[p.stem]>=threshold\n        gt=strict_label(labels[p.stem],cfg.size)\n        native=np.asarray(Image.open(labels[p.stem]).convert(\'L\'))>127\n        native_pred=cv2.resize(probs[p.stem],(native.shape[1],native.shape[0]),interpolation=cv2.INTER_LINEAR)>=threshold\n        per_image.append(dict(id=p.stem,dice_05=dice_score(probs[p.stem]>=0.5,gt),dice=dice_score(pred,gt),\n                              native_dice=dice_score(native_pred,native),\n                              converter_target_proxy_dice=dice_score(pred,converter_label(labels[p.stem])),\n                              predicted_fraction=float(pred.mean()),target_fraction=float(gt.mean())))\n    with (output/\'validation_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(per_image[0]));writer.writeheader();writer.writerows(per_image)\n    report=dict(best_epoch=epoch,threshold=threshold,validation_ids=[p.stem for p in paths],\n                mean_vessel_dice_05=float(np.mean([r[\'dice_05\'] for r in per_image])),\n                mean_vessel_dice=best[\'mean_vessel_dice\'],\n                mean_native_vessel_dice=float(np.mean([r[\'native_dice\'] for r in per_image])),\n                converter_target_proxy_dice=best[\'converter_target_proxy_dice\'],\n                evaluation=\'single held-out fold (4 images), not full five-fold OOF\',\n                limitation=\'Epoch and threshold chosen on the same small validation set; development estimate only.\',\n                private_ground_truth_encoding_verified=False)\n    save_json(output/\'validation_metrics.json\',report)\n    preview(paths,probs,labels,threshold,output/\'validation_preview.png\')\n    return report\n\n\ndef export_submission(model, test, template_ids, cfg, device, root, output, threshold, epoch):\n    white_dir=output/\'masks_512_white_vessels\'\n    black_dir=output/\'masks_native_black_vessels\'\n    white_dir.mkdir(exist_ok=True);black_dir.mkdir(exist_ok=True)\n    probs={p.stem:predict(model,p,cfg,device) for p in test}\n    paths={p.stem:p for p in test}\n    white_paths=[]\n    encoded=[]\n    for image_id in template_ids:\n        prob=probs[image_id]\n        mask=prob>=threshold\n        white_path=white_dir/f\'{image_id}.png\'\n        Image.fromarray((mask*255).astype(np.uint8)).save(white_path)\n        white_paths.append(str(white_path))\n        width,height=Image.open(paths[image_id]).size\n        native=cv2.resize(prob,(width,height),interpolation=cv2.INTER_LINEAR)>=threshold\n        Image.fromarray(np.where(native,0,255).astype(np.uint8)).save(black_dir/f\'{image_id}.png\')\n        value=rle_encode(mask)\n        if not np.array_equal(rle_decode(value),mask):\n            raise AssertionError(\'RLE roundtrip mismatch\')\n        encoded.append(value)\n    # Execute the supplied conversion function on the actual PNG predictions.\n    official=load_converter(root).turn_to_str(white_paths)\n    if encoded != official:\n        raise AssertionError(\'Vectorized encoding differs from the official converter\')\n    target=output/\'submission.csv\'\n    with target.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(template_ids,official))\n    with target.open(newline=\'\',encoding=\'utf-8\') as f:\n        rows=list(csv.DictReader(f))\n    if [r[\'Id\'] for r in rows]!=template_ids or len(rows)!=20:\n        raise AssertionError(\'Saved submission IDs/count mismatch\')\n    for row in rows:\n        rle_decode(row[\'Predicted\'])\n    np.savez_compressed(output/\'test_probabilities.npz\',**probs)\n    preview(test,probs,None,threshold,output/\'test_preview.png\')\n    save_json(output/\'submission_provenance.json\',dict(\n        csv_sha256=file_sha(target),model_sha256=file_sha(output/\'fulltrain/last.pt\'),\n        train_images=20,epochs=epoch,threshold=threshold,external_training_data=False,pretrained=False,\n        csv_polarity=\'white vessels, following observed training labels and supplied converter\',\n        image_output_polarity=\'masks_native_black_vessels: vessel=0, background=255\',\n        official_converter_exact_match=True,rle_roundtrip_all_20=True,template_order_preserved=True,\n        submitted_to_competition=False))\n    print(f\'Submission saved: {target}\',flush=True)\n    return target\n\n\ndef run(phase=\'full\', data_dir=None, output_dir=None, config=None):\n    cfg=config or Config()\n    if cfg.size!=512 or cfg.patch<16 or cfg.patch%16 or cfg.patch>cfg.size or cfg.base_channels<8 or cfg.base_channels%8:\n        raise ValueError(\'Require size=512, patch divisible by 16 <=512, base_channels divisible by 8.\')\n    if cfg.epochs<1 or cfg.patches_per_epoch<1 or cfg.batch_size<1 or cfg.threads<1 or not 2<=cfg.folds<=20 or not 0<=cfg.fold<cfg.folds:\n        raise ValueError(\'Invalid training configuration\')\n    torch.set_num_threads(cfg.threads)\n    cv2.setNumThreads(0)\n    root=find_data(data_dir)\n    output=Path(output_dir or \'task4_baseline_results\').resolve()\n    output.mkdir(parents=True,exist_ok=True)\n    source_sha=file_sha(Path(__file__)) if \'__file__\' in globals() else PIPELINE_SHA256\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and (previous[\'config\']!=asdict(cfg) or previous[\'pipeline_sha256\']!=source_sha):\n        raise ValueError(\'Existing output belongs to another config/source. Use a new directory.\')\n    train,test,labels,assignment,ids,summary=audit(root,output,cfg)\n    identity=dict(config=asdict(cfg),data_fingerprint=summary[\'data_fingerprint\'],\n                  pipeline_sha256=source_sha)\n    if previous and previous!=identity:\n        raise ValueError(\'Existing output belongs to another config/data/source. Use a new directory.\')\n    save_json(config_path,identity)\n    if phase==\'audit\':\n        print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n        return output\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(f\'Device: {device}; train=16, validation=4; no pretrained weights\',flush=True)\n    val=[p for p in train if assignment[p.stem]==cfg.fold]\n    development=[p for p in train if assignment[p.stem]!=cfg.fold]\n    # A single writer protects checkpoints from two simultaneous runs.\n    lock=output/\'training.lock\'\n    try:\n        descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    except FileExistsError:\n        raise RuntimeError(f\'Run lock exists: {lock}. Check its PID before removing a stale lock.\')\n    os.write(descriptor,str(os.getpid()).encode());os.close(descriptor)\n    try:\n        model,epoch=train_stage(development,val,labels,cfg,output/\'holdout\',device,cfg.epochs)\n        metrics=validation_report(model,val,labels,cfg,device,output,epoch)\n        print(json.dumps(metrics,ensure_ascii=False,indent=2),flush=True)\n        if phase==\'full\':\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n            model,full_epoch=train_stage(train,[],labels,cfg,output/\'fulltrain\',device,epoch)\n            export_submission(model,test,ids,cfg,device,root,output,metrics[\'threshold\'],full_epoch)\n    finally:\n        lock.unlink(missing_ok=True)\n    return output\n\n\ndef cli():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--phase\',choices=[\'audit\',\'train\',\'full\'],default=\'full\')\n    parser.add_argument(\'--data-dir\')\n    parser.add_argument(\'--output-dir\',default=\'task4_baseline_results\')\n    for key,value in asdict(Config()).items():\n        if isinstance(value,bool):continue\n        parser.add_argument(\'--\'+key.replace(\'_\',\'-\'),type=type(value),default=value)\n    args=vars(parser.parse_args())\n    phase,data_dir,output=args.pop(\'phase\'),args.pop(\'data_dir\'),args.pop(\'output_dir\')\n    run(phase,data_dir,output,Config(**args))\n\n\nif __name__==\'__main__\':\n    cli()\n', 'task4_kaggle_v2.py': '"""Fixed-epoch five-fold ResNet34 U-Net; native pixels, EMA, paired D4 evaluation.\n\nThe notebook embeds this file AND task4_pipeline.py, so no extra source upload is needed.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport socket\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models import ResNet34_Weights, resnet34\n\nimport task4_pipeline as base\n\n\n@dataclass\nclass ConfigV2:\n    seed: int = 42\n    folds: int = 5\n    fold: int = 0                 # audit helper only; run_cv uses all requested folds\n    epochs: int = 80\n    patch: int = 256\n    patches_per_epoch: int = 256\n    batch_size: int = 8\n    encoder_lr: float = 0.0001\n    decoder_lr: float = 0.0007\n    weight_decay: float = 0.0001\n    warmup_epochs: int = 3\n    ema_decay: float = 0.99\n    monitor_every: int = 10\n    threads: int = 4\n    amp: bool = True\n    pretrained: bool = True\n    tta_gain_required: float = 0.001\n\n\ndef audit_v2(root, output, cfg):\n    """Freeze the original ID folds; JPEG decoder changes must not reshuffle folds."""\n    train,test,labels,_,ids,summary=base.audit(root,output,cfg)\n    frozen=[[\'2\',\'6\',\'9\',\'10\'],[\'3\',\'14\',\'17\',\'20\'],[\'1\',\'4\',\'5\',\'11\'],\n            [\'15\',\'16\',\'18\',\'19\'],[\'7\',\'8\',\'12\',\'13\']]\n    assignment={i:fold for fold,group in enumerate(frozen) for i in group}\n    if set(assignment)!={p.stem for p in train}:\n        raise ValueError(\'This frozen task-4 split expects train IDs 1 through 20.\')\n    manifest=output/\'manifest.csv\'\n    with manifest.open(newline=\'\',encoding=\'utf-8\') as f:\n        reader=csv.DictReader(f);fields=reader.fieldnames;rows=list(reader)\n    groups={}\n    for row in rows:\n        if row[\'split\']==\'train\':\n            row[\'fold\']=assignment[row[\'id\']]\n            groups.setdefault(row[\'pixel_sha256\'],set()).add(row[\'fold\'])\n    if any(len(folds)>1 for folds in groups.values()):\n        raise ValueError(\'A decoded duplicate crosses frozen folds; inspect dataset before training.\')\n    with manifest.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(rows)\n    # Raw file hashes are stable across JPEG decoder/library versions, unlike pixel hashes.\n    stable=[{k:row[k] for k in [\'split\',\'id\',\'width\',\'height\',\'image_sha256\',\'label_sha256\',\'fold\']}\n            for row in rows]\n    summary.update(data_fingerprint=hashlib.sha256(json.dumps(stable,sort_keys=True).encode()).hexdigest(),\n                   validation_ids=frozen[0],folds=[4]*5,\n                   split_protocol=\'frozen original seed42 ID folds; training seed may vary\')\n    base.save_json(output/\'audit_summary.json\',summary)\n    return train,test,labels,assignment,ids,summary\n\n\ndef read_rgb(path):\n    return np.asarray(Image.open(path).convert(\'RGB\')).copy()\n\n\ndef read_mask(path):\n    return (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.float32)\n\n\ndef resize_probability(prob, shape=(512, 512)):\n    # PIL receives a float image: interpolate probabilities, then threshold once.\n    h, w = shape\n    return np.clip(np.asarray(Image.fromarray(np.asarray(prob, np.float32)).resize(\n        (w, h), Image.Resampling.BILINEAR)), 0, 1).copy()\n\n\nclass NativePatches(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.images = [read_rgb(p) for p in paths]\n        self.masks = [read_mask(labels[p.stem]) for p in paths]\n        self.positives = [np.argwhere(m > 0) for m in self.masks]\n        self.cfg = cfg\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        i = random.randrange(len(self.images))\n        image, mask = self.images[i], self.masks[i]\n        h, w = mask.shape\n        patch = self.cfg.patch\n        crop = random.randint(int(patch*0.9), min(int(patch*1.1), h, w))\n        if random.random() < 0.3 and len(self.positives[i]):\n            y, x = self.positives[i][random.randrange(len(self.positives[i]))]\n            top, left = int(np.clip(y-crop//2, 0, h-crop)), int(np.clip(x-crop//2, 0, w-crop))\n        else:\n            top, left = random.randint(0, h-crop), random.randint(0, w-crop)\n        image = cv2.resize(image[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = base.paired_d4(image,mask,random.randrange(4),random.random()<0.5)\n        image = image.astype(np.float32)/255\n        image = np.clip((image-0.5)*random.uniform(0.85,1.15)+0.5+random.uniform(-0.04,0.04),0,1)\n        image = image**random.uniform(0.85,1.15)\n        image = np.clip(image*np.array([random.uniform(0.95,1.05) for _ in range(3)],np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()), torch.from_numpy(mask[None].copy())\n\n\nclass ResNet34UNet(nn.Module):\n    def __init__(self, initial_state=None):\n        super().__init__()\n        self.encoder = resnet34(weights=None)\n        if initial_state is not None:\n            self.encoder.load_state_dict(initial_state,strict=True)\n        self.encoder.fc = nn.Identity()\n        self.shallow = base.ConvBlock(3,32)\n        self.up4 = base.ConvBlock(512+256,256)\n        self.up3 = base.ConvBlock(256+128,128)\n        self.up2 = base.ConvBlock(128+64,64)\n        self.up1 = base.ConvBlock(64+64,48)\n        self.up0 = base.ConvBlock(48+32,32)\n        self.head = nn.Conv2d(32,1,1)\n        self.aux2 = nn.Conv2d(48,1,1)\n        self.aux4 = nn.Conv2d(64,1,1)\n        self.register_buffer(\'mean\',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))\n        self.register_buffer(\'std\',torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Small batches: keep ImageNet running statistics; affine parameters still train.\n        for module in self.encoder.modules():\n            if isinstance(module,nn.BatchNorm2d):\n                module.eval()\n        return self\n\n    @staticmethod\n    def join(x, skip, block):\n        x = F.interpolate(x,size=skip.shape[-2:],mode=\'bilinear\',align_corners=False)\n        return block(torch.cat([x,skip],1))\n\n    def forward(self, image):\n        x = (image-self.mean)/self.std\n        s0 = self.shallow(x)\n        s1 = self.encoder.relu(self.encoder.bn1(self.encoder.conv1(x)))\n        s2 = self.encoder.layer1(self.encoder.maxpool(s1))\n        s3 = self.encoder.layer2(s2)\n        s4 = self.encoder.layer3(s3)\n        z = self.encoder.layer4(s4)\n        z = self.join(z,s4,self.up4)\n        z = self.join(z,s3,self.up3)\n        z4 = self.join(z,s2,self.up2)\n        z2 = self.join(z4,s1,self.up1)\n        z = self.join(z2,s0,self.up0)\n        return self.head(z),self.aux2(z2),self.aux4(z4)\n\n\ndef objective(outputs, target):\n    losses = []\n    for logits in outputs:\n        truth = target if logits.shape[-2:]==target.shape[-2:] else F.interpolate(\n            target,size=logits.shape[-2:],mode=\'area\')\n        losses.append(base.segmentation_loss(logits,truth))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\n@torch.no_grad()\ndef update_ema(ema, model, decay):\n    for target, source in zip(ema.parameters(),model.parameters()):\n        target.lerp_(source,1-decay)\n    for target, source in zip(ema.buffers(),model.buffers()):\n        target.copy_(source)\n\n\ndef transform_d4(x, rotation, flip):\n    x = torch.rot90(x,rotation,(-2,-1))\n    return torch.flip(x,(-1,)) if flip else x\n\n\ndef inverse_d4(x, rotation, flip):\n    if flip:\n        x = torch.flip(x,(-1,))\n    return torch.rot90(x,-rotation,(-2,-1))\n\n\ndef window_starts(length, patch, stride):\n    if length < patch:\n        raise ValueError(\'Pad before calculating window starts\')\n    starts = list(range(0,length-patch+1,stride))\n    if starts[-1]!=length-patch:\n        starts.append(length-patch)\n    return starts\n\n\n@torch.inference_mode()\ndef predict_pair(model, image, cfg, device, with_tta=True):\n    """Patchwise D4; return native probabilities for none and D4 using identical windows."""\n    model.eval()\n    if isinstance(image,(str,Path)):\n        image = read_rgb(image)\n    h,w = image.shape[:2]\n    pad_h,pad_w = max(0,cfg.patch-h),max(0,cfg.patch-w)\n    image = cv2.copyMakeBorder(image,0,pad_h,0,pad_w,cv2.BORDER_REFLECT_101)\n    height,width = image.shape[:2]\n    coords = [(y,x) for y in window_starts(height,cfg.patch,cfg.patch//2)\n              for x in window_starts(width,cfg.patch,cfg.patch//2)]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch),np.hanning(cfg.patch)),0.05).astype(np.float32)\n    accum = np.zeros((2,height,width),np.float32)\n    weights = np.zeros((height,width),np.float32)\n    transforms = [(0,False)] + ([(r,f) for r in range(4) for f in [False,True] if (r,f)!=(0,False)] if with_tta else [])\n    for start in range(0,len(coords),cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        batch = np.stack([image[y:y+cfg.patch,x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(batch.copy()).to(device).float()/255\n        plain,total = None,None\n        for rotation,flip in transforms:\n            transformed = transform_d4(tensor,rotation,flip)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                prediction = model(transformed)[0].float().sigmoid()\n            prediction = inverse_d4(prediction,rotation,flip)\n            if plain is None:\n                plain = prediction\n                total = prediction.clone()\n            else:\n                total.add_(prediction)\n        pair = torch.stack([plain,total/len(transforms)])[:,:,0].cpu().numpy()\n        for i,(y,x) in enumerate(positions):\n            accum[:,y:y+cfg.patch,x:x+cfg.patch] += pair[:,i]*weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    result = accum/weights[None]\n    if not np.isfinite(result).all():\n        raise FloatingPointError(\'Nonfinite inference output\')\n    return {\'none\':result[0,:h,:w].copy(),\'d4\':result[1,:h,:w].copy()}\n\n\ndef resolve_weights(cfg, output, weights_path=None):\n    if not cfg.pretrained:\n        return None,dict(source=\'random initialization; debug/ablation only\',sha256=None)\n    enum = ResNet34_Weights.IMAGENET1K_V1\n    target = Path(weights_path) if weights_path else output/\'pretrained_cache\'/Path(enum.url).name\n    if not target.is_file():\n        if weights_path:\n            raise FileNotFoundError(target)\n        target.parent.mkdir(parents=True,exist_ok=True)\n        print(\'Downloading official ImageNet encoder weights:\',enum.url,flush=True)\n        torch.hub.download_url_to_file(enum.url,str(target),hash_prefix=\'b627a593\',progress=True)\n    digest = base.file_sha(target)\n    if not digest.startswith(\'b627a593\'):\n        raise ValueError(\'Expected official torchvision resnet34-b627a593.pth, not task-finetuned weights.\')\n    state = torch.load(target,map_location=\'cpu\',weights_only=True)\n    probe = resnet34(weights=None)\n    probe.load_state_dict(state,strict=True)\n    del probe\n    cached=output/\'pretrained_cache\'/Path(enum.url).name\n    if target.resolve()!=cached.resolve():\n        cached.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copyfile(target,cached)\n    return state,dict(source=enum.url,sha256=digest,supervision=\'generic ImageNet classification, not vessel segmentation\')\n\n\ndef set_lrs(optimizer, step, steps_per_epoch, cfg):\n    total = cfg.epochs*steps_per_epoch\n    warm = min(cfg.warmup_epochs*steps_per_epoch,max(0,total-1))\n    if warm and step<warm:\n        factor = (step+1)/warm\n    else:\n        progress = (step-warm)/max(1,total-warm-1)\n        factor = 0.02+0.98*0.5*(1+math.cos(math.pi*min(1,progress)))\n    for group,initial in zip(optimizer.param_groups,[cfg.encoder_lr,cfg.decoder_lr]):\n        group[\'lr\'] = initial*factor\n\n\ndef train_fold(paths, val_paths, labels, cfg, fold, folder, initial_state, device):\n    folder.mkdir(parents=True,exist_ok=True)\n    base.seed_all(cfg.seed+1009*fold)\n    dataset = NativePatches(paths,labels,cfg)\n    loader = DataLoader(dataset,batch_size=cfg.batch_size,num_workers=0,shuffle=False,\n                        pin_memory=device.type==\'cuda\',drop_last=False)\n    model = ResNet34UNet(initial_state).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    encoder = list(model.encoder.parameters())\n    decoder = [p for name,p in model.named_parameters() if not name.startswith(\'encoder.\')]\n    optimizer = torch.optim.AdamW([{\'params\':encoder,\'lr\':cfg.encoder_lr},\n                                 {\'params\':decoder,\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step = [],1,0\n    train_ids,val_ids = [p.stem for p in paths],[p.stem for p in val_paths]\n    last = folder/\'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last,map_location=\'cpu\',weights_only=False)\n        if ckpt[\'train_ids\']!=train_ids or ckpt[\'val_ids\']!=val_ids:\n            raise ValueError(\'Checkpoint split mismatch\')\n        model.load_state_dict(ckpt[\'model\']);ema.load_state_dict(ckpt[\'ema\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\']);scaler.load_state_dict(ckpt[\'scaler\'])\n        history,first,step = ckpt[\'history\'],ckpt[\'epoch\']+1,ckpt[\'step\']\n        base.restore_rng(ckpt[\'rng\'])\n        del ckpt\n        print(f\'fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        for image,target in loader:\n            set_lrs(optimizer,step,len(loader),cfg)\n            image,target = image.to(device,non_blocking=True),target.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss = objective(model(image),target)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:\n                update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()))\n            step += 1\n        score = None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            score = float(np.mean([base.dice_score(\n                resize_probability(predict_pair(ema,p,cfg,device,False)[\'none\'])>=0.5,\n                base.strict_label(labels[p.stem])) for p in val_paths]))\n        row = dict(epoch=epoch,loss=float(np.mean(losses)),monitor_dice_05=score,\n                   encoder_lr=optimizer.param_groups[0][\'lr\'],decoder_lr=optimizer.param_groups[1][\'lr\'],\n                   seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:\n            raise OSError(\'Less than 1.2 GiB free: preserve old checkpoint and free space before resuming.\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),\n            optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),history=history,\n            epoch=epoch,step=step,rng=base.rng_state(),train_ids=train_ids,val_ids=val_ids))\n        with (folder/\'history.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,list(row));writer.writeheader();writer.writerows(history)\n        print(f\'fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} \'+\n              (f\'monitor_dice@0.5={score:.5f} \' if score is not None else \'\')+\n              f\'seconds={row["seconds"]}\',flush=True)\n        if epoch==1:\n            print(f\'fold{fold}: measured estimate for this fold training/monitoring: \'\n                  f\'{row["seconds"]*cfg.epochs/60:.1f} min (startup may inflate this; D4 export extra)\',flush=True)\n    # Final EMA is fixed in advance. Validation never selects epochs/checkpoints.\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n                          train_ids=train_ids,val_ids=val_ids,selection=\'predeclared final EMA\'))\n    del model,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    return ema\n\n\ndef infer_fold(model, val_paths, test, labels, cfg, folder, device):\n    val_probs,test_probs = {},{}\n    for p in val_paths:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            val_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n            val_probs[f\'native_{mode}_{p.stem}\'] = native\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**val_probs)\n    for p in test:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            test_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n    np.savez_compressed(folder/\'test_probabilities.npz\',**test_probs)\n    clean = {p.stem:base.strict_label(labels[p.stem]) for p in val_paths}\n    summary = dict(epoch=cfg.epochs,val_ids=[p.stem for p in val_paths],\n        none_dice_05=float(np.mean([base.dice_score(val_probs[f\'none_{i}\']>=0.5,clean[i]) for i in clean])),\n        d4_dice_05=float(np.mean([base.dice_score(val_probs[f\'d4_{i}\']>=0.5,clean[i]) for i in clean])))\n    files=[\'history.csv\',\'last.pt\',\'final_ema.pt\',\'validation_probabilities.npz\',\'test_probabilities.npz\']\n    summary[\'sha256\']={name:base.file_sha(folder/name) for name in files}\n    base.save_json(folder/\'completed.json\',summary)\n    print(\'Fold completed:\',summary[\'val_ids\'],summary[\'none_dice_05\'],summary[\'d4_dice_05\'],flush=True)\n\n\ndef threshold_sweep(probs, targets):\n    thresholds = np.round(np.arange(0.1,0.901,0.025),3)\n    rows=[dict(threshold=float(t),mean_dice=float(np.mean([\n        base.dice_score(probs[i]>=t,targets[i]) for i in probs]))) for t in thresholds]\n    best=max(rows,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n    return best,rows\n\n\ndef export_csv(probs, ids, threshold, name, root, output):\n    folder=output/\'masks_512_white_vessels\'/name\n    folder.mkdir(parents=True,exist_ok=True)\n    filenames,encoded=[],[]\n    for i in ids:\n        mask=probs[i]>=threshold\n        path=folder/f\'{i}.png\';Image.fromarray((mask*255).astype(np.uint8)).save(path)\n        value=base.rle_encode(mask)\n        if not np.array_equal(base.rle_decode(value),mask):\n            raise AssertionError(\'RLE mismatch\')\n        filenames.append(str(path));encoded.append(value)\n    official=base.load_converter(root).turn_to_str(filenames)\n    if official!=encoded:raise AssertionError(\'Official converter mismatch\')\n    path=output/f\'{name}.csv\'\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(ids,official))\n    return dict(file=path.name,threshold=threshold,sha256=base.file_sha(path),\n                rows=len(ids),official_converter_exact_match=True,rle_roundtrip=True)\n\n\ndef aggregate(root, output, cfg, train, test, labels, assignment, ids):\n    folds=[output/f\'fold{f}\' for f in range(cfg.folds)]\n    if not all((folder/\'completed.json\').exists() for folder in folds):\n        finished=[f for f,folder in enumerate(folds) if (folder/\'completed.json\').exists()]\n        base.save_json(output/\'partial_cv_summary.json\',dict(completed_folds=finished,complete_oof=False))\n        print(\'Partial CV only; no full-OOF score or ensemble submission. Completed:\',finished,flush=True)\n        return None\n    oof={mode:{} for mode in [\'none\',\'d4\']}\n    native={mode:{} for mode in [\'none\',\'d4\']}\n    test_mean={mode:{i:np.zeros((512,512),np.float64) for i in ids} for mode in oof}\n    for fold,folder in enumerate(folds):\n        metadata=json.loads((folder/\'completed.json\').read_text(encoding=\'utf-8\'))\n        for name,digest in metadata[\'sha256\'].items():\n            if base.file_sha(folder/name)!=digest:raise ValueError(f\'Artifact changed: {folder/name}\')\n        expected={p.stem for p in train if assignment[p.stem]==fold}\n        if set(metadata[\'val_ids\'])!=expected:raise ValueError(\'Validation IDs mismatch\')\n        with np.load(folder/\'validation_probabilities.npz\') as values:\n            for mode in oof:\n                for i in expected:\n                    if i in oof[mode]:raise AssertionError(\'Repeated OOF ID\')\n                    oof[mode][i]=values[f\'{mode}_{i}\'].copy()\n                    native[mode][i]=values[f\'native_{mode}_{i}\'].copy()\n        with np.load(folder/\'test_probabilities.npz\') as values:\n            for mode in oof:\n                for i in ids:test_mean[mode][i]+=values[f\'{mode}_{i}\']/cfg.folds\n    train_ids={p.stem for p in train}\n    if any(set(probs)!=train_ids for probs in oof.values()):raise AssertionError(\'Incomplete OOF coverage\')\n    clean={i:base.strict_label(labels[i]) for i in train_ids}\n    proxy={i:base.converter_label(labels[i]) for i in train_ids}\n    rows,mode_summary,exported=[],{},{}\n    for mode in oof:\n        calibrated,sweep=threshold_sweep(oof[mode],clean)\n        proxy_calibrated,proxy_sweep=threshold_sweep(oof[mode],proxy)\n        threshold=calibrated[\'threshold\']\n        cross_thresholds={}\n        for fold in range(cfg.folds):\n            other={j:oof[mode][j] for j in train_ids if assignment[j]!=fold}\n            cross_thresholds[fold]=threshold_sweep(other,{j:clean[j] for j in other})[0][\'threshold\']\n        with (output/f\'threshold_sweep_{mode}.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,[\'threshold\',\'clean_dice\',\'converter_proxy_dice\']);writer.writeheader()\n            writer.writerows(dict(threshold=a[\'threshold\'],clean_dice=a[\'mean_dice\'],converter_proxy_dice=b[\'mean_dice\'])\n                             for a,b in zip(sweep,proxy_sweep))\n        per=[]\n        for i in sorted(train_ids,key=int):\n            # Threshold calibration excluding this outer fold: diagnostic, not nested CV.\n            cross_t=cross_thresholds[assignment[i]]\n            gt_native=read_mask(labels[i])>0\n            row=dict(id=i,fold=assignment[i],mode=mode,\n                dice_05=base.dice_score(oof[mode][i]>=0.5,clean[i]),\n                dice_global_threshold=base.dice_score(oof[mode][i]>=threshold,clean[i]),\n                dice_other_fold_threshold=base.dice_score(oof[mode][i]>=cross_t,clean[i]),\n                other_fold_threshold=cross_t,\n                native_dice_05=base.dice_score(native[mode][i]>=0.5,gt_native),\n                converter_proxy_dice=base.dice_score(oof[mode][i]>=threshold,proxy[i]))\n            rows.append(row);per.append(row)\n        mode_summary[mode]=dict(mean_dice_05=float(np.mean([r[\'dice_05\'] for r in per])),\n            global_threshold=threshold,mean_dice_global_threshold=calibrated[\'mean_dice\'],\n            mean_dice_other_fold_threshold=float(np.mean([r[\'dice_other_fold_threshold\'] for r in per])),\n            mean_native_dice_05=float(np.mean([r[\'native_dice_05\'] for r in per])),\n            mean_converter_proxy_dice=float(np.mean([r[\'converter_proxy_dice\'] for r in per])),\n            converter_proxy_best_threshold=proxy_calibrated[\'threshold\'],\n            converter_proxy_best_dice=proxy_calibrated[\'mean_dice\'])\n        exported[mode]=export_csv(test_mean[mode],ids,threshold,f\'submission_{mode}\',root,output)\n    chosen=\'d4\' if mode_summary[\'d4\'][\'mean_dice_05\']>=mode_summary[\'none\'][\'mean_dice_05\']+cfg.tta_gain_required else \'none\'\n    source=output/exported[chosen][\'file\']\n    shutil.copyfile(source,output/\'submission.csv\')\n    # Explicitly named diagnostic candidate; never silently substitute it for clean vessels.\n    proxy_file=export_csv(test_mean[chosen],ids,mode_summary[chosen][\'converter_proxy_best_threshold\'],\n                         \'submission_converter_proxy\',root,output)\n    threshold=mode_summary[chosen][\'global_threshold\']\n    base.preview(test,{p.stem:test_mean[chosen][p.stem] for p in test},None,threshold,output/\'test_preview.png\')\n    fold0=[p for p in train if assignment[p.stem]==0]\n    base.preview(fold0,oof[chosen],labels,threshold,output/\'oof_fold0_preview.png\')\n    black=output/\'masks_native_black_vessels\';black.mkdir(exist_ok=True)\n    for p in test:\n        width,height=Image.open(p).size\n        prob=resize_probability(test_mean[chosen][p.stem],(height,width))\n        Image.fromarray(np.where(prob>=threshold,0,255).astype(np.uint8)).save(black/f\'{p.stem}.png\')\n    with (output/\'oof_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    np.savez_compressed(output/\'oof_probabilities.npz\',**{f\'{mode}_{i}\':p for mode,probs in oof.items() for i,p in probs.items()})\n    np.savez_compressed(output/\'test_ensemble_probabilities.npz\',**{f\'{mode}_{i}\':p.astype(np.float32) for mode,probs in test_mean.items() for i,p in probs.items()})\n    summary=dict(complete_oof=True,unique_oof_images=len(train_ids),epochs=cfg.epochs,\n        checkpoint_selection=\'fixed final EMA; no outer-validation epoch selection\',\n        modes=mode_summary,selected_mode=chosen,tta_gain_required=cfg.tta_gain_required,\n        test_ensemble=\'equal average of five fold models; each trained on 16 images\',\n        baseline_reference=dict(ids=[\'2\',\'6\',\'9\',\'10\'],baseline_dice_05=0.7839949354578135,\n            v2_same_ids_none_dice_05=float(np.mean([r[\'dice_05\'] for r in rows if r[\'id\'] in {\'2\',\'6\',\'9\',\'10\'} and r[\'mode\']==\'none\'])),\n            note=\'Matched subset only; baseline did not produce full five-fold OOF.\'),\n        selection_caveat=\'TTA mode and global threshold use OOF labels. Tuned scores are development estimates. \'\n                         \'Other-fold calibration is diagnostic, not a nested independent test.\',\n        private_target_preprocessing_verified=False,submitted_to_competition=False)\n    base.save_json(output/\'cv_summary.json\',summary)\n    base.save_json(output/\'submission_provenance.json\',dict(chosen_mode=chosen,\n        primary_csv_sha256=base.file_sha(output/\'submission.csv\'),alternatives=exported,\n        converter_proxy_candidate=proxy_file,fold_models=[base.file_sha(p/\'final_ema.pt\') for p in folds],\n        calibrated_on=\'20 OOF clean masks; proxy candidate separately calibrated on JPEG converter targets\',\n        image_polarity=\'CSV white vessels; separate native-size PNGs have vessel=0, background=255\',\n        official_converter_exact_match=True,external_vessel_training_data=False))\n    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n    return summary\n\n\ndef make_backups(output, full=True):\n    """Portable analysis archive; optional full archive includes model and resume states."""\n    output=Path(output)\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix not in {\'.zip\',\'.tmp\'} and p.name!=\'training.lock\']\n    light=[p for p in files if p.suffix in {\'.json\',\'.csv\',\'.png\',\'.py\',\'.md\'} and \'pretrained_cache\' not in p.parts\n           and not (p.suffix==\'.png\' and \'masks_512_white_vessels\' in p.parts)]\n    for name,paths in [(\'analysis_results.zip\',light)]+([(\'full_backup.zip\',files)] if full else []):\n        target=output/name;tmp=output/(name+\'.tmp\')\n        estimated=sum(p.stat().st_size for p in paths)\n        if shutil.disk_usage(output).free<estimated+256*1024**2:\n            raise OSError(f\'Not enough free space to create {name}; completed results remain intact.\')\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_STORED,allowZip64=True) as z:\n            for path in paths:z.write(path,path.relative_to(output).as_posix())\n        with zipfile.ZipFile(tmp) as z:\n            bad=z.testzip()\n            if bad:raise IOError(f\'Archive CRC failed: {bad}\')\n        os.replace(tmp,target)\n        print(\'Backup saved:\',target,f\'{target.stat().st_size/1024**2:.1f} MiB\',flush=True)\n\n\ndef restore_output(source, output):\n    source,output=Path(source).resolve(),Path(output).resolve()\n    if source==output:return\n    if output.exists() and any(output.iterdir()):\n        # Existing working output is the authoritative continuation; do not overlay backup.\n        if (output/\'run_config.json\').exists():return\n        raise ValueError(\'Restore target is not empty\')\n    output.mkdir(parents=True,exist_ok=True)\n    if source.is_dir():\n        shutil.copytree(source,output,dirs_exist_ok=True,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\'))\n    elif source.suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(source) as z:\n            for item in z.infolist():\n                resolved=(output/item.filename).resolve()\n                if not resolved.is_relative_to(output):raise ValueError(\'Unsafe archive path\')\n            z.extractall(output)\n        (output/\'training.lock\').unlink(missing_ok=True)\n    else:raise ValueError(\'RESUME_FROM must be an existing output folder or full_backup.zip\')\n\n\n@contextmanager\ndef run_lock(output):\n    lock=output/\'training.lock\'\n    if lock.exists():\n        info=json.loads(lock.read_text())\n        alive=False\n        if info.get(\'host\')==socket.gethostname():\n            if os.name==\'nt\':\n                import ctypes\n                from ctypes import wintypes\n                kernel=ctypes.WinDLL(\'kernel32\',use_last_error=True)\n                kernel.OpenProcess.argtypes=[wintypes.DWORD,wintypes.BOOL,wintypes.DWORD]\n                kernel.OpenProcess.restype=wintypes.HANDLE\n                kernel.GetExitCodeProcess.argtypes=[wintypes.HANDLE,ctypes.POINTER(wintypes.DWORD)]\n                kernel.CloseHandle.argtypes=[wintypes.HANDLE]\n                handle=kernel.OpenProcess(0x1000,False,int(info[\'pid\']))\n                if handle:\n                    code=wintypes.DWORD()\n                    alive=bool(kernel.GetExitCodeProcess(handle,ctypes.byref(code))) and code.value==259\n                    kernel.CloseHandle(handle)\n            else:\n                try:os.kill(info[\'pid\'],0);alive=True\n                except OSError:pass\n        if alive:raise RuntimeError(f\'Output is in use by PID {info["pid"]}\')\n        lock.unlink()\n    descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    os.write(descriptor,json.dumps(dict(pid=os.getpid(),host=socket.gethostname())).encode());os.close(descriptor)\n    try:yield\n    finally:lock.unlink(missing_ok=True)\n\n\ndef run_cv(data_dir=None, output_dir=\'task4_v2_results\', cfg=None, folds_to_run=(0,1,2,3,4),\n           weights_path=None, resume_from=None, backups=True, full_backup=True):\n    cfg=cfg or ConfigV2()\n    if cfg.folds!=5 or cfg.fold!=0 or cfg.patch<64 or cfg.patch%32 or cfg.patch>512:\n        raise ValueError(\'Require 5 folds, audit fold=0, patch multiple of 32 in [64,512].\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1:\n        raise ValueError(\'Invalid training configuration\')\n    if not 0<cfg.ema_decay<1 or len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):\n        raise ValueError(\'Invalid EMA decay/fold list\')\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    output=Path(output_dir).resolve()\n    if resume_from:restore_output(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n                  baseline_helpers_sha256=base.file_sha(base.__file__))\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):\n        raise ValueError(\'Changed config/source: use a NEW OUTPUT_DIR to preserve prior results.\')\n    with run_lock(output):\n        train,test,labels,assignment,ids,audit=audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous and previous[\'data_fingerprint\']!=identity[\'data_fingerprint\']:\n            raise ValueError(\'Dataset changed: use a new output directory\')\n        initial_state,provenance=resolve_weights(cfg,output,weights_path)\n        identity[\'pretrained_sha256\']=provenance[\'sha256\']\n        if previous and previous!=identity:raise ValueError(\'Pretrained weights differ from saved run\')\n        base.save_json(config_path,identity)\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        base.save_json(output/\'environment.json\',dict(python=platform.python_version(),torch=torch.__version__,\n            torchvision=torchvision.__version__,opencv=cv2.__version__,numpy=np.__version__,\n            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            training_device_count=1,torch_cuda=torch.version.cuda))\n        code_dir=output/\'runtime_source\';code_dir.mkdir(exist_ok=True)\n        for p in [Path(__file__),Path(base.__file__)]:\n            if p.resolve()!=(code_dir/p.name).resolve():shutil.copyfile(p,code_dir/p.name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\',device,\'fixed epochs:\',cfg.epochs,\'folds requested:\',folds_to_run,flush=True)\n        if device.type!=\'cuda\':print(\'CPU detected; formal five-fold training will be slow. Enable GPU.\',flush=True)\n        for fold in folds_to_run:\n            folder=output/f\'fold{fold}\'\n            if (folder/\'completed.json\').exists():\n                print(f\'fold{fold}: completed, reusing verified artifacts\',flush=True)\n                continue\n            development=[p for p in train if assignment[p.stem]!=fold]\n            validation=[p for p in train if assignment[p.stem]==fold]\n            model=train_fold(development,validation,labels,cfg,fold,folder,initial_state,device)\n            infer_fold(model,validation,test,labels,cfg,folder,device)\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n        summary=aggregate(root,output,cfg,train,test,labels,assignment,ids)\n        if backups:make_backups(output,full=full_backup)\n    return output,summary\n', 'task4_controlled_ablation.py': '"""OOF-only boundary/EMA replay and paired label-supervision experiments.\n\nNo test-set predictions, competition submissions, or public-score-based selection.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nimport task4_pipeline as base\nimport task4_kaggle_v2 as v2\n\n\nVARIANTS=(\'L0_clean\',\'L1_proxy\',\'L2_soft\')\nCALIBRATION_GRID=(0.05,0.075,0.1,0.125,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9)\n\n\n@dataclass\nclass AblationConfig(v2.ConfigV2):\n    soft_boundary: float = 0.5\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()\n\n\ndef write_table(path, rows):\n    if not rows:return\n    with Path(path).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n\n\ndef skeleton(mask):\n    try:\n        from skimage.morphology import skeletonize\n    except ImportError as exc:\n        raise RuntimeError(\'Install scikit-image for the preregistered centerline metrics.\') from exc\n    return skeletonize(np.asarray(mask,bool))\n\n\ndef targets(labels):\n    result={}\n    for i,path in labels.items():\n        c,p=base.strict_label(path),base.converter_label(path)\n        if np.any(c&~p):raise ValueError(\'Expected clean mask to be a subset of converter proxy\')\n        sc,sp=skeleton(c),skeleton(p)\n        radius=cv2.distanceTransform(c.astype(np.uint8),cv2.DIST_L2,5)\n        result[i]=dict(clean=c,proxy=p,clean_skeleton=sc,proxy_skeleton=sp,\n                       thin_clean_centerline=sc&(radius<=1.5))\n    return result\n\n\ndef segmentation_metrics(pred, truth, truth_skeleton, pred_skeleton):\n    pred,truth=np.asarray(pred,bool),np.asarray(truth,bool)\n    tp=int((pred&truth).sum());n_pred=int(pred.sum());n_true=int(truth.sum())\n    precision=tp/n_pred if n_pred else (1.0 if not n_true else 0.0)\n    recall=tp/n_true if n_true else 1.0\n    top_precision=float((pred_skeleton&truth).sum()/pred_skeleton.sum()) if pred_skeleton.any() else (1.0 if not n_true else 0.0)\n    top_recall=float((truth_skeleton&pred).sum()/truth_skeleton.sum()) if truth_skeleton.any() else 1.0\n    cldice=2*top_precision*top_recall/(top_precision+top_recall) if top_precision+top_recall else 0.0\n    return dict(dice=base.dice_score(pred,truth),precision=precision,recall=recall,\n                centerline_precision=top_precision,centerline_recall=top_recall,cldice=cldice,\n                foreground_fraction=n_pred/pred.size)\n\n\ndef evaluate_masks(experiment, masks, truth, assignments):\n    rows=[]\n    for i in sorted(masks,key=int):\n        mask=np.asarray(masks[i],bool);sp=skeleton(mask)\n        thin=truth[i][\'thin_clean_centerline\']\n        thin_recall=float((mask&thin).sum()/thin.sum()) if thin.any() else None\n        for target in [\'clean\',\'proxy\']:\n            values=segmentation_metrics(mask,truth[i][target],truth[i][target+\'_skeleton\'],sp)\n            rows.append(dict(experiment=experiment,id=i,fold=assignments[i],target=target,\n                             **values,thin_clean_centerline_recall=thin_recall))\n    return rows\n\n\ndef summarize_rows(rows):\n    numeric=[\'dice\',\'precision\',\'recall\',\'centerline_precision\',\'centerline_recall\',\'cldice\',\n             \'foreground_fraction\',\'thin_clean_centerline_recall\']\n    summaries,folds=[],[]\n    for experiment,target in sorted({(r[\'experiment\'],r[\'target\']) for r in rows}):\n        group=[r for r in rows if r[\'experiment\']==experiment and r[\'target\']==target]\n        assert len({r[\'id\'] for r in group})==len(group)\n        values={key:float(np.mean([r[key] for r in group if r[key] is not None]))\n                if any(r[key] is not None for r in group) else None for key in numeric}\n        fold_values=[]\n        for fold in sorted({r[\'fold\'] for r in group}):\n            subset=[r for r in group if r[\'fold\']==fold]\n            score=float(np.mean([r[\'dice\'] for r in subset]));fold_values.append(score)\n            folds.append(dict(experiment=experiment,target=target,fold=fold,n_images=len(subset),\n                              **{key:float(np.mean([r[key] for r in subset if r[key] is not None]))\n                                 if any(r[key] is not None for r in subset) else None for key in numeric}))\n        summaries.append(dict(experiment=experiment,target=target,n_images=len(group),\n            complete_20_image_oof=len(group)==20,**values,worst_fold_dice=min(fold_values),\n            fold_dice_std=float(np.std(fold_values))))\n    return summaries,folds\n\n\ndef probability_preview(images, truth, probs, output):\n    ids=sorted(probs,key=int)[:4];tile=220\n    canvas=Image.new(\'RGB\',(tile*4,(tile+24)*len(ids)),\'white\');draw=ImageDraw.Draw(canvas)\n    for row,i in enumerate(ids):\n        y=row*(tile+24)\n        p=probs[i]\n        hard=p>=0.4;low=p>=0.1\n        dil=cv2.dilate(hard.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))>0\n        panels=[Image.open(images[i]).convert(\'RGB\')]+[Image.fromarray((m*255).astype(np.uint8)).convert(\'RGB\') for m in [hard,low,dil]]\n        for col,(panel,label) in enumerate(zip(panels,[f\'OOF {i}\',\'p >= 0.4\',\'p >= 0.1\',\'dilate(p >= 0.4)\'])):\n            draw.text((col*tile+3,y+4),label,fill=\'black\')\n            canvas.paste(panel.resize((tile,tile)),(col*tile,y+24))\n    canvas.save(output)\n\n\ndef materialize_previous(source, output):\n    """Use a mounted folder read-only, or extract only the artifacts used by replay."""\n    if source is None:\n        preferred=Path(\'/kaggle/working/task4_v2_results\')\n        roots=[preferred] if (preferred/\'run_config.json\').exists() else []\n        if not roots and Path(\'/kaggle/input\').exists():\n            for p in Path(\'/kaggle/input\').rglob(\'run_config.json\'):\n                try:cfg=json.loads(p.read_text())\n                except (ValueError,OSError):continue\n                if cfg.get(\'source_sha256\')==base.file_sha(v2.__file__) and all((p.parent/f\'fold{f}/validation_probabilities.npz\').exists() for f in range(5)):\n                    roots.append(p.parent)\n        roots=sorted(set(p.resolve() for p in roots))\n        if len(roots)==1:return roots[0]\n        zips=list(Path(\'/kaggle/input\').rglob(\'full_backup.zip\')) if Path(\'/kaggle/input\').exists() else []\n        if not roots and len(zips)==1:source=zips[0]\n        else:raise ValueError(f\'Set PREVIOUS_RUN to the previous complete output folder or full_backup.zip. Folders found: {roots}\')\n    source=Path(source).resolve()\n    if source.is_dir():return source\n    if not source.is_file() or source.suffix.lower()!=\'.zip\':raise ValueError(\'PREVIOUS_RUN must be a folder or full_backup.zip\')\n    cached=output/\'previous_run_cache\';cached.mkdir(parents=True,exist_ok=True)\n    archive_sha=base.file_sha(source);marker=cached/\'archive_identity.json\'\n    if marker.exists():\n        if json.loads(marker.read_text())[\'archive_sha256\']!=archive_sha:raise ValueError(\'Different previous archive: use a new OUTPUT_DIR\')\n        return cached\n    with zipfile.ZipFile(source) as z:\n        names=z.namelist()\n        if len(names)!=len(set(names)):raise ValueError(\'Duplicate ZIP members\')\n        wanted=[]\n        for name in names:\n            dest=(cached/name).resolve()\n            if not dest.is_relative_to(cached):raise ValueError(\'Unsafe archive path\')\n            parts=Path(name).parts\n            if name in {\'run_config.json\',\'pretrained_source.json\'} or (\n                len(parts)==2 and parts[0] in {f\'fold{f}\' for f in range(5)} and parts[1] in {\n                    \'completed.json\',\'last.pt\',\'validation_probabilities.npz\',\'history.csv\'}) or (\n                len(parts)==2 and parts[0]==\'pretrained_cache\' and parts[1]==\'resnet34-b627a593.pth\'):\n                wanted.append(name)\n        required_members={\'run_config.json\'}|{f\'fold{f}/{name}\' for f in range(5) for name in [\'completed.json\',\'validation_probabilities.npz\']}\n        if not required_members<=set(wanted):raise ValueError(\'The analysis-only ZIP cannot support these ablations; use the full backup with all five OOF probability files.\')\n        required=sum(z.getinfo(n).file_size for n in wanted)\n        if shutil.disk_usage(output).free<required+512*1024**2:raise OSError(\'Not enough disk space to extract the required prior artifacts\')\n        for name in wanted:z.extract(name,cached)\n    base.save_json(marker,dict(archive_sha256=archive_sha,extracted_files=wanted))\n    return cached\n\n\ndef validate_previous(previous, assignment, fingerprint, expected=None, require_last=True, expected_epochs=80):\n    identity=json.loads((previous/\'run_config.json\').read_text())\n    if identity[\'source_sha256\']!=base.file_sha(v2.__file__) or identity[\'baseline_helpers_sha256\']!=base.file_sha(base.__file__):\n        raise ValueError(\'Previous source differs from the pinned v2 implementation\')\n    if identity[\'data_fingerprint\']!=fingerprint:raise ValueError(\'Previous run uses different training data/split\')\n    cfg=v2.ConfigV2(**identity[\'config\'])\n    if cfg.epochs!=expected_epochs:raise ValueError(f\'Expected a completed {expected_epochs}-epoch run, found {cfg.epochs}\')\n    records={}\n    for fold in range(5):\n        folder=previous/f\'fold{fold}\'\n        done=json.loads((folder/\'completed.json\').read_text())\n        if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:\n            raise ValueError(f\'Incomplete/wrong split: fold{fold}\')\n        wanted=[\'validation_probabilities.npz\']+([\'last.pt\'] if require_last else [])\n        for name in wanted:\n            if not (folder/name).is_file():raise ValueError(f\'Missing fold{fold}/{name}; use the complete prior output/full backup, not the analysis-only ZIP.\')\n            actual=base.file_sha(folder/name)\n            if actual!=done[\'sha256\'][name]:raise ValueError(f\'Changed previous artifact: fold{fold}/{name}\')\n            if expected is not None and actual!=expected[str(fold)][\'sha256\'][name]:\n                raise ValueError(f\'Previous artifact does not match the returned analysis archive: fold{fold}/{name}\')\n        records[str(fold)]=done\n    return cfg,records\n\n\ndef replay_fold(previous, fold, paths, cfg, device, cache):\n    checkpoint=previous/f\'fold{fold}/last.pt\'\n    key=dict(checkpoint_sha256=base.file_sha(checkpoint),config=asdict(cfg),\n             gpu=torch.cuda.get_device_name(0) if device.type==\'cuda\' else \'cpu\',torch=torch.__version__)\n    marker=cache/f\'fold{fold}_identity.json\';npz=cache/f\'fold{fold}_replay.npz\'\n    if marker.exists() and npz.exists():\n        old=json.loads(marker.read_text())\n        if old[\'identity\']!=key or old[\'npz_sha256\']!=base.file_sha(npz):raise ValueError(\'Replay cache changed; use a new output directory\')\n        with np.load(npz) as arrays:return {k:arrays[k].copy() for k in arrays.files}\n    # Hashes were checked against the known previous run before loading its own resume file.\n    state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    expected_ids={p.stem for p in paths}\n    if state[\'epoch\']!=cfg.epochs or set(state[\'val_ids\'])!=expected_ids or set(state[\'train_ids\'])&expected_ids:\n        raise ValueError(\'OOF leakage/epoch check failed\')\n    model=v2.ResNet34UNet().to(device);values={}\n    for name,key_name in [(\'replay_ema\',\'ema\'),(\'raw\',\'model\')]:\n        model.load_state_dict(state[key_name]);model.eval()\n        for p in paths:\n            pair=v2.predict_pair(model,p,cfg,device)\n            for mode,prob in pair.items():values[f\'{name}_{mode}_{p.stem}\']=v2.resize_probability(prob)\n    del model,state\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    cache.mkdir(parents=True,exist_ok=True)\n    np.savez_compressed(npz,**values)\n    base.save_json(marker,dict(identity=key,npz_sha256=base.file_sha(npz)))\n    return values\n\n\ndef run_replay(previous, output, train, labels, assignment, fingerprint, expected=None, raw_replay=True):\n    cfg,records=validate_previous(previous,assignment,fingerprint,expected,raw_replay)\n    truth=targets(labels);cached={};replayed={\'replay_ema\':{},\'raw\':{}};drift=[]\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if raw_replay and device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for paired EMA/raw replay, or turn RUN_RAW_REPLAY off\')\n    for fold in range(5):\n        val=[p for p in train if assignment[p.stem]==fold]\n        with np.load(previous/f\'fold{fold}/validation_probabilities.npz\') as arrays:\n            for p in val:\n                prob=arrays[f\'d4_{p.stem}\'].copy()\n                if prob.shape!=(512,512) or not np.isfinite(prob).all() or prob.min()<0 or prob.max()>1:raise ValueError(\'Invalid cached OOF probability\')\n                cached[p.stem]=prob\n        if raw_replay:\n            values=replay_fold(previous,fold,val,cfg,device,output/\'replay_cache\')\n            for p in val:\n                for name in replayed:replayed[name][p.stem]=values[f\'{name}_d4_{p.stem}\']\n                delta=np.abs(replayed[\'replay_ema\'][p.stem]-cached[p.stem])\n                drift.append(dict(id=p.stem,fold=fold,mean_absolute_probability_drift=float(delta.mean()),\n                                  max_absolute_probability_drift=float(delta.max())))\n        print(f\'Replay fold{fold} completed; no training\',flush=True)\n    if set(cached)!=set(assignment):raise AssertionError(\'Incomplete OOF coverage\')\n    experiments={\n        \'E1_ema_t04\':{i:p>=0.4 for i,p in cached.items()},\n        \'E1_ema_t01\':{i:p>=0.1 for i,p in cached.items()},\n        \'E1_ema_t04_dilate1\':{i:cv2.dilate((p>=0.4).astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))>0 for i,p in cached.items()}}\n    if raw_replay:\n        for name,probs in replayed.items():\n            for t,label in [(0.1,\'t01\'),(0.4,\'t04\')]:experiments[f\'E2_{name}_{label}\']={i:p>=t for i,p in probs.items()}\n    rows=[row for name,masks in experiments.items() for row in evaluate_masks(name,masks,truth,assignment)]\n    summary,folds=summarize_rows(rows)\n    write_table(output/\'replay_per_image.csv\',rows);write_table(output/\'replay_summary.csv\',summary)\n    write_table(output/\'replay_per_fold.csv\',folds);write_table(output/\'ema_replay_drift.csv\',drift)\n    probability_preview({p.stem:p for p in train},truth,cached,output/\'boundary_ablation_preview.png\')\n    base.save_json(output/\'replay_audit.json\',dict(complete_20_image_oof=True,fold_records=records,\n        raw_and_ema_replayed_in_same_environment=raw_replay,previous_probability_hashes_verified=True,\n        training_started=False,competition_csv_generated=False,\n        interpretation=\'Morphology and centerline changes are paired diagnostics; no winner selected from public scores.\'))\n    return summary\n\n\nclass PairedPatches(Dataset):\n    """All label variants receive exactly the same RGB patches and augmentations."""\n    def __init__(self, paths, labels, cfg, fold):\n        self.ids=[p.stem for p in paths];self.cfg=cfg;self.fold=fold\n        self.images=[cv2.resize(v2.read_rgb(p),(512,512),interpolation=cv2.INTER_AREA) for p in paths]\n        self.clean=[base.strict_label(labels[i]).astype(np.float32) for i in self.ids]\n        self.proxy=[base.converter_label(labels[i]).astype(np.float32) for i in self.ids]\n        self.positive=[np.argwhere(c>0) for c in self.clean]\n        self.plans=[]\n\n    def set_epoch(self, epoch):\n        self.plans=[]\n        for index in range(self.cfg.patches_per_epoch):\n            rng=random.Random((self.cfg.seed+1009*self.fold)*10**10+epoch*10**6+index)\n            i=rng.randrange(len(self.ids));patch=self.cfg.patch\n            crop=rng.randint(int(patch*0.9),min(int(patch*1.1),512))\n            if rng.random()<0.3 and len(self.positive[i]):\n                y,x=self.positive[i][rng.randrange(len(self.positive[i]))]\n                top,left=int(np.clip(y-crop//2,0,512-crop)),int(np.clip(x-crop//2,0,512-crop))\n            else:top,left=rng.randint(0,512-crop),rng.randint(0,512-crop)\n            self.plans.append((i,top,left,crop,rng.randrange(4),rng.random()<0.5,\n                rng.uniform(0.85,1.15),rng.uniform(-0.04,0.04),rng.uniform(0.85,1.15),\n                [rng.uniform(0.95,1.05) for _ in range(3)]))\n        return digest_json(self.plans)\n\n    def __len__(self):return self.cfg.patches_per_epoch\n\n    def __getitem__(self,index):\n        i,top,left,crop,rotation,flip,contrast,brightness,gamma,gains=self.plans[index]\n        patch=self.cfg.patch\n        image=cv2.resize(self.images[i][top:top+crop,left:left+crop],(patch,patch),interpolation=cv2.INTER_LINEAR)\n        pair=np.stack([self.clean[i],self.proxy[i]],axis=-1)\n        pair=cv2.resize(pair[top:top+crop,left:left+crop],(patch,patch),interpolation=cv2.INTER_NEAREST)\n        image,pair=base.paired_d4(image,pair,rotation,flip)\n        image=np.clip((image.astype(np.float32)/255-0.5)*contrast+0.5+brightness,0,1)**gamma\n        image=np.clip(image*np.array(gains,np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()),torch.from_numpy(pair.transpose(2,0,1).copy())\n\n\ndef supervised_loss(outputs, pair, variant, alpha=0.5):\n    c,p=pair[:,0:1],pair[:,1:2]\n    if variant==\'L0_clean\':bce_target,dice_target=c,c\n    elif variant==\'L1_proxy\':bce_target,dice_target=p,p\n    elif variant==\'L2_soft\':bce_target,dice_target=c+alpha*(p-c),c\n    else:raise ValueError(variant)\n    losses=[]\n    for logits in outputs:\n        bt=F.interpolate(bce_target,size=logits.shape[-2:],mode=\'area\')\n        dt=F.interpolate(dice_target,size=logits.shape[-2:],mode=\'area\')\n        prob=logits.float().sigmoid()\n        dice=(2*(prob*dt).sum((1,2,3))+1)/(prob.sum((1,2,3))+dt.sum((1,2,3))+1)\n        losses.append(0.5*F.binary_cross_entropy_with_logits(logits.float(),bt.float())+0.5*(1-dice.mean()))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\ndef train_label_fold(paths, validation, labels, cfg, fold, variant, folder, initial, device):\n    folder.mkdir(parents=True,exist_ok=True);base.seed_all(cfg.seed+1009*fold)\n    dataset=PairedPatches(paths,labels,cfg,fold)\n    loader=DataLoader(dataset,batch_size=cfg.batch_size,shuffle=False,num_workers=0,pin_memory=device.type==\'cuda\')\n    model=v2.ResNet34UNet(initial).to(device);ema=copy.deepcopy(model).eval().requires_grad_(False)\n    initial_model_hash=digest_json({k:hashlib.sha256(v.detach().cpu().numpy().tobytes()).hexdigest() for k,v in model.state_dict().items()})\n    optimizer=torch.optim.AdamW([{\'params\':model.encoder.parameters(),\'lr\':cfg.encoder_lr},\n        {\'params\':[p for n,p in model.named_parameters() if not n.startswith(\'encoder.\')],\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step=[],1,0;last=folder/\'last.pt\'\n    train_ids,val_ids=[p.stem for p in paths],[p.stem for p in validation]\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=False)\n        if saved[\'variant\']!=variant or saved[\'train_ids\']!=train_ids or saved[\'val_ids\']!=val_ids or saved[\'initial_model_hash\']!=initial_model_hash:\n            raise ValueError(\'Label experiment checkpoint mismatch\')\n        model.load_state_dict(saved[\'model\']);ema.load_state_dict(saved[\'ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        first,step,history=saved[\'epoch\']+1,saved[\'step\'],saved[\'history\'];base.restore_rng(saved[\'rng\'])\n        del saved\n        print(f\'{variant}/fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin=time.perf_counter();model.train();plan_hash=dataset.set_epoch(epoch);losses=[]\n        for image,pair in loader:\n            v2.set_lrs(optimizer,step,len(loader),cfg)\n            image,pair=image.to(device,non_blocking=True),pair.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss=supervised_loss(model(image),pair,variant,cfg.soft_boundary)\n            if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite label experiment loss\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:v2.update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()));step+=1\n        monitor_c,monitor_p=None,None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            scores=[]\n            for path in validation:\n                image=cv2.resize(v2.read_rgb(path),(512,512),interpolation=cv2.INTER_AREA)\n                prob=v2.predict_pair(ema,image,cfg,device,False)[\'none\']\n                scores.append((base.dice_score(prob>=0.5,base.strict_label(labels[path.stem])),\n                               base.dice_score(prob>=0.5,base.converter_label(labels[path.stem]))))\n            monitor_c,monitor_p=map(float,np.mean(scores,axis=0))\n        row=dict(epoch=epoch,loss=float(np.mean(losses)),monitor_clean_05=monitor_c,monitor_proxy_05=monitor_p,\n                 plan_sha256=plan_hash,seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:raise OSError(\'Less than 1.2 GiB free; preserve checkpoint and resume after freeing space\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),optimizer=optimizer.state_dict(),\n            scaler=scaler.state_dict(),epoch=epoch,step=step,history=history,rng=base.rng_state(),\n            train_ids=train_ids,val_ids=val_ids,variant=variant,initial_model_hash=initial_model_hash))\n        write_table(folder/\'history.csv\',history)\n        print(f\'{variant}/fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} seconds={row["seconds"]}\',flush=True)\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n        variant=variant,train_ids=train_ids,val_ids=val_ids,initial_model_hash=initial_model_hash))\n    values={}\n    for path in validation:\n        image=cv2.resize(v2.read_rgb(path),(512,512),interpolation=cv2.INTER_AREA)\n        for mode,prob in v2.predict_pair(ema,image,cfg,device).items():values[f\'{mode}_{path.stem}\']=prob\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**values)\n    files=[\'last.pt\',\'final_ema.pt\',\'history.csv\',\'validation_probabilities.npz\']\n    base.save_json(folder/\'completed.json\',dict(epoch=cfg.epochs,variant=variant,train_ids=train_ids,val_ids=val_ids,\n        initial_model_hash=initial_model_hash,plan_hashes=[r[\'plan_sha256\'] for r in history],\n        sha256={name:base.file_sha(folder/name) for name in files}))\n    del model,ema,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n\n\ndef aggregate_labels(output, variants, cfg, labels, assignment):\n    complete=[variant for variant in variants if all((output/variant/f\'fold{fold}/completed.json\').exists() for fold in range(5))]\n    truth=targets(labels);all_rows=[];calibrated=[];sweeps=[]\n    for variant in complete:\n        probs={mode:{} for mode in [\'none\',\'d4\']}\n        for fold in range(5):\n            folder=output/variant/f\'fold{fold}\';done=json.loads((folder/\'completed.json\').read_text())\n            if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:raise ValueError(\'Wrong label OOF fold\')\n            for name,digest in done[\'sha256\'].items():\n                if base.file_sha(folder/name)!=digest:raise ValueError(f\'Changed label experiment artifact: {folder/name}\')\n            with np.load(folder/\'validation_probabilities.npz\') as arrays:\n                for mode in probs:\n                    for i in done[\'val_ids\']:probs[mode][i]=arrays[f\'{mode}_{i}\'].copy()\n        if any(set(p)!=set(assignment) for p in probs.values()):raise AssertionError(\'Incomplete label OOF\')\n        for mode,pred in probs.items():\n            for t in [0.1,0.4,0.5]:\n                all_rows+=evaluate_masks(f\'{variant}_{mode}_t{t:g}\',{i:p>=t for i,p in pred.items()},truth,assignment)\n            for target in [\'clean\',\'proxy\']:\n                grid=[]\n                for t in CALIBRATION_GRID:\n                    fold_scores={f:float(np.mean([base.dice_score(pred[i]>=t,truth[i][target]) for i in pred if assignment[i]==f])) for f in range(5)}\n                    row=dict(variant=variant,mode=mode,target=target,threshold=t,mean_dice=float(np.mean(list(fold_scores.values()))),\n                             worst_fold_dice=min(fold_scores.values()),**{f\'fold{f}\':v for f,v in fold_scores.items()})\n                    grid.append(row);sweeps.append(row)\n                best=max(grid,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n                calibrated.append(dict(**best,selection_caveat=\'OOF-tuned development estimate; not nested or independent validation\'))\n    # Pairing is verified even when only some folds/variants are ready.\n    pairing=[]\n    for fold in range(5):\n        records=[]\n        for variant in variants:\n            path=output/variant/f\'fold{fold}/completed.json\'\n            if path.exists():records.append((variant,json.loads(path.read_text())))\n        if len(records)>1:\n            first=records[0][1]\n            for variant,done in records[1:]:\n                if done[\'initial_model_hash\']!=first[\'initial_model_hash\'] or done[\'plan_hashes\']!=first[\'plan_hashes\']:\n                    raise AssertionError(\'Ablation pairing failed: initial weights or sampled RGB patches differ\')\n            pairing.append(dict(fold=fold,variants=[v for v,_ in records],same_initial_weights=True,same_rgb_plan=True))\n    summary,fold_rows=summarize_rows(all_rows)\n    write_table(output/\'label_per_image.csv\',all_rows);write_table(output/\'label_summary.csv\',summary)\n    write_table(output/\'label_per_fold.csv\',fold_rows);write_table(output/\'label_threshold_grid.csv\',sweeps)\n    write_table(output/\'label_calibration_development.csv\',calibrated)\n    base.save_json(output/\'label_audit.json\',dict(complete_variants=complete,requested_variants=list(variants),\n        pairing=pairing,complete_all_requested=len(complete)==len(variants),\n        no_automatic_winner=True,competition_csv_generated=False,public_scores_used=False,\n        calibration_grid=list(CALIBRATION_GRID),geometry=\'RGB/targets both 512; matched across L0/L1/L2\'))\n    return summary\n\n\ndef analysis_archive(output):\n    path=output/\'ablation_analysis_results.zip\';tmp=output/\'ablation_analysis_results.zip.tmp\'\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix in {\'.json\',\'.csv\',\'.png\',\'.md\',\'.py\'}\n           and \'previous_run_cache\' not in p.parts and p.name!=\'training.lock\']\n    with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in files:z.write(p,p.relative_to(output).as_posix())\n    with zipfile.ZipFile(tmp) as z:\n        if z.testzip():raise IOError(\'Analysis ZIP CRC failed\')\n    os.replace(tmp,path)\n    return path\n\n\ndef run_ablation(data_dir=None,output_dir=\'task4_ablation_results\',cfg=None,previous_run=None,\n                 run_inference=True,raw_replay=True,run_training=False,variants=VARIANTS,\n                 folds_to_run=(0,1,2,3,4),weights_path=None,expected_previous=None,resume_from=None):\n    cfg=cfg or AblationConfig()\n    if cfg.patch<64 or cfg.patch%32 or cfg.patch>512 or cfg.soft_boundary!=0.5 or cfg.folds!=5:\n        raise ValueError(\'Require patch multiple of 32 in [64,512], five folds, fixed soft band=0.5\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1 or cfg.threads<1:\n        raise ValueError(\'Invalid experiment settings\')\n    if not set(variants)<=set(VARIANTS) or len(set(variants))!=len(variants):raise ValueError(\'Invalid label variants\')\n    if not set(folds_to_run)<=set(range(5)) or len(set(folds_to_run))!=len(folds_to_run):raise ValueError(\'Invalid fold list\')\n    output=Path(output_dir).resolve()\n    if resume_from and not (output/\'ablation_config.json\').exists():\n        source=Path(resume_from)\n        if source.is_dir() and not (source/\'ablation_config.json\').exists():\n            raise ValueError(\'RESUME_ABLATION_FROM must point to this ablation output, not the original v2 run\')\n        if source.is_file():\n            if source.name==\'ablation_analysis_results.zip\':raise ValueError(\'The lightweight analysis ZIP cannot resume training\')\n            with zipfile.ZipFile(source) as z:\n                if \'ablation_config.json\' not in z.namelist():raise ValueError(\'Not an ablation resume archive\')\n        v2.restore_output(source,output)\n    output.mkdir(parents=True,exist_ok=True)\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n        v2_source_sha256=base.file_sha(v2.__file__),base_source_sha256=base.file_sha(base.__file__))\n    path=output/\'ablation_config.json\'\n    previous_identity=json.loads(path.read_text()) if path.exists() else None\n    if previous_identity and any(previous_identity.get(k)!=v for k,v in identity.items()):raise ValueError(\'Changed recipe/source: use a new OUTPUT_DIR\')\n    with v2.run_lock(output):\n        train,_,labels,assignment,_,audit=v2.audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous_identity and previous_identity!=identity:raise ValueError(\'Dataset changed\')\n        base.save_json(path,identity)\n        sources=output/\'runtime_source\';sources.mkdir(exist_ok=True)\n        for source in [Path(__file__),Path(v2.__file__),Path(base.__file__)]:shutil.copyfile(source,sources/source.name)\n        previous=None;inference_summary=[];label_summary=[]\n        if run_inference:\n            previous=materialize_previous(previous_run,output)\n            inference_summary=run_replay(previous,output,train,labels,assignment,audit[\'data_fingerprint\'],expected_previous,raw_replay)\n        if run_training:\n            device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n            if device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for label-supervision training\')\n            if weights_path is None and previous is not None:\n                candidate=previous/\'pretrained_cache/resnet34-b627a593.pth\'\n                if candidate.exists():weights_path=candidate\n            initial,weight_info=v2.resolve_weights(cfg,output,weights_path)\n            base.save_json(output/\'pretrained_source.json\',weight_info)\n            for variant in variants:\n                for fold in folds_to_run:\n                    folder=output/variant/f\'fold{fold}\'\n                    if (folder/\'completed.json\').exists():\n                        print(f\'{variant}/fold{fold}: completed, reusing\',flush=True);continue\n                    development=[p for p in train if assignment[p.stem]!=fold]\n                    validation=[p for p in train if assignment[p.stem]==fold]\n                    train_label_fold(development,validation,labels,cfg,fold,variant,folder,initial,device)\n            label_summary=aggregate_labels(output,variants,cfg,labels,assignment)\n        base.save_json(output/\'experiment_status.json\',dict(inference_done=run_inference,raw_replayed=run_inference and raw_replay,\n            label_training_requested=run_training,requested_folds=list(folds_to_run),requested_variants=list(variants),\n            competition_csv_generated=False,public_scores_used=False,automatic_model_selection=False,\n            inference_summary=inference_summary,label_summary=label_summary))\n        archive=analysis_archive(output)\n        print(\'OOF-only analysis saved:\',archive,flush=True)\n    return output\n', 'task4_cldice_ablation.py': '"""One controlled extension: L2 + 0.1 soft-clDice on the full-resolution head.\n\nThe three previous modules stay unchanged. Original L2 OOF files are read-only;\nonly the new five-fold arm trains. No test inference or competition submissions.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path, PurePosixPath\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\n\nimport task4_controlled_ablation as abl\n\nbase, v2 = abl.base, abl.v2\nREFERENCE = \'L2_reference\'\nMETHOD = \'T1_L2_cldice\'\n\n\n@dataclass\nclass StructureConfig(abl.AblationConfig):\n    cldice_weight: float = 0.1\n    skeleton_iterations: int = 10\n    skeleton_smooth: float = 1.0\n\n\ndef environment():\n    import PIL\n    import skimage\n    return dict(python=platform.python_version(), torch=str(torch.__version__),\n                torchvision=str(v2.torchvision.__version__), numpy=np.__version__,\n                opencv=cv2.__version__, pillow=PIL.__version__, skimage=skimage.__version__,\n                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)\n\n\ndef soft_erode(image):\n    return torch.minimum(-F.max_pool2d(-image, (3, 1), 1, (1, 0)),\n                         -F.max_pool2d(-image, (1, 3), 1, (0, 1)))\n\n\ndef soft_open(image):\n    return F.max_pool2d(soft_erode(image), 3, 1, 1)\n\n\ndef soft_skeleton(image, iterations=10):\n    """2D min/max-pool skeleton as in the authors\' differentiable formulation."""\n    skeleton = F.relu(image - soft_open(image))\n    for _ in range(iterations):\n        image = soft_erode(image)\n        delta = F.relu(image - soft_open(image))\n        skeleton = skeleton + F.relu(delta - skeleton * delta)\n    return skeleton\n\n\ndef soft_cldice_loss(logits, clean, iterations=10, smooth=1.0):\n    # Per-image reduction matches the reference\'s per-image Dice convention.\n    # Float32 morphology avoids half-precision reductions under mixed precision.\n    with torch.autocast(device_type=logits.device.type, enabled=False):\n        prob, clean = logits.float().sigmoid(), clean.float()\n        pred_skeleton = soft_skeleton(prob, iterations)\n        with torch.no_grad():\n            true_skeleton = soft_skeleton(clean, iterations)\n        axes = (1, 2, 3)\n        precision = ((pred_skeleton * clean).sum(axes) + smooth) / (pred_skeleton.sum(axes) + smooth)\n        recall = ((true_skeleton * prob).sum(axes) + smooth) / (true_skeleton.sum(axes) + smooth)\n        return (1 - 2 * precision * recall / (precision + recall)).mean()\n\n\ndef objective(outputs, pair, cfg):\n    reference = abl.supervised_loss(outputs, pair, \'L2_soft\', cfg.soft_boundary)\n    if cfg.cldice_weight == 0:\n        return reference, reference, reference.detach() * 0\n    topology = soft_cldice_loss(outputs[0], pair[:, :1], cfg.skeleton_iterations, cfg.skeleton_smooth)\n    return reference + cfg.cldice_weight * topology, reference, topology\n\n\ndef valid_probability(path, val_ids):\n    values = {}\n    with np.load(path, allow_pickle=False) as arrays:\n        expected = {f\'{mode}_{i}\' for mode in [\'none\', \'d4\'] for i in val_ids}\n        if set(arrays.files) != expected:\n            raise ValueError(f\'Unexpected OOF probability keys: {path}\')\n        for key in expected:\n            value = arrays[key]\n            if value.shape != (512, 512) or not np.isfinite(value).all() or value.min() < 0 or value.max() > 1:\n                raise ValueError(f\'Invalid probability: {path}:{key}\')\n            values[key] = value.copy()\n    return values\n\n\ndef safe_members(archive):\n    names = archive.namelist()\n    if len(names) != len(set(names)):\n        raise ValueError(\'Duplicate archive paths\')\n    for item in archive.infolist():\n        path = PurePosixPath(item.filename)\n        if (path.is_absolute() or \'..\' in path.parts or \'\\\\\' in item.filename or \':\' in item.filename\n                or (item.external_attr >> 16) & 0o170000 == 0o120000):\n            raise ValueError(\'Unsafe archive path\')\n    if archive.testzip() is not None:\n        raise ValueError(\'Archive CRC failed\')\n\n\ndef materialize_reference(source, output):\n    """Find a full label output; extract only JSON, L2 history/OOF and initializer."""\n    if source is None:\n        candidates = []\n        local = Path(\'/kaggle/working/task4_label_supervision_ablation\')\n        if (local / \'L2_soft/fold0/validation_probabilities.npz\').exists():\n            candidates.append(local)\n        if Path(\'/kaggle/input\').exists():\n            candidates += [p.parent for p in Path(\'/kaggle/input\').rglob(\'ablation_config.json\')\n                           if all((p.parent / f\'L2_soft/fold{f}/validation_probabilities.npz\').exists() for f in range(5))]\n        candidates = list(dict.fromkeys(p.resolve() for p in candidates))\n        if len(candidates) != 1:\n            raise ValueError(f\'Found {len(candidates)} full L2 outputs. Set BASELINE_RUN to the label-stage output folder. \'\n                             \'The lightweight ablation_analysis_results ZIP has no OOF probabilities.\')\n        return candidates[0]\n    source = Path(source).resolve()\n    if source.is_dir():\n        return source\n    if not source.is_file() or source.suffix.lower() != \'.zip\':\n        raise ValueError(\'BASELINE_RUN must be the full label-stage folder or a full backup ZIP\')\n    cache = Path(output) / \'reference_cache\'\n    required = {\'ablation_config.json\', \'execution_environment.json\', \'manifest.csv\'}\n    required |= {f\'L2_soft/fold{f}/{name}\' for f in range(5)\n                 for name in [\'completed.json\', \'history.csv\', \'validation_probabilities.npz\']}\n    optional = {\'pretrained_cache/resnet34-b627a593.pth\'}\n    with zipfile.ZipFile(source) as z:\n        safe_members(z)\n        if not required <= set(z.namelist()):\n            raise ValueError(\'Missing full L2 probabilities/records. A lightweight analysis ZIP cannot serve as baseline.\')\n        for name in sorted(required | (optional & set(z.namelist()))):\n            target = cache / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with z.open(name) as src, target.open(\'wb\') as dst:\n                shutil.copyfileobj(src, dst)\n    return cache\n\n\ndef verify_reference(source, expected, expected_config, assignment):\n    source = Path(source)\n    config = json.loads((source / \'ablation_config.json\').read_text())\n    if config != expected_config:\n        raise ValueError(\'Baseline config/source/dataset differs from the returned formal L2 experiment\')\n    for fold in range(5):\n        folder = source / \'L2_soft\' / f\'fold{fold}\'\n        done = json.loads((folder / \'completed.json\').read_text())\n        if done != expected[str(fold)] or done[\'epoch\'] != expected_config[\'config\'][\'epochs\']:\n            raise ValueError(f\'Baseline completed record differs: fold{fold}\')\n        if set(done[\'val_ids\']) != {i for i, f in assignment.items() if f == fold}:\n            raise ValueError(\'Baseline uses different folds\')\n        for name in [\'history.csv\', \'validation_probabilities.npz\']:\n            if base.file_sha(folder / name) != done[\'sha256\'][name]:\n                raise ValueError(f\'Baseline artifact hash mismatch: fold{fold}/{name}\')\n        history = read_table(folder / \'history.csv\')\n        if [int(r[\'epoch\']) for r in history] != list(range(1, done[\'epoch\'] + 1)):\n            raise ValueError(\'Incomplete baseline history\')\n        if [r[\'plan_sha256\'] for r in history] != done[\'plan_hashes\']:\n            raise ValueError(\'Baseline sampling history mismatch\')\n        valid_probability(folder / \'validation_probabilities.npz\', done[\'val_ids\'])\n    return config\n\n\ndef read_table(path):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        return list(csv.DictReader(f))\n\n\ndef train_fold(paths, validation, labels, cfg, fold, folder, initial, device, expected):\n    """Same L2 loop/seed/scheduler; the only optimization change is objective()."""\n    folder.mkdir(parents=True, exist_ok=True)\n    base.seed_all(cfg.seed + 1009 * fold)\n    dataset = abl.PairedPatches(paths, labels, cfg, fold)\n    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0,\n                        pin_memory=device.type == \'cuda\')\n    model = v2.ResNet34UNet(initial).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    initial_hash = abl.digest_json({k: hashlib.sha256(v.detach().cpu().numpy().tobytes()).hexdigest()\n                                   for k, v in model.state_dict().items()})\n    train_ids, val_ids = [p.stem for p in paths], [p.stem for p in validation]\n    if (initial_hash != expected[\'initial_model_hash\'] or train_ids != expected[\'train_ids\']\n            or val_ids != expected[\'val_ids\']):\n        raise ValueError(\'Initial model/split differs from the paired L2 reference; no training started\')\n    optimizer = torch.optim.AdamW([\n        {\'params\': model.encoder.parameters(), \'lr\': cfg.encoder_lr},\n        {\'params\': [p for n, p in model.named_parameters() if not n.startswith(\'encoder.\')], \'lr\': cfg.decoder_lr}],\n        weight_decay=cfg.weight_decay)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg.amp and device.type == \'cuda\')\n    history, first, step = [], 1, 0\n    last = folder / \'last.pt\'\n    if last.exists():\n        saved = torch.load(last, map_location=\'cpu\', weights_only=False)\n        if (saved[\'method\'] != METHOD or saved[\'config\'] != asdict(cfg)\n                or saved[\'initial_model_hash\'] != initial_hash\n                or saved[\'train_ids\'] != train_ids or saved[\'val_ids\'] != val_ids):\n            raise ValueError(\'Structural checkpoint identity mismatch\')\n        model.load_state_dict(saved[\'model\'])\n        ema.load_state_dict(saved[\'ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        first, step, history = saved[\'epoch\'] + 1, saved[\'step\'], saved[\'history\']\n        base.restore_rng(saved[\'rng\'])\n        del saved\n        print(f\'{METHOD}/fold{fold}: resume at {first}/{cfg.epochs}\', flush=True)\n    for epoch in range(first, cfg.epochs + 1):\n        begin = time.perf_counter()\n        model.train()\n        plan_hash = dataset.set_epoch(epoch)\n        if plan_hash != expected[\'plan_hashes\'][epoch - 1]:\n            raise ValueError(f\'RGB sampling/augmentation plan differs: fold{fold}, epoch{epoch}\')\n        losses, reference_losses, topology_losses = [], [], []\n        for image, pair in loader:\n            v2.set_lrs(optimizer, step, len(loader), cfg)\n            image, pair = image.to(device, non_blocking=True), pair.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == \'cuda\'):\n                loss, reference, topology = objective(model(image), pair, cfg)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite structural loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.)\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            if scaler.get_scale() >= old_scale:\n                v2.update_ema(ema, model, cfg.ema_decay)\n            losses.append(float(loss.detach()))\n            reference_losses.append(float(reference.detach()))\n            topology_losses.append(float(topology.detach()))\n            step += 1\n        monitor_c, monitor_p = None, None\n        if epoch % cfg.monitor_every == 0 or epoch == cfg.epochs:\n            scores = []\n            for path in validation:\n                image = cv2.resize(v2.read_rgb(path), (512, 512), interpolation=cv2.INTER_AREA)\n                prob = v2.predict_pair(ema, image, cfg, device, False)[\'none\']\n                scores.append((base.dice_score(prob >= .5, base.strict_label(labels[path.stem])),\n                               base.dice_score(prob >= .5, base.converter_label(labels[path.stem]))))\n            monitor_c, monitor_p = map(float, np.mean(scores, axis=0))\n        row = dict(epoch=epoch, loss=float(np.mean(losses)), reference_loss=float(np.mean(reference_losses)),\n                   soft_cldice_loss=float(np.mean(topology_losses)), monitor_clean_05=monitor_c,\n                   monitor_proxy_05=monitor_p, plan_sha256=plan_hash, seconds=round(time.perf_counter()-begin, 3))\n        history.append(row)\n        if shutil.disk_usage(folder).free < 1200 * 1024**2:\n            raise OSError(\'Less than 1.2 GiB free; preserve checkpoint and free space before resuming\')\n        base.atomic_checkpoint(last, dict(model=model.state_dict(), ema=ema.state_dict(),\n            optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), epoch=epoch, step=step, history=history,\n            rng=base.rng_state(), train_ids=train_ids, val_ids=val_ids, method=METHOD,\n            config=asdict(cfg), initial_model_hash=initial_hash))\n        abl.write_table(folder/\'history.csv\', history)\n        print(f\'{METHOD}/fold{fold} {epoch:03d}/{cfg.epochs} \'\n              f\'loss={row["loss"]:.4f} cl={row["soft_cldice_loss"]:.4f} seconds={row["seconds"]}\', flush=True)\n    base.atomic_checkpoint(folder/\'final_ema.pt\', dict(model=ema.state_dict(), epoch=cfg.epochs,\n        method=METHOD, config=asdict(cfg), train_ids=train_ids, val_ids=val_ids, initial_model_hash=initial_hash))\n    values = {}\n    for path in validation:\n        image = cv2.resize(v2.read_rgb(path), (512, 512), interpolation=cv2.INTER_AREA)\n        for mode, prob in v2.predict_pair(ema, image, cfg, device).items():\n            values[f\'{mode}_{path.stem}\'] = prob\n    np.savez_compressed(folder/\'validation_probabilities.npz\', **values)\n    names = [\'last.pt\', \'final_ema.pt\', \'history.csv\', \'validation_probabilities.npz\']\n    base.save_json(folder/\'completed.json\', dict(epoch=cfg.epochs, method=METHOD, config=asdict(cfg),\n        train_ids=train_ids, val_ids=val_ids, initial_model_hash=initial_hash,\n        plan_hashes=[r[\'plan_sha256\'] for r in history], sha256={n:base.file_sha(folder/n) for n in names}))\n    del model, ema, optimizer, scaler, dataset, loader\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n\n\ndef verify_new_fold(folder, cfg, expected):\n    done = json.loads((folder/\'completed.json\').read_text())\n    if (done[\'method\'] != METHOD or done[\'epoch\'] != cfg.epochs or done[\'config\'] != asdict(cfg)\n            or any(done[k] != expected[k] for k in [\'initial_model_hash\',\'train_ids\',\'val_ids\',\'plan_hashes\'])):\n        raise ValueError(f\'New method fold identity/pairing mismatch: {folder}\')\n    for name, digest in done[\'sha256\'].items():\n        if base.file_sha(folder/name) != digest:\n            raise ValueError(f\'New method artifact changed: {folder/name}\')\n    valid_probability(folder/\'validation_probabilities.npz\', done[\'val_ids\'])\n    return done\n\n\ndef evaluate_probabilities(name, probabilities, truth, assignment):\n    rows = []\n    grid = []\n    calibration = []\n    for mode, values in probabilities.items():\n        for threshold in [.1, .4, .5]:\n            rows += abl.evaluate_masks(f\'{name}_{mode}_t{threshold:g}\',\n                {i:p>=threshold for i,p in values.items()}, truth, assignment)\n        for target in [\'clean\',\'proxy\']:\n            group = []\n            for threshold in abl.CALIBRATION_GRID:\n                scores = [float(np.mean([base.dice_score(p>=threshold, truth[i][target])\n                    for i,p in values.items() if assignment[i]==fold])) for fold in range(5)]\n                r = dict(method=name, mode=mode, target=target, threshold=threshold,\n                    mean_dice=float(np.mean(scores)), worst_fold_dice=min(scores),\n                    **{f\'fold{f}\':v for f,v in enumerate(scores)})\n                group.append(r)\n            grid += group\n            best = max(group, key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-.5)))\n            calibration.append(dict(**best,selection_caveat=\'OOF-tuned development estimate; not nested or independent validation\'))\n    return rows, grid, calibration\n\n\ndef reference_probabilities(source, expected):\n    probabilities = {mode:{} for mode in [\'none\',\'d4\']}\n    for fold in range(5):\n        values = valid_probability(Path(source)/\'L2_soft\'/f\'fold{fold}\'/\'validation_probabilities.npz\',\n                                   expected[str(fold)][\'val_ids\'])\n        for mode in probabilities:\n            for i in expected[str(fold)][\'val_ids\']:\n                probabilities[mode][i] = values[f\'{mode}_{i}\']\n    return probabilities\n\n\ndef compare_reference_metrics(rows, expected_rows):\n    lookup = {(r[\'experiment\'].replace(REFERENCE,\'L2_soft\',1),r[\'target\'],r[\'id\']):r for r in rows}\n    metrics = [\'dice\',\'precision\',\'recall\',\'centerline_precision\',\'centerline_recall\',\'cldice\',\n               \'foreground_fraction\',\'thin_clean_centerline_recall\']\n    if len(expected_rows) != 240 or len(lookup) != 240:\n        raise ValueError(\'Need the full 240-row frozen L2 baseline metric table\')\n    drift = {k:0. for k in metrics}\n    for expected in expected_rows:\n        actual = lookup[expected[\'experiment\'],expected[\'target\'],expected[\'id\']]\n        for k in metrics:\n            drift[k] = max(drift[k],abs(float(actual[k])-float(expected[k])))\n    if any(v>1e-10 for v in drift.values()):\n        raise ValueError(f\'Baseline metric replay differs from returned L2 tables: {drift}\')\n    return drift\n\n\ndef previews(output, paths, truth, probabilities):\n    folder=output/\'oof_error_previews\';folder.mkdir(exist_ok=True)\n    images={p.stem:p for p in paths}\n    for i in sorted(images,key=int):\n        c,p=truth[i][\'clean\'],truth[i][\'proxy\']\n        a=probabilities[REFERENCE][\'d4\'][i]>=.5\n        b=probabilities[METHOD][\'d4\'][i]>=.5\n        rgb=cv2.resize(v2.read_rgb(images[i]),(512,512),interpolation=cv2.INTER_AREA)\n        def mask(m):return np.repeat((m.astype(np.uint8)*255)[...,None],3,axis=2)\n        def errors(m):\n            view=mask(m&c)\n            view[m&~c]=[235,65,65]\n            view[c&~m]=[70,125,245]\n            return view\n        change=mask(a&b)\n        change[b&~a]=[70,200,100]\n        change[a&~b]=[230,110,70]\n        panels=[rgb,mask(c),mask(p),change,mask(a),mask(b),errors(a),errors(b)]\n        names=[f\'OOF {i} RGB\',\'Clean C\',\'Converter proxy P\',\'Green added / orange removed\',\n               \'L2 reference D4 >=0.5\',\'L2 + clDice D4 >=0.5\',\'Ref vs C: red FP / blue FN\',\'New vs C: red FP / blue FN\']\n        canvas=Image.new(\'RGB\',(2048,1080),\'white\');draw=ImageDraw.Draw(canvas)\n        for j,(panel,name) in enumerate(zip(panels,names)):\n            x,y=(j%4)*512,(j//4)*540\n            draw.text((x+6,y+7),name,fill=\'black\')\n            canvas.paste(Image.fromarray(panel),(x,y+28))\n        canvas.save(folder/f\'oof_{int(i):02d}.png\')\n\n\ndef aggregate(output, source, cfg, labels, paths, assignment, expected, make_previews=True):\n    complete=[]\n    for fold in range(5):\n        folder=output/METHOD/f\'fold{fold}\'\n        if (folder/\'completed.json\').exists():\n            verify_new_fold(folder,cfg,expected[str(fold)])\n            complete.append(fold)\n    if complete != list(range(5)):\n        return [],dict(complete_folds=complete,complete_20_image_oof=False,\n                       partial_results_are_not_full_oof=True,paired_initialization_and_plans=True)\n    truth=abl.targets(labels)\n    probabilities={REFERENCE:reference_probabilities(source,expected),METHOD:{mode:{} for mode in [\'none\',\'d4\']}}\n    for fold in range(5):\n        values=valid_probability(output/METHOD/f\'fold{fold}\'/\'validation_probabilities.npz\',expected[str(fold)][\'val_ids\'])\n        for mode in [\'none\',\'d4\']:\n            for i in expected[str(fold)][\'val_ids\']:\n                probabilities[METHOD][mode][i]=values[f\'{mode}_{i}\']\n    rows,grid,calibration=[],[],[]\n    for method,probs in probabilities.items():\n        if any(set(values)!=set(assignment) for values in probs.values()):\n            raise ValueError(\'Incomplete OOF coverage\')\n        r,g,c=evaluate_probabilities(method,probs,truth,assignment)\n        rows+=r;grid+=g;calibration+=c\n    summary,folds=abl.summarize_rows(rows)\n    abl.write_table(output/\'structure_per_image.csv\',rows)\n    abl.write_table(output/\'structure_per_fold.csv\',folds)\n    abl.write_table(output/\'structure_summary.csv\',summary)\n    abl.write_table(output/\'structure_threshold_grid.csv\',grid)\n    abl.write_table(output/\'structure_calibration_development.csv\',calibration)\n    lookup={(r[\'experiment\'],r[\'target\'],r[\'id\']):r for r in rows}\n    paired,paired_summary,paired_folds=[],[],[]\n    metrics=[\'dice\',\'precision\',\'recall\',\'centerline_precision\',\'centerline_recall\',\'cldice\',\n             \'foreground_fraction\',\'thin_clean_centerline_recall\']\n    for mode in [\'none\',\'d4\']:\n        for threshold in [.1,.4,.5]:\n            for target in [\'clean\',\'proxy\']:\n                for key in metrics:\n                    ds=[];fs=[]\n                    for i in sorted(assignment,key=int):\n                        new=lookup[f\'{METHOD}_{mode}_t{threshold:g}\',target,i][key]\n                        old=lookup[f\'{REFERENCE}_{mode}_t{threshold:g}\',target,i][key]\n                        delta=float(new-old);ds.append((i,delta))\n                        paired.append(dict(mode=mode,threshold=threshold,target=target,metric=key,\n                                           id=i,fold=assignment[i],delta=delta))\n                    for fold in range(5):\n                        delta=float(np.mean([d for i,d in ds if assignment[i]==fold]));fs.append(delta)\n                        paired_folds.append(dict(mode=mode,threshold=threshold,target=target,metric=key,fold=fold,delta=delta))\n                    paired_summary.append(dict(mode=mode,threshold=threshold,target=target,metric=key,\n                        mean_delta=float(np.mean([d for _,d in ds])),positive_images=sum(d>0 for _,d in ds),\n                        negative_images=sum(d<0 for _,d in ds),positive_folds=sum(d>0 for d in fs),\n                        negative_folds=sum(d<0 for d in fs),min_fold_delta=min(fs),max_fold_delta=max(fs)))\n    abl.write_table(output/\'structure_paired_per_image.csv\',paired)\n    abl.write_table(output/\'structure_paired_per_fold.csv\',paired_folds)\n    abl.write_table(output/\'structure_paired_summary.csv\',paired_summary)\n    if make_previews:\n        previews(output,paths,truth,probabilities)\n    return summary,dict(complete_folds=complete,complete_20_image_oof=True,\n        paired_initialization_and_plans=True,reference_source=\'verified original L2 OOF probabilities\',\n        comparison=\'new fixed final80 EMA versus old fixed final80 EMA; no outer-fold checkpoint selection\',\n        automatic_model_selection=False,public_scores_used=False,competition_csv_generated=False)\n\n\ndef analysis_archive(output):\n    path=output/\'structure_ablation_analysis_results.zip\'\n    temporary=path.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in output.rglob(\'*\'):\n            if (p.is_file() and p.suffix in {\'.json\',\'.csv\',\'.png\',\'.md\',\'.py\'}\n                    and \'reference_cache\' not in p.relative_to(output).parts and p.name!=\'training.lock\'):\n                z.write(p,p.relative_to(output).as_posix())\n    with zipfile.ZipFile(temporary) as z:\n        if z.testzip() is not None:raise IOError(\'Analysis ZIP CRC failed\')\n    os.replace(temporary,path)\n    return path\n\n\ndef restore_output(source,output):\n    source=Path(source).resolve()\n    if source==output:return\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(\'Resume requires a new empty OUTPUT_DIR, or use the existing working output directly\')\n    if output.is_relative_to(source):\n        raise ValueError(\'Resume destination cannot be inside its source\')\n    if source.is_dir():\n        if not (source/\'structure_config.json\').exists():raise ValueError(\'Not a structure-stage resume folder\')\n        shutil.copytree(source,output,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\',\'reference_cache\'))\n    elif source.is_file() and source.suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(source) as z:\n            safe_members(z)\n            if (\'structure_config.json\' not in z.namelist()\n                    or not any(n.endswith(\'/last.pt\') for n in z.namelist())):\n                raise ValueError(\'Resume needs full structure output with checkpoints, not the lightweight analysis ZIP\')\n            output.mkdir(parents=True,exist_ok=True)\n            for item in z.infolist():\n                if item.filename.endswith((\'training.lock\',\'.tmp\',\'.zip\')):continue\n                z.extract(item,output)\n    else:\n        raise ValueError(\'RESUME_FROM must be a full structure-stage folder/backup\')\n\n\ndef run_structure(data_dir,output_dir,cfg,baseline_run,expected,expected_config,\n                  expected_rows=None,folds_to_run=(0,1,2,3,4),weights_path=None,resume_from=None,\n                  engineering=False):\n    if cfg.cldice_weight!=.1 or cfg.skeleton_iterations!=10 or cfg.skeleton_smooth!=1.:\n        raise ValueError(\'This experiment fixes clDice weight=0.1, iterations=10, smooth=1; no loss-weight search\')\n    if any(asdict(cfg)[k]!=v for k,v in expected_config[\'config\'].items()):\n        raise ValueError(\'Training recipe must match paired L2 reference exactly\')\n    if not engineering and (cfg.epochs!=80 or cfg.patches_per_epoch!=256 or cfg.batch_size!=8 or expected_rows is None):\n        raise ValueError(\'Formal run requires the frozen 5-fold80 recipe and frozen reference metrics\')\n    if len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):\n        raise ValueError(\'Invalid requested folds\')\n    output=Path(output_dir).resolve()\n    if resume_from and not (output/\'structure_config.json\').exists():restore_output(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n        reference_source_sha256=base.file_sha(abl.__file__),v2_source_sha256=base.file_sha(v2.__file__),\n        base_source_sha256=base.file_sha(base.__file__),reference_expected_records_hash=abl.digest_json(expected),\n        reference_expected_config=expected_config,engineering_only=engineering)\n    marker=output/\'structure_config.json\'\n    previous=json.loads(marker.read_text()) if marker.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):\n        raise ValueError(\'Changed source/config/baseline: use a new output directory\')\n    for key,name in [(\'source_sha256\',abl.__file__),(\'v2_source_sha256\',v2.__file__),(\'base_source_sha256\',base.__file__)]:\n        if expected_config[key]!=base.file_sha(name):raise ValueError(\'Original L2 helper source changed\')\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    with v2.run_lock(output):\n        train,_,labels,assignment,_,audit=v2.audit_v2(base.find_data(data_dir),output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if identity[\'data_fingerprint\']!=expected_config[\'data_fingerprint\']:\n            raise ValueError(\'Dataset fingerprint differs from the frozen L2 reference\')\n        if previous and previous!=identity:raise ValueError(\'Dataset identity changed\')\n        source=materialize_reference(baseline_run,output)\n        if source==output or output.is_relative_to(source):\n            raise ValueError(\'Keep the new output outside the read-only baseline output\')\n        verify_reference(source,expected,expected_config,assignment)\n        if expected_rows is not None:\n            truth=abl.targets(labels)\n            replay_rows,_,_=evaluate_probabilities(REFERENCE,reference_probabilities(source,expected),truth,assignment)\n            drift=compare_reference_metrics(replay_rows,expected_rows)\n        else:drift=None\n        reference_env=json.loads((source/\'execution_environment.json\').read_text()) if (source/\'execution_environment.json\').exists() else {}\n        current_env=environment()\n        differences={k:dict(reference=reference_env.get(k),current=current_env.get(k))\n                     for k in set(reference_env)|set(current_env) if reference_env.get(k)!=current_env.get(k)}\n        base.save_json(marker,identity)\n        base.save_json(output/\'reference_audit.json\',dict(source=str(source),all_five_probability_hashes_verified=True,\n            reference_records=expected,baseline_metric_max_absolute_drift=drift,\n            environment_differences=differences,reference_environment=reference_env))\n        base.save_json(output/\'execution_environment.json\',current_env)\n        runtime=output/\'runtime_source\';runtime.mkdir(exist_ok=True)\n        for path in [__file__,abl.__file__,v2.__file__,base.__file__]:\n            shutil.copyfile(path,runtime/Path(path).name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        if device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for the new structural-loss training arm\')\n        if weights_path is None and (source/\'pretrained_cache/resnet34-b627a593.pth\').exists():\n            weights_path=source/\'pretrained_cache/resnet34-b627a593.pth\'\n        initial,provenance=v2.resolve_weights(cfg,output,weights_path)\n        if provenance[\'sha256\']!=\'b627a593bcbe140c234610266fe4f8ae95ea42fc881d091c9b6052e6b1d0590f\':\n            raise ValueError(\'Expected the same official generic ImageNet initializer\')\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        for fold in folds_to_run:\n            folder=output/METHOD/f\'fold{fold}\'\n            if (folder/\'completed.json\').exists():\n                verify_new_fold(folder,cfg,expected[str(fold)])\n                print(f\'{METHOD}/fold{fold}: completed, reusing\',flush=True)\n                continue\n            development=[p for p in train if assignment[p.stem]!=fold]\n            validation=[p for p in train if assignment[p.stem]==fold]\n            train_fold(development,validation,labels,cfg,fold,folder,initial,device,expected[str(fold)])\n        summary,status=aggregate(output,source,cfg,labels,train,assignment,expected)\n        status.update(requested_folds=list(folds_to_run),new_training_arms=1,\n            reference_retrained=False,engineering_only=engineering,automatic_model_selection=False,\n            public_scores_used=False,competition_csv_generated=False,test_predictions_generated=False,\n            environment_differences=differences,summary=summary)\n        base.save_json(output/\'structure_status.json\',status)\n        print(\'OOF-only analysis:\',analysis_archive(output),flush=True)\n    return output\n'}
EXPECTED_L2 = {'0': {'epoch': 80, 'variant': 'L2_soft', 'train_ids': ['1', '3', '4', '5', '7', '8', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20'], 'val_ids': ['2', '6', '9', '10'], 'initial_model_hash': '554e9ecba699a48f180c59f5dd7dbb998e1a9a24e21b13684d40192494495172', 'plan_hashes': ['242132163dc7dc6439a5f4ac1728a54454b7fc46202b4403f18319a46196eb1d', 'b7e9fc189550613fe3c976ed4b8a2328ec95e4d362ccac11569b0bfc86108134', '5ee347a5cca68958928e62d92883861ae2150e0b03057bff56e314a3760e6757', '90ecf03fbfae9c4a12ddfcf8d7e8ef1c3771706f0eb082eaa5b8e28759e8c4b7', '42b0a72173d6171ce5bb573c960e27df39156937e4da70b8e6b11d2680867cc0', '2aeb8a559966023a7c59784c87eb6d3d7385d4181e90383fa580e79b08acf5c3', '4fb00eff521d20f113edaf159fdb2c67b4c273ffaed15b1e1ce1d2400103bc66', 'f962c6b2ddb9c01ba1121f50343cc28ed8f473f97b3c8f8bb42bfe662c908ed8', 'a7ece28bdce15d69b7440f8ba80bdd28c3088bce53fd3618b578ec5ab0815272', 'e40d633cffe41509c03296db73ba2ce9a217b80206fbda45a8c86718415b1d83', '340940a3b3cf4fdb7790a997fcb7dc426f95da47ce46398415da543e0d0db1ef', '18ef8ba96305100fbf5e933431b1ea286ff00acbc7e50ec769170be050b5d0fe', 'e87dbf1b20e01be162457ec8accae7e706c199fb6af6b4693aefd6d6b5fe7484', '9fbae9a9e0df5ba948357bbb0990da7f74c2b0fee6ec5b8aa92ad1756b73c69d', 'ae3dc00ce1420c373428ff9e3357eee55ddcb648a8e2eb685ee5e207acacd381', 'c2c8e09f664ed532fcfeabfed957a76586b780523fe0ef53912091134b0c1abe', '51c7e9a4eab1dab4335c3af50dddc142639ad34672c19bd60027dde9eb2daacd', '3f7e1bf10a3f17d084d7efc66584c05788b9f6f99bb306f8e17db67a5e9df8a1', 'd5c048a299397d6f22557ba851e3a5ca5959bb14acbd499eac55c1f62889bacc', '920b03ff794873f2a745d84388f9337a8e51f2901dd18c79faf7bf0290c76586', '34e3ea1d6e9f57a7122c9e43168367d5b3a54f474337127413f4803ef01eb962', '94ffaee6dcdf1794bf118ed31d52c962158a41a54677ee0a95f6b02172a0cba2', '86bf84c34952ad8ab2510f7d5e103efa97642c88b83f60520256e9b33b7419ba', '4131b964e0080fbccaf4d345b6e2bd52206057044e2fb54c3272a94cf0fba923', 'efd66094c31bf162efbf6a85fff421c5ea5507f077f4a2ec11a890457ddd8c40', 'fbb119259a476094f998d424f256c6b151de856427cd4e632cec4cc69f4eeca1', 'c10e3efdca0e96421345c137502eb970965b18c9aa8d38216c3e23dad8c88917', '8fbeb1269c77afdf971bd54609f323d529b4d84edefaf0b6e654e0951dd8f817', '2c3d25529f682fda51a5e0b34747767b6b5f839342dfbbc402e6461b445c4aae', 'c9fe0c417660f49fa0d719adea78b29e377f77539ca3d742e15c24e2f7da14cd', '4a3ec9d6e65c822c2b7b8baa2b6fda4a29f8f45f0f0ff45211e134e4ae8c0902', '60ffa348398aa6a2888cd796e10c398c39bf0554717969280bdbbc06dedb208c', '34ff2734ab5f1c336f89d53ddddccfb1f22bac63bfdec843c0c083fbf43331e7', '874eb170e66ab11968e057357664284af2252d3699baa2ea8668644d3cebc122', 'b30d2ed94aa6fde7cf751f75d91cd8bb132fb4f15d3495ac54276e4413672a0b', '2fb3b866729f98120bb0f254143cfb991fc31cc8a49f54aebfa56a65599e7e91', '0860fec0c1b2fa6bb8edc191841ba577bb4ca3f5bbac3e2ae41ad765f0e8a55a', '03a936d3d6c27d73d1bea945b5a807e7338387e9e436ec5feae1f6c550f00dc5', '40dc73333098f7a9b8f1b2bf95284c6dea62490cd8056d74e687f7414995c1ce', 'a59259a07526cb4ee7f8ef841649b49635cb4db0feea80b55ec1f76839566d31', '194836dc10e2fb7de79ef42652e03956e4b8316d43de1de71585ff01c6f628a1', 'd7db719738a8581c97672c72f66e19a9e743c04529e6d0dfade7302e46195a7d', '33c7679bc09370e5af7a477d45073729dfa34850b5f97b65ccd95f1ae1243049', '04f278bf55b97ff603477dfc463c0bcb8899f82e1144dceb053b2ce98330aa0a', 'e234e7457a9be1f140302820024f33520b3af2191413ecf53d9e34bcb92c3883', '3da5fd9fd2bd41c192d4a6e07a7a27cab202133b6fd322b5f07e68759722358e', 'bfbc208b7f05c828c37f266956d52fc2ffcbc2991144e278f0f5677d975be49b', '423967fbd9fde6e276afb6178b30318b59e079eb981c53eea01249fd42c075ab', '809a5b5e1ea936fd7688253ec57a82cec5370357c6d6ce3f3f8927d16805df67', 'c1da6f70b2c4e1b494f2cf58b711913d617e8b65f39bf2ab9f147d35ee9c1194', 'd3e022f90ed51962c8affeb742ed6a70e11de3666cce772c3e0d8bf092954827', '37de2378bfb5ce5db9ddd3f9253c8e42bda5e61a8f0658660045fbe7a75cb213', '43253c142e3858c2ec81ea58957ba67e595e4c6a32286f9758ac1d7415a21247', '78a0b40a78ba6c1c4c8776966bb1ae0757c803a4abc5ba903321490acf2363b9', 'e8bdde6a690ece792dabd2cc5cbae8612a70bb29a13ba632188487af103dae19', '172a6bff04b0f806dc7035397b12e9d9203f59bfca6425b2850307f755370a67', '7c2840869466e70bf334380a90c3c8dc8b0dfe2f862ee288e5166edd665cd589', '8c9fca6d32b7a09c3701f05e4a955cfd4429bcec8a0ebab65244f6387fe445aa', 'f665c155cbe1579054f4c51ccd1f15d38cd78cdbce9b883a976593cf3020f964', 'f7f4c0bbd888c6f828e1bdb1daf35c5f4f2a3231d25958c973cc73eb1f1b10c7', '6adef9920b8a2c87f2f7fef3d1a5b3a6b3064e62846978fb488e880c2ae56c29', '0a2a02ebe1a5eb50c47dc7a2dd0fbb3fc4fb8b1e66b095ad87c0c35fb8d4e09f', 'def5b202586579806f84212ea09a536d3aad0aa2e0a9d6c5147f02ea4b26aa8d', '58b2b41d79bca0ef3937ca0239fa8f9a3f4e6f9b01dd28e38cfe5506efd031a3', '5b43418ea5f0d9dcae655bf149db2f6f638f67e3ac8379f6bed0edea31fdb456', '1e8b61a43396869a42d72c0dd0f7b9e0856fb32f1fc1ac5b9d931255de12b7c3', '0932a7b36c82e0f075c08e43dbdc51a70ffdc866a911563c99728265d744ebdb', 'bea97340820a0d1989cbecfc19f402a4a42f361c0a7079ffd0e8b74843155e87', '45d8e1f16530bf82eac6c4727b9ce0111feb1a3cfb5889622cd45d56d4d90064', '2cf2af5e42a47a75a48c119751fd932f3cbe6dd109f405842503799352642307', 'c4537a7dcdd21413c19e467b6e2d7d50589e8f332b58e8cb27ac2d3241504c0b', '35ec3145f111cdceb0f6a4ff26ba597b499ce8b1b7201897310d7cef257a1031', '8b43eef27448d5216750440ce377a064bf94cba4d50847d6141742fd3d534bf8', '6719fb66607dacbcc8bc6d555345054a45eeacf9f24e831c0fe1b5e65fe728ae', '5d0ff59383857dab0d8c63b550ef652a8896adfb2d9d2556af1ffafedecbec38', '57a1a1baac01dafd7e2c9003158366b8cbf293c080a93d2ebbb9b31c51f89c09', 'b181619045fc9dd5646b515802442c81bb7896d64a2a40eca163ff81c44551be', 'c029f8c7f5cbe900b2107505eee4e37a3f58541e3a6525b6488446f3b16a8ae5', '0880587a0f435d163cb4910835ad44cf32cf2f40b2fd770bbed3e24379c20d02', '81bae5930b366991e5b91e48c575969e6165c170e3b7fe5a85c83bafe0bf29b0'], 'sha256': {'last.pt': '64475820adc5a2c045c6077203a24fd504dc5dbe51a29878ee1cde39c43a135e', 'final_ema.pt': '7202a51eba5e6b4fdc11e12a40f68a4ebae88c674a77e8aa8edb766d34312eca', 'history.csv': 'c312f1cd150a69e29efe8d14af0bce4fe7aadcb10b0f629b8657d89285d029f5', 'validation_probabilities.npz': '23ece0d6957e19a2ba93ece8fb01512648dce4fee32eddbe239725d8197c401b'}}, '1': {'epoch': 80, 'variant': 'L2_soft', 'train_ids': ['1', '2', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '15', '16', '18', '19'], 'val_ids': ['3', '14', '17', '20'], 'initial_model_hash': 'de11a410772039ba0b124d230bab99f122be7de14c073e5f90f98b59dd59b695', 'plan_hashes': ['c2fe080e2fcdab031905af5dc0fd6059d93ca74a8af01fa54002eeeab1bcef8c', 'e5f970fb2d8f41a826a7f102fcd9bd03cc5bf597156b22f1c972ce6de6d0a57a', '9750469041100b41b9a530581587c8359c9a31a7c568701c6353270925091780', '33a0f60d0f0e0a3836627c307e038693eda3399c5be0e5e7c33bdbc5ea2f79ee', 'b5ce08802371411cb8da7357d80aefeb9ab5c6a4ec80e6b278fb98e51f3457aa', '5ca360cf09ee38e2cf59a9e3a15b84b15156dcd4e5e5647fe2e052f3f7c88a4f', '9554e0f5c7390a7676aaf3f9619830e95d934b6eafa6ce9622486582d5e84da3', 'c87a21a82776c40195c3075133a7cf3ee231450b40e67e40718276fbec49693c', 'ccc9e0155ada69edff16d0a1d935fb94087c518d50c83af6c78313cd0e5b1887', '13ce5f5a75a639c792e021bbdc1206bdd838cb382f61bf04998b9a9e0c47c6eb', 'dbc21728c975d76f84f408eb2edffc125b810a3adb40b105a0e53a1c5849b6f1', '04a01f500d0a9747501b087f6cb7a90efda720943c42c551d10200d17ecc490e', 'ca1726c891370a6d763414131063de9927ccea456fe08a3fea35d5b2c2ce7a73', '6c4cf9dc4cfc166db48f496309309910a01e9cd41ad3c9b1cf00019f8ddbe485', '45b9fd007d47c5d6166fb2790c894b81c94ccd2c9f91900b0c84622a63e3959c', '54a1f9aeaeb56dcced4c7577231806684d59ffb7dbd2aa095b5440ed45d21065', '7bdd33b2150876bb3929ea6772c0a226be5e3b076f854597b8244671a552ecfd', 'eea186cca6de68e34550b1e5753acf55f6fbe6f00ad0c83339483f07948dc6de', 'fbbf92e6e22c98b5dd01553a622861009963c224b111f781cb227cc555e34b85', 'f1626fddf7db898d7b9bcf38dafd3316a3eabd0724b3cd66e7046b4c8afd4589', 'becd9d1960eaab45e694221cae1d6f21f4ca503673ec83d82946e18851e61182', '62d7cd6437e562895d151d74d9ff27e48bd7dbafca6657035bc5c9c4f43f467d', 'b2bbb490f178a6d4096fcda4e74de13b6699e450b61d258f234eeaae0d6b51f4', 'f32b871ae3e91eece1c0a49d422038c32ccbf99d7b9af722cb12e3f5400040e2', 'e34190e88a1b2cde8528ab72af509c3d701bb511bf2fcf536e3fa29ae280088e', '8dd4c22a304583ab460b6934054c11dfcd87e1bf7c14d2710cfc55fb6c7fb8e7', 'bb9f4eb9cc43efd7056be575594267027a8a1f9b14978239079aaf831541376d', '9f8f5300d65aa4f8852cc50a28ac66264596cecd28a510d8d51ac6f029c6ebf2', '423feb5db827f43539930e1e3b2e96895de2cba88f46512b5d4be67cc1f7d8ef', '004acf252284cf62968f098892fb2f6ddfb53874b3b163d3639baba1a0b9d57a', 'd679cbafc17a858df0a94b31d5b32729820dc381ab4472a52a5f2c1f68c26caf', 'd832de421d9e87fd9cb8ec4ce9edf0a77d2d600014c54f561c483127e51b1c50', '2105eb2e732b156bf893d9da4abacfefe128a7382d5d9b9cb569afa8b989810b', 'dea53cc49113e12dfaec52aac82f4265851a7d6a75cf15ef3423549cf3e059b3', '90252974060ad9a7ee8a3a2888c1536cca17fc1149002d69227e416e45b48a06', '08d9bf9a76ff00ab0190944320961cb7d513787082d53d41db2788c415e4005b', '8662037bb4f695cd06b1b0f9f2bec18340d071b70a642a66f7dfb94b550c14f9', 'bfa0040f0024a98e0a9e4e3ed92d79f48d75063e5ca0fb68860ab69845cee279', 'cf1c98bc1324f49d34fb14241fdb45525eb289d88ba9d6680c146fd3fdd9ea42', '5277cd3d9b9904d24fe75421e68fe7db683828bb12a70f5dd6a8018fe8dabecf', '0793510783ccbcb15964532989835e6dc2e9d588140d6455180f84ba55fa1165', '49b67b75c15614c87fea5620e7b11294ff8710c91e4e0b51e5b120a1c1303a5f', '5221eb046bc31a9ee67dbc2a040bbf73a3e4f01b8f2eca071d0827c6bf75d523', '6325c3fdb4587bd3a6b5166f5e6e538654318afa4177afc550aad9903b7ee9dd', 'cfaa05d6234b70cf7978f919cc3f14a77bcc5206b536323900b2d6e6ea581274', '6bcacbfa5bd20531f456bd3915436e2ea75de6b24353180c69db2afc8e31e223', '8abb18ad57a6940147990bb0e200af810053821d38e38602e5460bf189dc0947', '89c140e63d89e7c824771029668af1affc43547e2fc97fcac258f4dfdf1ba612', '70b1ff2c69f99198bb41c72e263b8afb936bbbe3552378c9fb9363900ae04ac4', 'a66c046476db6e98b8bd0426db0a8d9049d485e5b0c3fa92820e836c677221ec', '15d81d90bbdae536e1bbecc6d32e183ab7125d12042b823a75836f155ca58e2c', 'a2dc4d08bb2faaf585d2dbfb9a023007f1f1cd51db062c5dcf9eb6613ef91115', 'a209f0558038fb79656f1b89e2afdaa30ed14d34bc403c4f3a3ccc47844e6d77', 'e107d7d56abdf4224d9302913a81f10b922b48db99d3825d867952b8c9a076b3', '0e711ba5a47417ea46f815d0ad964618eb70de1222ba54ba7262a6b44415e8d0', '787d72ced217d3ab3d70d30503cc270bbc1345593153d3a15f7a8c5b7434114b', '53605ad078384fba433f6d8f4c9876d2451d691027d7149b40944ba9282c3e1a', '9394f08929ca64bef43ac34776f818be47852b02404dbd2b1f184f0f7e873c71', '93b0f5ed8895e85fbc9ab36032a38ca151ddcbff733effe6a130c55e1ac272e4', '1003d4197cf4ac97eb0ff6bd29b335f54e1d31d7fad1958983e2b3782d263110', '5ccde161dd05fe0ec845e7460a5920cc3c4ce5cfeaf892e0f9d72d65afed1618', '8489206b09ab80a9092667010c3c9576177ba99695d3c4187770012be17a8ea2', '1ae60d84dce8076a2f83818aeab39506c25f98f0a0c6a0b8114279c0bf8585c9', 'c75b507f9607ea92265303487be09e399aba9aa53f348ba38b9587739057efaf', 'f6990f6ad8a6e999edb8e8d361bd193f7c11365c9037c2ae4e4bd7240b4334fc', 'a17b503a4aadf9190d47f7cc21e89e139114dece1e3149fe32a70c4c85b5d024', 'b0bbe6947e159db6d0a3f186ad96232f6d9d572f438835c7b4f0433530a80073', '3574395c1def98963945a0daa37fa29bd43d9c3cba977f24f59bb111ba03fa4c', 'b256da49a287f36c82af1a72cc0113cd61bf6c470a71cdbc096dea87f0b897b3', 'b03c1983df524a12a41f1a0b4693c0d8ba1b55fb6ea52f59e266b96392310ea4', 'ed6eb49f7060851c199f80e0e8c81bdf94ac472f5ff36eb77617a2e77dca00ff', '314e9cdb25c516d5416a428290a43609f09c0fa58352e4e160a48dab2a817f0a', 'e9f7cc7c3a0389bd9dfd83611834fd449af15bdaca08d6d80bf2f07615af55fb', 'b55050c86a34653fda593d756cab60b695a0dc123f2214f0d163d292e6ec4687', '28a21c3172478f7f614594684b7c4091b857f2cbf17794fe73226cdb3635b1d9', 'e5ac474a91308144187360f612dd5b9de1c321e958b13455a23ac60e76752756', '25820fef80e376995f39bb0d99fb63fe7637a6fb81f86b73d0a1ec1fa54d2a0b', '0a3a292dcda9303e9a6fc50806be2e9aa31427899f601e35481cc718922e773f', 'fa33ca5b31553887cbb6e3b45197c9c3e02c78846c561e1164fd539e6cdb6446', 'f40a52cd29fa4de3b080e0ab4afa3d93b4a442892dc123c30015529130cf7d88'], 'sha256': {'last.pt': 'c9f8d7b0b0aa4d972bbc05ba0b44ec6b9e7364c40f1f816e835a1f254449b471', 'final_ema.pt': '83bb88f5b0dc2a2bba071cc7cb5e528aba3be3ae5a9c011051851e42fe3c4d77', 'history.csv': '0838ba26c8966e2bd8a72a0355d5706c8c5ca5199ff45bb86e7b413cad2f13e3', 'validation_probabilities.npz': '23ec5adeb2b7b281a664d32c4900c8143b450f3073daf7abd7570d827c7591bd'}}, '2': {'epoch': 80, 'variant': 'L2_soft', 'train_ids': ['2', '3', '6', '7', '8', '9', '10', '12', '13', '14', '15', '16', '17', '18', '19', '20'], 'val_ids': ['1', '4', '5', '11'], 'initial_model_hash': '930a0e666a716697967ef01373c993cee807b0ccaaad3ac7d4fbb40115397000', 'plan_hashes': ['1244c68ba10fb206c9998ec3d215203733484d5544b235b33bbfdb0b7058de6c', 'dee06c40ffcf73af9ce876f0f937a2c4a890fd01ddade24ae74cca73ee48a69f', '0c0602df343625ff527f94c710bb1d9f3c386a753580405635f15aa142d866a8', '0f8f3dec7ba9681183a3611df435805cf7d66324986e10cb7ddf8c305d6eb611', 'f80877e7dedde111d6a05a3d327efd543d908c5ad1181b40162e35911dbeb833', '250c9aaa1e47d63db22ef63f9326f59ac71a42ec3b1ef71d90d0178ed3e72590', '1c04aaf2f1e06242bdf72681560ecd9edd4d031f33ffbbbde5fdab203a0ebc3b', '7c3783104de2cdca06289dcd45c6422756b407f673e13bb562fa6fa13f171e7d', 'cd69be5cba742c86844622f6bf3aa69d415e49fb90521be7eede082072002c5b', 'ba18fb4b42433c59fad02211720cf4326f95b705fdb4c2f929099a034e9e53e0', 'c3e5712693f958f6b7778d8d5ce50b307f158b2769a70c50232e0e4b9581e998', '9329bda092650a7012dae0e04cec4f961238c4a30665ecd80669aafaee66f24c', '52d9afa4aad8d7fcbfccdc79105559b820033c87a641f99407e22f5382d2d8f0', '20f2650e67a3c4003759ade9c6966f5a1dd8c72b3ce61680ddf458a755decafc', 'de0a6202106ad28ab73e2bb563220f561a4eb67412c02e9c88ad9d2f37d45842', '2cbfe6a284f2b19d99b2af75106f32fe65127db388869c41441cc367b5f4a849', 'f43c989f8d119bd3626fa57e1ee7621918336971f8d34d5db86729f56a48d527', '4e82d334dac1e868dcacc2a7c9b7b785822957cb4b2b1ba489108c893e7d8448', 'b74c7485a2975984bdb720b56d129094ec832cd22d52ccc3a3e1033fc3852589', '9bb0a01fc960b93ed73a6e06657f0beb357a8251a39b2a396c27ba4a7ad06300', 'd22e1fc2898428fe32abee0c228a381dad8528f1e544a78f6c60f748a4c54f75', '9505e4c75b146e9042a75fb3d26237b34c1cd6571b1f24d62740fabcd6ccd0f5', 'cde8ab5b81253abe08c41992758241823679e611bdc38556b8cd0c4595c4243b', 'c75d7a2aaf5bbfb6ab709d3249cddab682ba75fe5177389ebaca585b9eda77ea', 'd2c7821f0ece5c70494389800d086636d71a37b58ea75cfc7a360a9b38329011', '7cc000325bf23aff366c1327796229ffd1ae666181665c4ae35acfe7c3dbbb83', 'eff7f762f94d1c2f12886abcb462251d022b2b0ef843510f28ce9099ac4fca3a', 'e9f26a144c4b7ea32d4a43ca9927b95facd6de9777c24f8408fb0b8381e63c50', 'eb0157c243c0f74518833e6393e60023ea4ec42f32c57fcda83b965a8a728dbe', '51eca664f273ac664bfdddb5df4c6ac1337d5f5f6bee06c010ba211c61d9bd5c', '84ed65d82a4e0d6621c909ccc593da7d3e512c1b9d136558e79594e5dbc29c31', '2cdcda523f7840c7d5a7dd4d0ee717ca2bffed6caadd521f504fc0a1f16d83fb', '9dac9634bb23588f79c819783d5e5e757803a1431d927f746512ed52a184ee00', 'df62e98679f0e29a81d3d93790fdbd2223b3f4444b1360c1da83f104e2ada19f', 'd3a874aa38faea354a90979f346b0d64d5506b68c677cb13b0c147cd4a302fc7', 'e8c3c7932561e3ac85c6fcc768a8a1255b6e8dfd091b27111f16092a759b71c0', 'd739b90e5a5b3650cb36982fa46559f7e3061816bb873a5e4c83ac7eff73b73a', '59ac84b4a0a66dae4ee7d3647d15673f93be59c69028ab7064fabadabca9cde7', '9b766626ff9f34ed023abf626c4196ebd2ea781ff46dbe78d6507e0b4344b104', 'edc3ffb1b563394b12a1390eabf17fae63486c6592739d60fc95690bd2a1cd32', 'c66409df7208be6a1c375bbb82d6d4619e9ae2ea068aac802464ae66e338a094', '63675031934d117073d0828dbe5fe85638e65a03887047777209bf3d4e83c4f3', '3f75e4d55e66bff043f6dd1ec28fc8951e37aeb60c38f1cfc5f90eb9925ed33e', '0678322d25e45436fbdd0278cb6706517c5a00e26bba28fa97552fc05ae1880c', '7ba342ed9025fa80235f6ee1f347654660c1c7dbe55c6a3818fb798456a75154', 'bb967e09dfdf1098a184d96543cac6565df220337a68469cd985aa51b4a9345f', 'bf7ee1eaf5ab4491e95347f7ac92ea017c51c101a524f7bc96505d219c1cad5d', '2061c865594e5da2c3886821d19d7af304d5bec03bda29175d37a6b3a2d6160d', '61b07f5a04577981bf138576a109e948e514637deb0c176109999acbff6410bb', 'b92833dc28a61db3302bbdde08c28dd6bdf3bd25d838666ae76e13fc20d94937', 'df73e137ef6f8aae81bc2e41e090b29232123d45ed06391287904f66e8348710', '19fb045acc7492d472e9b11e982c01449b038a651c6eb1d7b003a60bea7df2c9', 'ae090a8f93968acee5af30b8cabf0de53b5310593e1c6c70904f89a82248c604', '083a3c53346d0ea1ee2628e2e90fcadfb69b410982542f73a7316dc0dd799455', 'b5c695eb83eb8e66edf20fceb633cfa76adfbc158b659fe01f6da24651ed8396', '14665367c0484b3d9030943dc6062a0685b78beadbc11a4e1a2a1f810789476c', '8789aa9099d6e5babd3c2ded2a00ad068009d188485b10ea548a0070add4ec68', 'd7c9b1bb8e25e486f51f6733e423edb500bdb7549ee6eac52e799db50e51987f', 'f2bfcb9d4ae808ef0937b090e41b4e55884e13a80cdc0bd6dc40e81b2a5f30d6', '9a61c5649f4970f6b73bd58dfd8d313a5c7c6d9fa72e99f5cc243f1b238744d3', 'eadb362c4ecefda0593ff213da06975610ae8b7e55fcfbebdbc3ca6c8df1ba2d', '7109ab3e8ec43b73ae3bc185471a5715b13ac053e17e1a31e77e77052008e6e4', '1e7a8bc427790cba2adaad6cea2b20453c27dfc9178524c9c267f3a5be0d82b4', '5ff5cb48f13181a91fb6e803db87c41e9138d2297b03ca500fdf9c0f61149a15', 'f31bcdf53b8e10968e306236770922835f6602dc9944dafa29ae65fa8b3da5f7', '5bc125241e1cb4f42c1cced0823cb066ee4c07e7cd11c5cf49333320efce0c3c', '1bf33806aad119b0307e372cc8e0c2087ad7d86d7da87b414e162d7774b6cc63', '125ec2929f73112301fe0b4ce79aad38e31955e8b860422e5db6a25035390041', '76eab83f137de96c2b12c4afe76d685cf4ab41faa246468df52425a80029961c', '4f3a214c17ba433ed46035836574b8a6e4a55f9e9b5d5cd17d549cd9a049350c', '6c8a91833c4831d75070ddd084f49034f72af3b35a42eed85ea84afa2564f598', '2a25befe1b12a5404fb0a4e8fd75a42319a676081367520022d3df09958b790c', '28ec5cd0f695f9969a0fef9cf618c56aacf25c54b4eafd4d19afdd2c6eedfd91', 'cbf8cab5cd14321a99913baf678483413619b1314fdf66273487c9f3f8b3ea0c', '0fd94afad6eea27ef02138019bdd1412a90842d5f27f5639c800f0680e5a5b8f', 'ee210975871490982da1aec55da64e30acc9c5139f7ce4a4fa85dae5a6475f91', '1acd5f3385d8de7ba797fd6ddb8859adcac41ffd917c43e9af40c44f1f7245e2', '0e22acf7c902e06f7353a6b8ef0a851df2b66c76a74c012034bc3f9b8d6b1402', '15a6e3ecdb9977493b59bad49f07e4d126940029163cd246ac911afe666fecac', 'df2d1ed2ba1cadd27cff5b7a62b24c9987f4d9718a5e5eb7b66a6b8c6b8dfacb'], 'sha256': {'last.pt': 'c0fa184cf0007ef5ee7f0adbb6551929b30c879147b0810338a30fdd2b8836a5', 'final_ema.pt': '4cf38b6fbe55ab438b358b37094cb0a704a81ba01bd1831cf98cd75827c181df', 'history.csv': '7066d9ca1107fdce4a34830ef3d3c5a8b74241dd0e0f3e6f4578d72696490c48', 'validation_probabilities.npz': 'c74b431437fadcfbaea06b18c923b974a49e7c86945c4cabce5939b49044f9aa'}}, '3': {'epoch': 80, 'variant': 'L2_soft', 'train_ids': ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '17', '20'], 'val_ids': ['15', '16', '18', '19'], 'initial_model_hash': '30be10d91a902aaa0811e387e3d878b94fc609d5e94e86b691ac8b1179ee2327', 'plan_hashes': ['81b4d90d1f56fc0625202ad8a0ef2c352e006bf93c7b9bbc47f8c95576f05362', 'd17d26122c0ed914242428f1564632de804d3c83eb62c46c33600c566005b400', '98bdf0da80b045d98d8e86fb2548a8c5c20e11f554099dcafec0f123f6413a5c', '449b1946f2d4cffe3175cbc3ceb529e5fc7baf2aec0acdeb6cbc53b98725ac22', 'f28636b43951f848b1df1a659804d0780b499c505b155efe6c3abf2b91f6fc6c', '063b64652af11af66155bc68b12f1af852a100b05f3dca60f608f61d5294c9ec', '4fc056c423d2d2943edf4ba4c861c87296685defdcb707684c57f68d2336c45b', 'a5df0b13567114ee944481a574713f23e297a57a0070665e79e4a97a7884330a', '8898b7f3e38db7a5e1dbe29f86207b0e21c573a1e40c3761bdcf8911452eb408', 'fbc38ca65e818b31fe6d6d95e38be321965d1e56decfde78920d834eb59209a6', '002fa527f409675a543141e6f60c4515641bdc4cecd3dd40adbe55d346eb3cce', 'd2ffca0f9129a8736dd9bd743374d4a0bd32bc8d54a4c0a6f43642381593ac05', '9b54b0ad17260bf96f48607cf87912aed251a1a945dc1623280ac2db9dfa5c0c', '2caf3808c90969fdd0bfa238ca3283d82e04d0e5765152e3d2cdc42eecdfceb8', '87d8179785eccc0c9924aa0c1f02aa630d0be76c196a373e6086eb0519132624', '6f97e1967c4bc62ba9c54e8b6af2dc6c6fad7e89fb04b810adb43b4d3dfcea80', 'e726f1482397d40ba99b7e76110fff84f6d6254c09ff5076649260b2cb967784', 'a54eaa62f2d75b8e2688231b97c0215093a399e91405071bfad1d96413772fad', '8b9c9f00bcb45641a8c58f90852560ab8fd09b708bfb2cd59518aa2d84c48cd0', '1948b6dc9a6d10fc4b4e64a8565147f4bcd518045182943d1336ad668fbc3cee', '276f7cec4a296e50eacad2639f0f6735adf65be387da8d56a402d15a36634057', 'a9fc3fb63567f50a45aac6c97a6955ea3ac24c452fbae04653e860591184a58b', '8e79f0f8b1241d6d5235f6355cf20dbc6b3e8b5461c471963600dd1d277c9bc2', '3b9451e851a1012f0d0742dd83b8dff07c6ae64c18f109f4207f844ad24a65cd', 'df31d200fdea8f907817225fb67342d3945a375373c27355273b67564ffe565c', 'a6cfdb25bac0d5c0c1e49982257e78b022ddf14ac979dec4cfa4043ff013fcfa', '419fb3f15ff2bdd57aa043fbfcdbd66da136511f5cd37924fd1f8e0eac913046', '5b32acc8fa068dc601132e4c4ba26587de06f2433e8e9e5f2487e541ab0bfa7b', 'b05dd0cd56a34f03a8437cc1f44dd59e822e81bae163cff823e6aa46111dbc1e', 'ee584a58fc10e9f855a41f3809a112b3db9542e4124903352ac7a2d462f6df60', 'cbfee3c96d92c87432df7437ebb1f8562707a70605797d8e9931543e73bdbcbb', '135831995d1838378fab49526d7eeaeb43945e36fbcac2cd20352fc8e54b679a', 'b7739eaedcede36b61c2fced09988590552eb911f4448075f88a58aac3d28973', 'e139d2fdab6c00c6babfa932d9b798616cd5fd4b9cb684e030aa82addb642a6c', '89c847285842660bdf78a38e97b99ed98781fe6e0bba8a2ae1102b2b4d01869f', '5024f2152ef2ff8e580fdfd88428c2278de99e376efb860386a42f84f5feb870', 'b50cc8671f5eeaf93ad07c29cf14ffc98279776008d9e3ea28d61d168c69993d', 'f62a016fb0ca1ad34f33a2e4ca2364f023d402ec5ae12658aaed22d46a5b0633', 'b7b7b802a313de0fbf0753e03ed2ed663e764a7c09aede800d5c6213f696c76d', '761014389e4f4337356700264862c548f0065b7ae0fbc08927424c6ac22a34ef', '733333dec00441275f08948f17c095d355f97d30287628cd7024a1cef5446e05', '0f265a111d54ccbb5d2efeba9c1aeadbc587935a19fddc193dae463b3ca9e28c', 'a61d72b819465ee2c701e87f62af83e46abd8816deefb5ce269d2e4d4b6193ab', 'fcbed2281e2343f609b082cb5c65a6cf1e1f3795ff9d130bf5c9b0f7cbaae578', 'a4596c5c336b6ded8db5f0144cc70adaf38c4c9c2acd4b006f161074ab0b9adb', '2f634da0dc2afb4c124866846ccb93316ccf822e522a4e77a2700e6424e97007', '1ad21afb4bf197f88f85d37e0c30fd2c05dec2da51a477e5b12d8039f99f047c', '78e00704e1c3337399a511351b0e7603f54d032708233fb3d28ebae0b469a3c1', 'c5958700ce74efbc1eb62169bbbb9c086a1acf7a481d948f0089e344c52ba74d', '9dcf2ab18b0771e1359a3a2e66f4eadfe8f09515659f9229c7231dddeb0c055e', '838887e8db55e0352277564ab5c37e8daa7c692daca5a70c6350af05629cb534', '8e2cf6ef7ca0963aa4c8144c0b28ca95137319d47d3b9af3fdca44b42204e115', '7450247525c4261dddc1b9f8b6f576c8a76f2401a5cbae140491fcce8d477a01', '87961dd7798ddd38fb9a4f6750431caa306af7194831ad169091ac2b9cc6ca3a', '6e16e6e177ad37cd97d13078c994081d169529f9a25fd900a27b850e96e3e2f0', 'a0ddcd3fdc18597ad8979454ea81ece261eae500231dfb56b7eddf8d6ef863ba', '7ce38219a7ba45c334a50a1ac1a87ae588e765accc493604229c60f0ed21c79b', '1306ac9788685633477c81d7bc65254ac206a8a7e0bfa669879d66f7ba4a03c8', '1ebd0d8c4cd3aa70cddb5957893b7bf48bcfce0d8fa5224b39af037e96a063df', '05a2a00999d82ece1f826a0531d83c8997be5f9cda7874da14fc7ac9e309d1ea', '7c79f26870c4ee0ee685fd43a5684e85dd6db9b602e2fb8baa7c58c06f5f1d6d', '2d76ee34a51a153bc835e817d09b2270330835d13c7de054876e6fb9c0c8d381', '356a51d148cc837e03bdc07c8b966507fbb8af9ce08a972673f9b9062b7cd074', '2366a237e3121c2ed4a040911dc3713d5aed65ac21b5ef16dc5eecce35994457', '96bdb41c2583df99b26fcddabcf513e5d9d6ff3d86149b472e0387eee5463739', 'de9eb043108373db2ac00707ab2b69c61e341db04393520f63820e86e2fd31de', '621f095855f3b88973b1f21d8ca139d11f8ad5f2928613e541d20044ad7df7be', '2b4283369eff2299a04ab8db4746750c4f8889f206be423a052343524ceea10e', '7953d72c15b9cb7fcf2721f2f68e104d9858c57e26c72723cf636d37e53f7059', '2a50d9bbbec34a62239c35a39d06f2693f29f9d04bcbbb6f179e75d1db705146', '81075a8666bbdc0e7c8b866b6d41278d9e90fdc7925e070de251282861112947', 'f4d4a47ccd5198ba46ac98871ae1395e3979dc4638eef9096dea3a848122d980', 'c22fc8312c41aa297db92c7c884361411c06cffea48e735777d6d96a0ca5d62e', 'e9cf5652c1c168f2c25249e006148240d591b3f8b47ec0869c49e03130192d7a', '9d812bcba7381de6f64e0a775de31c6f6fb7afe706951bdf6a445507cd635920', 'c35cd6e9f4cb64f83496235e5e8c8d3de8f9c7ff314d49b7b09e2b2dd23b9a4b', '031011f010b91daac8f1afe3c93c3d10b9ca64488650322a3e2790d2ac216cc4', 'c5ef2b62e01d5af71fd54fa706d5ffd46fc9ddd28d8292d21f134a5b1795593a', '88947c60ff6a5cf9dd32691289d895c1339250eaba5d4bd6677b8ffefa3a5fb9', '067a846bfb50e1f18cff7156f5a8add4b50f2414a176821ddc69556bc3a49444'], 'sha256': {'last.pt': '1533c655c53db61c0d5250959a692805f0c88558ed99da08e81a57edf034dd6c', 'final_ema.pt': 'ce1498f7ce3f8f94b068410c743b5500edc2e2a330d29c30ecc87ec299f18882', 'history.csv': '582ee30a892d67e34297a232ca4d22796bff2ab81ca01e7e915411e9e6cb636e', 'validation_probabilities.npz': '73c3d3e9480f3c7cef0ce68495acdb856b19bd4ad2e4a2c475095335a57a7da0'}}, '4': {'epoch': 80, 'variant': 'L2_soft', 'train_ids': ['1', '2', '3', '4', '5', '6', '9', '10', '11', '14', '15', '16', '17', '18', '19', '20'], 'val_ids': ['7', '8', '12', '13'], 'initial_model_hash': '673ef3e38716ac52bd423a8a6275ee34c5b9e67ed9c415f2cf29517a284164b3', 'plan_hashes': ['e589f0aa26da48ed4d2926ec468bda6c4f54200dcdd3fa5852ff86783f1d3cf2', '22cc0c24a26d134f4da358d448e9b7a73add501581cc29c6dfec2b7c795f574a', '7d6225367cc063f12e0c461ccff09461e2233df2f7605f126815933d9bbca8d3', '28fedd890f66176a1a29695ed7ef274a5190d66dea38854e916fbe052db2529a', 'ae824b0d9c9e3743c967d4933d48557d7fac05441bf23495cd99f01e6d05f859', 'aa1b1a4423d137563855f871e4ca12dcb72da7c83ce652a40bc0422a82370bf6', '03afec12422556368f21bc86176c8d55906f3e0af0e0d11f78b5ea8bab88ae98', 'ef8df46c3af125962914035d725eb32c9a2e454f2811ff44420f258a1bd04e58', 'a1b0b7809aa4e4a292fdf18471148fcb885c8978ea9cd4bf2a6ae2a241616958', '941347e13b7a53bb55430473261572f6fcad9a6a2bfbe4f9c76d2295e0ead783', 'aee66cff5fd71efe20bc54b119083468a29408a698d72316014d8fdd2e07208c', '98ced5de7c192861ee511cee3bd08e648f67c4d47ae87c04ae102d286342db04', 'dbe7ddcc867a87d1fe9f165011cf3515f3ad5daf13b69790248268d834c889d2', '209a141c6590b166019071976f83c3203f751447452d255373fcb833a1a54d4e', '58ac602e910e28368c4786aac5ac99ff3c89a4b99039a48fbb70f43bc8642145', 'a4c0d14358915a44f9fa789390a859eff0bfcb10acb102ff20dd039f62f4c6b0', 'ff61bf72c5c575dbd5d5e771fe21ae9de1c2388947fcbd6f60800bb092f08f85', '5495626b251fbfaea3947301a59b4df7184b14d2285a9a407ef6f1daeaa140ba', 'fcbc462ef187965efe648a248b96631afd7c0c9d6b175498698dfc90b2158849', 'e720069a8439e847523d8fbd95bbc5c4a08b4de3bc44a1d1f2e6066a98f1cef1', '97ecfaec725998e0d2a120641931b793ed6b3948dc97132eef09931d222c3631', '3175713e7c12fadb5b2736b0240b27916576c064cf5e3bd80090fc11169f28fb', '5dec6603327987aa92a3226f868e38a32408e16723e4c057e44594e20a18b3e2', 'b723c10ff24d32afb831a383f5c21cce677b749dfa9c5f48c3806b94ce8399f5', '7fbbbac8609257604f0a2b6cb235f7df129c51c0eb8caedd980b06d5518ea51e', '6ce507c85cfddfbbd1cf709eafcd8eee78b2dce0c36cc3c18fe167a2ca64a3e6', 'c92d3873cfc23684337ef047a8704411f2aaceae91d0a39846be0f22437a32b1', 'f8ea80ae277551e3e5f835a614a66c59d661cb0abbd9372dacc032068283bd2d', 'a6065632bcb566512a21aa23ce8b53ee108eaa39c43b74731b155a715802452c', '90c6e3f747e1add7ca09c684ada100d4963d1878420805026286ce050c7009d7', '8c4fdbbff1a5ad92aaae47a9db48009cbb4ada2cd4816347210dc6e749c11acf', 'bb31a567b1892eb2807dc24d4f11528faab6e41cae1f8cb02e7d3618d494ca5b', '076258eb13512bdab8eb3eb6b0fb4db677681fbe5199a7d6f5bca84a86b74bc5', '36b030ba78751582100876570848f7c2d37025146848a4162d344ed1c976e06d', '3ec827ef40aeb044ec0efd8f786fd535cd66db3c43a94c572a7cda0a0dde476e', 'd12ee7f6941ce4720ecf42e437775e9fb410dd7d76ebb9af9d4d3736455f1e4b', 'd0b3542b3b6c575da73c3c7cc67e0731a593e7ab1f825429cf1d35277baa4245', 'f79f5e5cb818cd8577d3d6f83bbbcc63d81cbde85593c8aa9a157446f4c8ce28', 'acf19d2e4daa21c5539efdfd4b49aa345798abca4bbd411d525ca469617cdff2', 'be0dafbbbdcde1baae5e4bcba3868835afefdbf49d6fbbca5559836bb6b581c2', '2cd5dc1069c68de3e13655550c783856ef272f37f1d32dc6413d90a2223017df', '1fa5f7727d1be306e8ca04f7c637f78485fa35a81fdea0f597646bdba8328ba6', 'a280296e60b642612d90deb72b2149b60827dcb923466167f98256e3a7ccbb87', 'c659af1b6df39d5723407ea69f18753b6581f13209a442a5767fec76120b2cde', '171cf713387dbe16747dc3d2746921707cf357404fc7965f77a5fab23e722946', '96eaed0aa9c37a4c95af7c28e4da4b76643c4a444a3da5c17287d717a2be2ae0', 'b27a3a2fe284d32971b0ab1ee6cff429ea3c64ae79c951acf17c9079e9b53cdc', '68380678ce3c3cba4754f5e4167414b9e80d922ebcd1af141059979a3519c086', '1b664d243df3c25e9b2808c84e4bc389d0a6dc18dae330af8f83570a17095f2d', '194046435220224359482d3a11e302afda7b00aa138ad4f549ee9024b60e1267', '0c2c5bb68683f53afb905dc14c27adb9af981115ee970459897f35a2f8e5527e', '806a945e75fb0e9eb94e9e59eea8f49053ac81f43e15177d1e4e0652f0cd6303', 'b94fb4867abb6a0bbf733cbe86bf92e3e980615b182294d54836bf37422f6a30', 'a1d50f65c38606aa2234660dcd225a9c0fcffd25dffe94b29627e98a829f0461', 'e4c935aade0438ec220151babf2536fe648a0261bb83aecd15e1f8807415bea4', 'dc6421a51223c75bc1e7ed09f9759a15fe886c6a028c595ca79f5f3af1fb0969', 'b82e13fe1f70555d0ab62b529fb538c7751fb653ab0efba4238204b386edb8be', '2b55709b2c63d9d4b178bce8368d416d3e7d27cc822dc5fc5778a7c6a787f330', '2d0bcd14543ff5a845c4c09951526b5f355aa536cd24e6f906af8c29d3ed15e7', 'aa897406a1b0ee07d2a454fd6b6105d32f93d61e0e89bc1ea34c3cb13f51a8c4', 'b2cf5eb48561867280d27c9d20055e5c255a9d17c24e1454c3f422a312de91b0', '7dffb9d2fc545f44405d38179182a3325bbaa5d483358a074023f82755e0f14a', '26a7b626967e41bf5fa5e435186da101057da6eabe2d70a75787ab1767012ce6', '323fa480a3bcedaaf9ff4aab0179eb1b4c96a344d578a83ef3d863e74d4fa851', '6d3fff3e495ffc5c459f1b4383bd589d7cebc8ddbc16efbaed063869467649b8', 'e60f71b1e2f4e63233c7832651c955e81870cf378607b91b25fa77f784239602', 'fe1bcae1857869e890528942b6f1bb0a979749b34bdfb4aa5279bc22aa816785', '29c92bc6a28467a3edb9cb6298d1abbe21b3e98e0db1aa7e33051f39f692109f', '3a5d68f263c0a249b8bcffc98fa58c8244b3975c14d1a422730294d79d67e850', '49aadaa4fb6c2b2689002cdb2c4fe682134f382dc17d63748959a9d14d129798', '57f7806d927c29e2b1789df5ed80681b06e51d3c83e7669549cfac0b6673b20b', '0b75ff61a40282e8d16501158aeb887a8f805b6923ff68349b46447d4425bf26', 'a5dd52b3579ce9d2977b028a14fd410cd38609e7a93cf8062c10c7dbce0b8394', 'dbdd1446c580bbd16a8e9121eb7f48f23e2579f562bb9b050ea3ff7a1e569aad', 'c1b441cc2cd9f758fb59cdb06f81e65d6951dd9f5ec2cac745c9dedb04a79677', '570032d1b9d43ce269e0c18a8edbaddda643ec6288ea98d0ff0f8ec445584018', 'e0b52d2662dd3d7fe82ff1ad23d971a4eff50b473d2e42b02b91cc2db7f49611', '0e3c36eb1a09dcbde1434473eeecea0402eaf4ec455701aeadae11a8401ae8f5', 'ae825a3dc9ed3cb3187970f069349675dd78c2d4ca8da27f047d51366fff7169', '3a724859c08f0ff60594da247fb9ac77c1fbcb9915d55ba897bae53b09e61f11'], 'sha256': {'last.pt': 'ea5a801553359f4a6a333cc59365497a504e72051c2146de6140c25f0f7a9ac0', 'final_ema.pt': '3235a3f83eb02f7d416d61d96aea4b103f087cd88222d970697f7049bd3865d7', 'history.csv': '9d108e89b8abe9e77744721260383be0499eac5136ee625bd72f154854ce3c4c', 'validation_probabilities.npz': '4596232e42800da96032c939e0a742501790a4b9849dcfa416e3012dbfc8da04'}}}
EXPECTED_CONFIG = {'config': {'seed': 42, 'folds': 5, 'fold': 0, 'epochs': 80, 'patch': 256, 'patches_per_epoch': 256, 'batch_size': 8, 'encoder_lr': 0.0001, 'decoder_lr': 0.0007, 'weight_decay': 0.0001, 'warmup_epochs': 3, 'ema_decay': 0.99, 'monitor_every': 10, 'threads': 4, 'amp': True, 'pretrained': True, 'tta_gain_required': 0.001, 'soft_boundary': 0.5}, 'source_sha256': '8353368810c8988d5e2d4a349de8bec049e50dc838e1f9397691496d835f0485', 'v2_source_sha256': '42dbe8eb0347828b2b3ebb5cac49a82856232da741a6126972fa5800f65b0016', 'base_source_sha256': 'bf4fef344a0965b8a1ca23fc734ee2938b7316e2e65b769f947750c8a842a135', 'data_fingerprint': '703ca1abcfc30fa5d5a5b7dbb8bdba6751aca8a873b6652d841d45abb47ec9a4'}
EXPECTED_L2_ROWS = [{'experiment': 'L2_soft_none_t0.1', 'id': '1', 'fold': '2', 'target': 'clean', 'dice': '0.737158367333452', 'precision': '0.604603623533365', 'recall': '0.944157775056848', 'centerline_precision': '0.7771605735693063', 'centerline_recall': '0.923693668786287', 'cldice': '0.8441150141614625', 'foreground_fraction': '0.13622665405273438', 'thin_clean_centerline_recall': '0.8682038834951457'}, {'experiment': 'L2_soft_none_t0.1', 'id': '1', 'fold': '2', 'target': 'proxy', 'dice': '0.8003910799272471', 'precision': '0.7794237069810422', 'recall': '0.8225177304964539', 'centerline_precision': '0.8510528355509624', 'centerline_recall': '0.8810596749965861', 'cldice': '0.8657963384357997', 'foreground_fraction': '0.13622665405273438', 'thin_clean_centerline_recall': '0.8682038834951457'}, {'experiment': 'L2_soft_none_t0.1', 'id': '2', 'fold': '0', 'target': 'clean', 'dice': '0.739177364018852', 'precision': '0.6380819273901035', 'recall': '0.8783381430076497', 'centerline_precision': '0.7210687268630291', 'centerline_recall': '0.8368566176470589', 'cldice': '0.7746598872405187', 'foreground_fraction': '0.11463546752929688', 'thin_clean_centerline_recall': '0.6988966318234611'}, {'experiment': 'L2_soft_none_t0.1', 'id': '2', 'fold': '0', 'target': 'proxy', 'dice': '0.804642291404318', 'precision': '0.8282586270007654', 'recall': '0.7823353763947823', 'centerline_precision': '0.8841139546112989', 'centerline_recall': '0.8271076332496679', 'cldice': '0.8546612615326227', 'foreground_fraction': '0.11463546752929688', 'thin_clean_centerline_recall': '0.6988966318234611'}, {'experiment': 'L2_soft_none_t0.1', 'id': '3', 'fold': '1', 'target': 'clean', 'dice': '0.7861276931005331', 'precision': '0.6917579926939178', 'recall': '0.9103126425488614', 'centerline_precision': '0.880640282329216', 'centerline_recall': '0.8853296193129062', 'cldice': '0.8829787248142288', 'foreground_fraction': '0.14306259155273438', 'thin_clean_centerline_recall': '0.8009239815203696'}, {'experiment': 'L2_soft_none_t0.1', 'id': '3', 'fold': '1', 'target': 'proxy', 'dice': '0.823142385367937', 'precision': '0.8640108791296697', 'recall': '0.785965508040847', 'centerline_precision': '0.9427779178220318', 'centerline_recall': '0.8600596467079605', 'cldice': '0.8995211314420045', 'foreground_fraction': '0.14306259155273438', 'thin_clean_centerline_recall': '0.8009239815203696'}, {'experiment': 'L2_soft_none_t0.1', 'id': '4', 'fold': '2', 'target': 'clean', 'dice': '0.6959002963868561', 'precision': '0.5472681472044474', 'recall': '0.9553679741205015', 'centerline_precision': '0.7577968526466381', 'centerline_recall': '0.9298115447395756', 'cldice': '0.8350376346188654', 'foreground_fraction': '0.13174819946289062', 'thin_clean_centerline_recall': '0.8886769964243146'}, {'experiment': 'L2_soft_none_t0.1', 'id': '4', 'fold': '2', 'target': 'proxy', 'dice': '0.7774113129251987', 'precision': '0.7268436748993833', 'recall': '0.8355412062308614', 'centerline_precision': '0.8469241773962805', 'centerline_recall': '0.872969837587007', 'cldice': '0.8597497928932843', 'foreground_fraction': '0.13174819946289062', 'thin_clean_centerline_recall': '0.8886769964243146'}, {'experiment': 'L2_soft_none_t0.1', 'id': '5', 'fold': '2', 'target': 'clean', 'dice': '0.6742873560866167', 'precision': '0.5242532550421241', 'recall': '0.9446265309642918', 'centerline_precision': '0.596100278551532', 'centerline_recall': '0.9364720394736842', 'cldice': '0.7284892686895047', 'foreground_fraction': '0.119537353515625', 'thin_clean_centerline_recall': '0.8609332113449222'}, {'experiment': 'L2_soft_none_t0.1', 'id': '5', 'fold': '2', 'target': 'proxy', 'dice': '0.7308909510276763', 'precision': '0.6522529997447026', 'recall': '0.8310901476029765', 'centerline_precision': '0.6446268875531447', 'centerline_recall': '0.9014980504822492', 'cldice': '0.7517243504991532', 'foreground_fraction': '0.119537353515625', 'thin_clean_centerline_recall': '0.8609332113449222'}, {'experiment': 'L2_soft_none_t0.1', 'id': '6', 'fold': '0', 'target': 'clean', 'dice': '0.7832765474165484', 'precision': '0.6918022920480478', 'recall': '0.902627359879593', 'centerline_precision': '0.8332909460834181', 'centerline_recall': '0.8122448979591836', 'cldice': '0.8226333348157863', 'foreground_fraction': '0.15211868286132812', 'thin_clean_centerline_recall': '0.6438496117695137'}, {'experiment': 'L2_soft_none_t0.1', 'id': '6', 'fold': '0', 'target': 'proxy', 'dice': '0.8015064034027116', 'precision': '0.8459262231361436', 'recall': '0.761518838747545', 'centerline_precision': '0.9230671414038657', 'centerline_recall': '0.7553799771285996', 'cldice': '0.8308470710372602', 'foreground_fraction': '0.15211868286132812', 'thin_clean_centerline_recall': '0.6438496117695137'}, {'experiment': 'L2_soft_none_t0.1', 'id': '7', 'fold': '4', 'target': 'clean', 'dice': '0.7057488840118733', 'precision': '0.5796232529876443', 'recall': '0.9020300088261254', 'centerline_precision': '0.7752149297546655', 'centerline_recall': '0.8193479422768573', 'cldice': '0.7966706972269538', 'foreground_fraction': '0.09416580200195312', 'thin_clean_centerline_recall': '0.7118015963511972'}, {'experiment': 'L2_soft_none_t0.1', 'id': '7', 'fold': '4', 'target': 'proxy', 'dice': '0.777374495330987', 'precision': '0.7722098440348389', 'recall': '0.782608695652174', 'centerline_precision': '0.905640595512686', 'centerline_recall': '0.7807477288609365', 'cldice': '0.8385694183140483', 'foreground_fraction': '0.09416580200195312', 'thin_clean_centerline_recall': '0.7118015963511972'}, {'experiment': 'L2_soft_none_t0.1', 'id': '8', 'fold': '4', 'target': 'clean', 'dice': '0.7055021132557412', 'precision': '0.5811279143037177', 'recall': '0.8976105893230815', 'centerline_precision': '0.7056753862628117', 'centerline_recall': '0.850596789683084', 'cldice': '0.7713884851133979', 'foreground_fraction': '0.1210784912109375', 'thin_clean_centerline_recall': '0.7629613356766256'}, {'experiment': 'L2_soft_none_t0.1', 'id': '8', 'fold': '4', 'target': 'proxy', 'dice': '0.7987265764136126', 'precision': '0.798361688720857', 'recall': '0.7990917977988711', 'centerline_precision': '0.8979654275661618', 'centerline_recall': '0.8195070235886562', 'cldice': '0.8569441382718577', 'foreground_fraction': '0.1210784912109375', 'thin_clean_centerline_recall': '0.7629613356766256'}, {'experiment': 'L2_soft_none_t0.1', 'id': '9', 'fold': '0', 'target': 'clean', 'dice': '0.7348549726333268', 'precision': '0.6078600468153259', 'recall': '0.9289277981737739', 'centerline_precision': '0.7680163718754568', 'centerline_recall': '0.8759912496581898', 'cldice': '0.8184580321221598', 'foreground_fraction': '0.1238555908203125', 'thin_clean_centerline_recall': '0.7993333333333333'}, {'experiment': 'L2_soft_none_t0.1', 'id': '9', 'fold': '0', 'target': 'proxy', 'dice': '0.803064303954495', 'precision': '0.7990944930393002', 'recall': '0.8070737549382524', 'centerline_precision': '0.9014763923403011', 'centerline_recall': '0.8393686165273909', 'cldice': '0.8693146011464258', 'foreground_fraction': '0.1238555908203125', 'thin_clean_centerline_recall': '0.7993333333333333'}, {'experiment': 'L2_soft_none_t0.1', 'id': '10', 'fold': '0', 'target': 'clean', 'dice': '0.7172252252257347', 'precision': '0.5822314533114907', 'recall': '0.9337117658097204', 'centerline_precision': '0.7391435011269722', 'centerline_recall': '0.8741629464285714', 'cldice': '0.8010032585659009', 'foreground_fraction': '0.130401611328125', 'thin_clean_centerline_recall': '0.7807785888077858'}, {'experiment': 'L2_soft_none_t0.1', 'id': '10', 'fold': '0', 'target': 'proxy', 'dice': '0.8030772936680571', 'precision': '0.7802188158202668', 'recall': '0.8273155902971648', 'centerline_precision': '0.8934635612321563', 'centerline_recall': '0.8273602567531425', 'cldice': '0.8591422709223904', 'foreground_fraction': '0.130401611328125', 'thin_clean_centerline_recall': '0.7807785888077858'}, {'experiment': 'L2_soft_none_t0.1', 'id': '11', 'fold': '2', 'target': 'clean', 'dice': '0.7147810844802142', 'precision': '0.5706376247262107', 'recall': '0.9563577036810441', 'centerline_precision': '0.704670523493721', 'centerline_recall': '0.9342400239127185', 'cldice': '0.8033768624665514', 'foreground_fraction': '0.125396728515625', 'thin_clean_centerline_recall': '0.8922318125770653'}, {'experiment': 'L2_soft_none_t0.1', 'id': '11', 'fold': '2', 'target': 'proxy', 'dice': '0.8168295383433354', 'precision': '0.7742759795570698', 'recall': '0.8643325296295039', 'centerline_precision': '0.8584732609002399', 'centerline_recall': '0.8969688137569222', 'cldice': '0.8772989477560637', 'foreground_fraction': '0.125396728515625', 'thin_clean_centerline_recall': '0.8922318125770653'}, {'experiment': 'L2_soft_none_t0.1', 'id': '12', 'fold': '4', 'target': 'clean', 'dice': '0.6493877479931557', 'precision': '0.588415404040404', 'recall': '0.7244568808052544', 'centerline_precision': '0.7429237947122862', 'centerline_recall': '0.7208691499522445', 'cldice': '0.7317303261032974', 'foreground_fraction': '0.120849609375', 'thin_clean_centerline_recall': '0.66'}, {'experiment': 'L2_soft_none_t0.1', 'id': '12', 'fold': '4', 'target': 'proxy', 'dice': '0.6815690798966231', 'precision': '0.7458964646464646', 'recall': '0.6274561869357408', 'centerline_precision': '0.8426127527216174', 'centerline_recall': '0.6824275796025856', 'cldice': '0.754107506790638', 'foreground_fraction': '0.120849609375', 'thin_clean_centerline_recall': '0.66'}, {'experiment': 'L2_soft_none_t0.1', 'id': '13', 'fold': '4', 'target': 'clean', 'dice': '0.7449959545507', 'precision': '0.6290059104814518', 'recall': '0.913435410825965', 'centerline_precision': '0.7823231632234232', 'centerline_recall': '0.8566662695103062', 'cldice': '0.8178086477010148', 'foreground_fraction': '0.12843704223632812', 'thin_clean_centerline_recall': '0.7774277242401779'}, {'experiment': 'L2_soft_none_t0.1', 'id': '13', 'fold': '4', 'target': 'proxy', 'dice': '0.801441337561543', 'precision': '0.8257447503638362', 'recall': '0.7785276245414579', 'centerline_precision': '0.923108496374333', 'centerline_recall': '0.810011641443539', 'cldice': '0.8628699327446474', 'foreground_fraction': '0.12843704223632812', 'thin_clean_centerline_recall': '0.7774277242401779'}, {'experiment': 'L2_soft_none_t0.1', 'id': '14', 'fold': '1', 'target': 'clean', 'dice': '0.7325316086535957', 'precision': '0.5986596752937218', 'recall': '0.9435213010429556', 'centerline_precision': '0.7998660415271266', 'centerline_recall': '0.921788464136161', 'cldice': '0.8565101621825157', 'foreground_fraction': '0.13604354858398438', 'thin_clean_centerline_recall': '0.8691525423728813'}, {'experiment': 'L2_soft_none_t0.1', 'id': '14', 'fold': '1', 'target': 'proxy', 'dice': '0.8225072046112067', 'precision': '0.8002972268177102', 'recall': '0.8459851201944453', 'centerline_precision': '0.8964501004688546', 'centerline_recall': '0.9073751840449739', 'cldice': '0.9018795577192459', 'foreground_fraction': '0.13604354858398438', 'thin_clean_centerline_recall': '0.8691525423728813'}, {'experiment': 'L2_soft_none_t0.1', 'id': '15', 'fold': '3', 'target': 'clean', 'dice': '0.7037771405166673', 'precision': '0.5592333133675356', 'recall': '0.949085123309467', 'centerline_precision': '0.7890747516989023', 'centerline_recall': '0.9324636499524392', 'cldice': '0.8547976883335181', 'foreground_fraction': '0.14648056030273438', 'thin_clean_centerline_recall': '0.8900668356764231'}, {'experiment': 'L2_soft_none_t0.1', 'id': '15', 'fold': '3', 'target': 'proxy', 'dice': '0.7789459065345754', 'precision': '0.7303315190499753', 'recall': '0.8344938403856454', 'centerline_precision': '0.8618661787767904', 'centerline_recall': '0.8778584880279795', 'cldice': '0.8697888292554169', 'foreground_fraction': '0.14648056030273438', 'thin_clean_centerline_recall': '0.8900668356764231'}, {'experiment': 'L2_soft_none_t0.1', 'id': '16', 'fold': '3', 'target': 'clean', 'dice': '0.72619286928286', 'precision': '0.6083029727826593', 'recall': '0.9007616284341282', 'centerline_precision': '0.7272727272727273', 'centerline_recall': '0.8068137318887874', 'cldice': '0.764981164767673', 'foreground_fraction': '0.12459945678710938', 'thin_clean_centerline_recall': '0.6928512053200333'}, {'experiment': 'L2_soft_none_t0.1', 'id': '16', 'fold': '3', 'target': 'proxy', 'dice': '0.7985319334559753', 'precision': '0.8126626458071824', 'recall': '0.7848842366716934', 'centerline_precision': '0.9050915243808645', 'centerline_recall': '0.7743673264845903', 'cldice': '0.8346418295363415', 'foreground_fraction': '0.12459945678710938', 'thin_clean_centerline_recall': '0.6928512053200333'}, {'experiment': 'L2_soft_none_t0.1', 'id': '17', 'fold': '1', 'target': 'clean', 'dice': '0.7278569719349676', 'precision': '0.6111717090533325', 'recall': '0.8996112895048166', 'centerline_precision': '0.7518716577540107', 'centerline_recall': '0.8369460500963392', 'cldice': '0.7921311690162469', 'foreground_fraction': '0.13289642333984375', 'thin_clean_centerline_recall': '0.7418001525553013'}, {'experiment': 'L2_soft_none_t0.1', 'id': '17', 'fold': '1', 'target': 'proxy', 'dice': '0.7994638826187931', 'precision': '0.8132786038234112', 'recall': '0.7861106486876422', 'centerline_precision': '0.8986631016042781', 'centerline_recall': '0.805681685877285', 'cldice': '0.8496360690093958', 'foreground_fraction': '0.13289642333984375', 'thin_clean_centerline_recall': '0.7418001525553013'}, {'experiment': 'L2_soft_none_t0.1', 'id': '18', 'fold': '3', 'target': 'clean', 'dice': '0.7096806320966242', 'precision': '0.5658434580057704', 'recall': '0.9515691075005565', 'centerline_precision': '0.7681420483787956', 'centerline_recall': '0.9395286195286195', 'cldice': '0.8452349178071658', 'foreground_fraction': '0.14411544799804688', 'thin_clean_centerline_recall': '0.9011982817092471'}, {'experiment': 'L2_soft_none_t0.1', 'id': '18', 'fold': '3', 'target': 'proxy', 'dice': '0.7882130608248222', 'precision': '0.7452023611000821', 'recall': '0.8364927501782743', 'centerline_precision': '0.8676016469377251', 'centerline_recall': '0.8904849762783342', 'cldice': '0.8788943864200452', 'foreground_fraction': '0.14411544799804688', 'thin_clean_centerline_recall': '0.9011982817092471'}, {'experiment': 'L2_soft_none_t0.1', 'id': '19', 'fold': '3', 'target': 'clean', 'dice': '0.7738780312923574', 'precision': '0.7184896900177523', 'recall': '0.8385194629268098', 'centerline_precision': '0.8105849582172702', 'centerline_recall': '0.7687253613666228', 'cldice': '0.7891004157917425', 'foreground_fraction': '0.1117401123046875', 'thin_clean_centerline_recall': '0.574741618351399'}, {'experiment': 'L2_soft_none_t0.1', 'id': '19', 'fold': '3', 'target': 'proxy', 'dice': '0.7826232399723525', 'precision': '0.8785675269698211', 'recall': '0.7055710917365795', 'centerline_precision': '0.9370801245289202', 'centerline_recall': '0.7426780918275995', 'cldice': '0.8286298254093449', 'foreground_fraction': '0.1117401123046875', 'thin_clean_centerline_recall': '0.574741618351399'}, {'experiment': 'L2_soft_none_t0.1', 'id': '20', 'fold': '1', 'target': 'clean', 'dice': '0.7761314655176184', 'precision': '0.6844449723822533', 'recall': '0.8961816626487286', 'centerline_precision': '0.8118592407869216', 'centerline_recall': '0.843354807921737', 'cldice': '0.8273073740613905', 'foreground_fraction': '0.12845611572265625', 'thin_clean_centerline_recall': '0.7405782652043869'}, {'experiment': 'L2_soft_none_t0.1', 'id': '20', 'fold': '1', 'target': 'proxy', 'dice': '0.8103692984167612', 'precision': '0.8647324345192137', 'recall': '0.7624371596145789', 'centerline_precision': '0.9379329454142422', 'centerline_recall': '0.8068804664723032', 'cldice': '0.867485047237584', 'foreground_fraction': '0.12845611572265625', 'thin_clean_centerline_recall': '0.7405782652043869'}, {'experiment': 'L2_soft_none_t0.4', 'id': '1', 'fold': '2', 'target': 'clean', 'dice': '0.7958567457861668', 'precision': '0.733208061604215', 'recall': '0.8702116494665034', 'centerline_precision': '0.8587115666178624', 'centerline_recall': '0.8599668233342549', 'cldice': '0.8593387365804003', 'foreground_fraction': '0.10353469848632812', 'thin_clean_centerline_recall': '0.7621359223300971'}, {'experiment': 'L2_soft_none_t0.4', 'id': '1', 'fold': '2', 'target': 'proxy', 'dice': '0.7704366933966802', 'precision': '0.8655171143288751', 'recall': '0.694178486997636', 'centerline_precision': '0.922108345534407', 'centerline_recall': '0.7791888570258091', 'cldice': '0.8446455407434714', 'foreground_fraction': '0.10353469848632812', 'thin_clean_centerline_recall': '0.7621359223300971'}, {'experiment': 'L2_soft_none_t0.4', 'id': '2', 'fold': '0', 'target': 'clean', 'dice': '0.7657605293754678', 'precision': '0.747871271996856', 'recall': '0.784526590628006', 'centerline_precision': '0.7843066353281504', 'centerline_recall': '0.7593443627450981', 'cldice': '0.7716236674525134', 'foreground_fraction': '0.08736038208007812', 'thin_clean_centerline_recall': '0.5664924506387921'}, {'experiment': 'L2_soft_none_t0.4', 'id': '2', 'fold': '0', 'target': 'proxy', 'dice': '0.7594122377371164', 'precision': '0.9072092921706476', 'recall': '0.653025302530253', 'centerline_precision': '0.9325619237027662', 'centerline_recall': '0.7383729514247749', 'cldice': '0.8241835277251229', 'foreground_fraction': '0.08736038208007812', 'thin_clean_centerline_recall': '0.5664924506387921'}, {'experiment': 'L2_soft_none_t0.4', 'id': '3', 'fold': '1', 'target': 'clean', 'dice': '0.8183306625795682', 'precision': '0.8124718984539826', 'recall': '0.8242745359486298', 'centerline_precision': '0.9238451935081149', 'centerline_recall': '0.8188254410399257', 'cldice': '0.8681708786848142', 'foreground_fraction': '0.11029434204101562', 'thin_clean_centerline_recall': '0.6881562368752625'}, {'experiment': 'L2_soft_none_t0.4', 'id': '3', 'fold': '1', 'target': 'proxy', 'dice': '0.7623039635019072', 'precision': '0.9246359768962059', 'recall': '0.6484585344555752', 'centerline_precision': '0.9718407546122902', 'centerline_recall': '0.7682954806148199', 'cldice': '0.8581636822802784', 'foreground_fraction': '0.11029434204101562', 'thin_clean_centerline_recall': '0.6881562368752625'}, {'experiment': 'L2_soft_none_t0.4', 'id': '4', 'fold': '2', 'target': 'clean', 'dice': '0.754277424585582', 'precision': '0.6525655223329642', 'recall': '0.8935503437120906', 'centerline_precision': '0.8090599876314162', 'centerline_recall': '0.8637780086066182', 'cldice': '0.8355240932250061', 'foreground_fraction': '0.10334014892578125', 'thin_clean_centerline_recall': '0.7845053635280095'}, {'experiment': 'L2_soft_none_t0.4', 'id': '4', 'fold': '2', 'target': 'proxy', 'dice': '0.7556271222043693', 'precision': '0.7968253968253968', 'recall': '0.7184795633071495', 'centerline_precision': '0.890074211502783', 'centerline_recall': '0.7642111368909513', 'cldice': '0.8223546509074978', 'foreground_fraction': '0.10334014892578125', 'thin_clean_centerline_recall': '0.7845053635280095'}, {'experiment': 'L2_soft_none_t0.4', 'id': '5', 'fold': '2', 'target': 'clean', 'dice': '0.7568763513859383', 'precision': '0.6724746459366483', 'recall': '0.8655051463400609', 'centerline_precision': '0.6964098989194841', 'centerline_recall': '0.8797286184210527', 'cldice': '0.7774084720230393', 'foreground_fraction': '0.08538436889648438', 'thin_clean_centerline_recall': '0.7461116193961573'}, {'experiment': 'L2_soft_none_t0.4', 'id': '5', 'fold': '2', 'target': 'proxy', 'dice': '0.7325868528616046', 'precision': '0.7687530715274985', 'recall': '0.6996706379864189', 'centerline_precision': '0.7380620425235274', 'centerline_recall': '0.8027908885696696', 'cldice': '0.7690668862428512', 'foreground_fraction': '0.08538436889648438', 'thin_clean_centerline_recall': '0.7461116193961573'}, {'experiment': 'L2_soft_none_t0.4', 'id': '6', 'fold': '0', 'target': 'clean', 'dice': '0.813822979226455', 'precision': '0.8162875670693571', 'recall': '0.8113732290678272', 'centerline_precision': '0.8955996548748921', 'centerline_recall': '0.7259935553168636', 'cldice': '0.8019268624173529', 'foreground_fraction': '0.11588668823242188', 'thin_clean_centerline_recall': '0.4897834082550061'}, {'experiment': 'L2_soft_none_t0.4', 'id': '6', 'fold': '0', 'target': 'proxy', 'dice': '0.74591568911063', 'precision': '0.9167846209552651', 'recall': '0.6287333227983837', 'centerline_precision': '0.9636180615473109', 'centerline_recall': '0.6415427799147521', 'cldice': '0.7702682423002997', 'foreground_fraction': '0.11588668823242188', 'thin_clean_centerline_recall': '0.4897834082550061'}, {'experiment': 'L2_soft_none_t0.4', 'id': '7', 'fold': '4', 'target': 'clean', 'dice': '0.7524729830186047', 'precision': '0.6884285415358862', 'recall': '0.8296557811120918', 'centerline_precision': '0.8284470911589555', 'centerline_recall': '0.7564582219846784', 'cldice': '0.7908177332606218', 'foreground_fraction': '0.0729217529296875', 'thin_clean_centerline_recall': '0.613740022805017'}, {'experiment': 'L2_soft_none_t0.4', 'id': '7', 'fold': '4', 'target': 'proxy', 'dice': '0.7459342580458079', 'precision': '0.848189997907512', 'recall': '0.6656813236441269', 'centerline_precision': '0.940219880897847', 'centerline_recall': '0.6974143955276031', 'cldice': '0.8008172390366517', 'foreground_fraction': '0.0729217529296875', 'thin_clean_centerline_recall': '0.613740022805017'}, {'experiment': 'L2_soft_none_t0.4', 'id': '8', 'fold': '4', 'target': 'clean', 'dice': '0.7458484400882999', 'precision': '0.6904653379190321', 'recall': '0.8108910409265657', 'centerline_precision': '0.7510442773600668', 'centerline_recall': '0.7711620249691316', 'cldice': '0.7609702113100465', 'foreground_fraction': '0.09206008911132812', 'thin_clean_centerline_recall': '0.6384007029876977'}, {'experiment': 'L2_soft_none_t0.4', 'id': '8', 'fold': '4', 'target': 'proxy', 'dice': '0.7621588711414616', 'precision': '0.8818215721211619', 'recall': '0.6710920500772602', 'centerline_precision': '0.9304928989139516', 'centerline_recall': '0.729525576464352', 'cldice': '0.8178443535955102', 'foreground_fraction': '0.09206008911132812', 'thin_clean_centerline_recall': '0.6384007029876977'}, {'experiment': 'L2_soft_none_t0.4', 'id': '9', 'fold': '0', 'target': 'clean', 'dice': '0.7863154681440271', 'precision': '0.7324910627234319', 'recall': '0.8486773980984655', 'centerline_precision': '0.833993399339934', 'centerline_recall': '0.7887612797374898', 'cldice': '0.8107469470739394', 'foreground_fraction': '0.093902587890625', 'thin_clean_centerline_recall': '0.6593333333333333'}, {'experiment': 'L2_soft_none_t0.4', 'id': '9', 'fold': '0', 'target': 'proxy', 'dice': '0.7632436622451892', 'precision': '0.879996750081248', 'recall': '0.6738420381373067', 'centerline_precision': '0.9501650165016502', 'centerline_recall': '0.7205199628597957', 'cldice': '0.8195594871058369', 'foreground_fraction': '0.093902587890625', 'thin_clean_centerline_recall': '0.6593333333333333'}, {'experiment': 'L2_soft_none_t0.4', 'id': '10', 'fold': '0', 'target': 'clean', 'dice': '0.7772408286763516', 'precision': '0.7038587411522947', 'recall': '0.8677050103208858', 'centerline_precision': '0.7989716370874108', 'centerline_recall': '0.7911551339285714', 'cldice': '0.7950441739197247', 'foreground_fraction': '0.10024261474609375', 'thin_clean_centerline_recall': '0.6389294403892944'}, {'experiment': 'L2_soft_none_t0.4', 'id': '10', 'fold': '0', 'target': 'proxy', 'dice': '0.7779068972592489', 'precision': '0.866123753710328', 'recall': '0.705999131459768', 'centerline_precision': '0.9333222756676066', 'centerline_recall': '0.7202460550949452', 'cldice': '0.8130558316532731', 'foreground_fraction': '0.10024261474609375', 'thin_clean_centerline_recall': '0.6389294403892944'}, {'experiment': 'L2_soft_none_t0.4', 'id': '11', 'fold': '2', 'target': 'clean', 'dice': '0.7838364822165808', 'precision': '0.6993401319736052', 'recall': '0.8915570510859591', 'centerline_precision': '0.7794561933534743', 'centerline_recall': '0.8737109550141982', 'cldice': '0.8238966226240705', 'foreground_fraction': '0.09538650512695312', 'thin_clean_centerline_recall': '0.7938347718865598'}, {'experiment': 'L2_soft_none_t0.4', 'id': '11', 'fold': '2', 'target': 'proxy', 'dice': '0.7992727539855299', 'precision': '0.8702659468106378', 'recall': '0.7389886915475261', 'centerline_precision': '0.9303545873747814', 'centerline_recall': '0.8047216554940251', 'cldice': '0.8629897237378105', 'foreground_fraction': '0.09538650512695312', 'thin_clean_centerline_recall': '0.7938347718865598'}, {'experiment': 'L2_soft_none_t0.4', 'id': '12', 'fold': '4', 'target': 'clean', 'dice': '0.6443155216292228', 'precision': '0.6764982474138668', 'recall': '0.615055769305507', 'centerline_precision': '0.8014388489208633', 'centerline_recall': '0.6203438395415473', 'cldice': '0.699358146264939', 'foreground_fraction': '0.08924102783203125', 'thin_clean_centerline_recall': '0.5369811320754717'}, {'experiment': 'L2_soft_none_t0.4', 'id': '12', 'fold': '4', 'target': 'proxy', 'dice': '0.6201395485969468', 'precision': '0.8092245875010686', 'recall': '0.5026818906001063', 'centerline_precision': '0.887589928057554', 'centerline_recall': '0.5530284893464209', 'cldice': '0.6814608381271596', 'foreground_fraction': '0.08924102783203125', 'thin_clean_centerline_recall': '0.5369811320754717'}, {'experiment': 'L2_soft_none_t0.4', 'id': '13', 'fold': '4', 'target': 'clean', 'dice': '0.786266306700356', 'precision': '0.7528613150209172', 'recall': '0.8227733448350226', 'centerline_precision': '0.8355010345846882', 'centerline_recall': '0.7727868461813416', 'cldice': '0.8029211899432066', 'foreground_fraction': '0.09665679931640625', 'thin_clean_centerline_recall': '0.6514084507042254'}, {'experiment': 'L2_soft_none_t0.4', 'id': '13', 'fold': '4', 'target': 'proxy', 'dice': '0.7433373192030281', 'precision': '0.8954929355118794', 'recall': '0.6353784548178433', 'centerline_precision': '0.9527046999704404', 'centerline_recall': '0.680209545983702', 'cldice': '0.7937205925284472', 'foreground_fraction': '0.09665679931640625', 'thin_clean_centerline_recall': '0.6514084507042254'}, {'experiment': 'L2_soft_none_t0.4', 'id': '14', 'fold': '1', 'target': 'clean', 'dice': '0.7956322677367295', 'precision': '0.7345328046682128', 'recall': '0.867818631783631', 'centerline_precision': '0.8592760180995475', 'centerline_recall': '0.8565446440632176', 'cldice': '0.8579081570767529', 'foreground_fraction': '0.10198211669921875', 'thin_clean_centerline_recall': '0.7609039548022599'}, {'experiment': 'L2_soft_none_t0.4', 'id': '14', 'fold': '1', 'target': 'proxy', 'dice': '0.7890724479506059', 'precision': '0.892421635370689', 'recall': '0.7071760974597623', 'centerline_precision': '0.9533936651583711', 'centerline_recall': '0.8214429125953687', 'cldice': '0.8825133299301768', 'foreground_fraction': '0.10198211669921875', 'thin_clean_centerline_recall': '0.7609039548022599'}, {'experiment': 'L2_soft_none_t0.4', 'id': '15', 'fold': '3', 'target': 'clean', 'dice': '0.7615547431214902', 'precision': '0.6743797709923665', 'recall': '0.8746132767612481', 'centerline_precision': '0.8463747307968413', 'centerline_recall': '0.8493001766544367', 'cldice': '0.8478349301778091', 'foreground_fraction': '0.1119384765625', 'thin_clean_centerline_recall': '0.7566259506798801'}, {'experiment': 'L2_soft_none_t0.4', 'id': '15', 'fold': '3', 'target': 'proxy', 'dice': '0.7539316918192948', 'precision': '0.8086832061068703', 'recall': '0.7061239064452777', 'centerline_precision': '0.9098348887293611', 'centerline_recall': '0.7597524885660479', 'cldice': '0.82804809175799', 'foreground_fraction': '0.1119384765625', 'thin_clean_centerline_recall': '0.7566259506798801'}, {'experiment': 'L2_soft_none_t0.4', 'id': '16', 'fold': '3', 'target': 'clean', 'dice': '0.7717405103470616', 'precision': '0.7235756229169973', 'recall': '0.8267748662616737', 'centerline_precision': '0.7786195286195287', 'centerline_recall': '0.7328025062002349', 'cldice': '0.755016572213567', 'foreground_fraction': '0.0961456298828125', 'thin_clean_centerline_recall': '0.5764754779717374'}, {'experiment': 'L2_soft_none_t0.4', 'id': '16', 'fold': '3', 'target': 'proxy', 'dice': '0.75929722311682', 'precision': '0.8890652277416283', 'recall': '0.662586120228274', 'centerline_precision': '0.9454545454545454', 'centerline_recall': '0.6767727386619895', 'cldice': '0.7888633957431751', 'foreground_fraction': '0.0961456298828125', 'thin_clean_centerline_recall': '0.5764754779717374'}, {'experiment': 'L2_soft_none_t0.4', 'id': '17', 'fold': '1', 'target': 'clean', 'dice': '0.7763120394125315', 'precision': '0.7490767659306985', 'recall': '0.8056025012675342', 'centerline_precision': '0.808987052551409', 'centerline_recall': '0.7434971098265896', 'cldice': '0.7748607683543709', 'foreground_fraction': '0.09709930419921875', 'thin_clean_centerline_recall': '0.5964912280701754'}, {'experiment': 'L2_soft_none_t0.4', 'id': '17', 'fold': '1', 'target': 'proxy', 'dice': '0.748243788214277', 'precision': '0.9038657971242241', 'recall': '0.638338604960879', 'centerline_precision': '0.948971820258949', 'centerline_recall': '0.6941436721387821', 'cldice': '0.8017973016729628', 'foreground_fraction': '0.09709930419921875', 'thin_clean_centerline_recall': '0.5964912280701754'}, {'experiment': 'L2_soft_none_t0.4', 'id': '18', 'fold': '3', 'target': 'clean', 'dice': '0.7668508287297419', 'precision': '0.6887116781853624', 'recall': '0.8649899844202092', 'centerline_precision': '0.829243648960739', 'centerline_recall': '0.8522558922558923', 'cldice': '0.8405923030240271', 'foreground_fraction': '0.10763168334960938', 'thin_clean_centerline_recall': '0.7607958399276509'}, {'experiment': 'L2_soft_none_t0.4', 'id': '18', 'fold': '3', 'target': 'proxy', 'dice': '0.7552488241668064', 'precision': '0.8280701754385965', 'recall': '0.6942001426194437', 'centerline_precision': '0.9217667436489607', 'centerline_recall': '0.7583025830258303', 'cldice': '0.8320824522637281', 'foreground_fraction': '0.10763168334960938', 'thin_clean_centerline_recall': '0.7607958399276509'}, {'experiment': 'L2_soft_none_t0.4', 'id': '19', 'fold': '3', 'target': 'clean', 'dice': '0.7712065754836979', 'precision': '0.8332562442183163', 'recall': '0.7177576795888282', 'centerline_precision': '0.8697563656347317', 'centerline_recall': '0.6743758212877793', 'cldice': '0.7597052485049074', 'foreground_fraction': '0.0824737548828125', 'thin_clean_centerline_recall': '0.41643559364759264'}, {'experiment': 'L2_soft_none_t0.4', 'id': '19', 'fold': '3', 'target': 'proxy', 'dice': '0.7028953076054276', 'precision': '0.9443570767807585', 'recall': '0.5597686022920436', 'centerline_precision': '0.9743542773401722', 'centerline_recall': '0.639851643432664', 'cldice': '0.7724444293244426', 'foreground_fraction': '0.0824737548828125', 'thin_clean_centerline_recall': '0.41643559364759264'}, {'experiment': 'L2_soft_none_t0.4', 'id': '20', 'fold': '1', 'target': 'clean', 'dice': '0.7961681952436729', 'precision': '0.7957352598461896', 'recall': '0.7966016019908235', 'centerline_precision': '0.8517446289806491', 'centerline_recall': '0.76795514197089', 'cldice': '0.8076826078545489', 'foreground_fraction': '0.09821319580078125', 'thin_clean_centerline_recall': '0.6203389830508474'}, {'experiment': 'L2_soft_none_t0.4', 'id': '20', 'fold': '1', 'target': 'proxy', 'dice': '0.747192592824694', 'precision': '0.927794608871281', 'recall': '0.6254451193967323', 'centerline_precision': '0.9676976992229164', 'centerline_recall': '0.7051895043731778', 'cldice': '0.8158473080924297', 'foreground_fraction': '0.09821319580078125', 'thin_clean_centerline_recall': '0.6203389830508474'}, {'experiment': 'L2_soft_none_t0.5', 'id': '1', 'fold': '2', 'target': 'clean', 'dice': '0.8006944587979234', 'precision': '0.7591612776797962', 'recall': '0.8470351582998076', 'centerline_precision': '0.8718449519230769', 'centerline_recall': '0.8408902405308266', 'cldice': '0.8560878699267043', 'foreground_fraction': '0.09733200073242188', 'thin_clean_centerline_recall': '0.7293689320388349'}, {'experiment': 'L2_soft_none_t0.5', 'id': '1', 'fold': '2', 'target': 'proxy', 'dice': '0.7585207648896327', 'precision': '0.882265334117186', 'recall': '0.6652186761229314', 'centerline_precision': '0.9323918269230769', 'centerline_recall': '0.7528335381674177', 'cldice': '0.8330468465067156', 'foreground_fraction': '0.09733200073242188', 'thin_clean_centerline_recall': '0.7293689320388349'}, {'experiment': 'L2_soft_none_t0.5', 'id': '2', 'fold': '0', 'target': 'clean', 'dice': '0.7644614745400387', 'precision': '0.7705085849890652', 'recall': '0.7585085428977143', 'centerline_precision': '0.8007462686567164', 'centerline_recall': '0.7359068627450981', 'cldice': '0.766958609435092', 'foreground_fraction': '0.08198165893554688', 'thin_clean_centerline_recall': '0.5261324041811847'}, {'experiment': 'L2_soft_none_t0.5', 'id': '2', 'fold': '0', 'target': 'proxy', 'dice': '0.7417551495146104', 'precision': '0.9199199664976037', 'recall': '0.6214049976426214', 'centerline_precision': '0.9438432835820896', 'centerline_recall': '0.7147497416211428', 'cldice': '0.8134747135916448', 'foreground_fraction': '0.08198165893554688', 'thin_clean_centerline_recall': '0.5261324041811847'}, {'experiment': 'L2_soft_none_t0.5', 'id': '3', 'fold': '1', 'target': 'clean', 'dice': '0.8159718481815936', 'precision': '0.8354718923489834', 'recall': '0.7973613109231903', 'centerline_precision': '0.9265744140073425', 'centerline_recall': '0.7996750232126276', 'cldice': '0.858460428781163', 'foreground_fraction': '0.10375595092773438', 'thin_clean_centerline_recall': '0.6562368752624947'}, {'experiment': 'L2_soft_none_t0.5', 'id': '3', 'fold': '1', 'target': 'proxy', 'dice': '0.7432554876804968', 'precision': '0.9349240780911063', 'recall': '0.616804521308851', 'centerline_precision': '0.974159841852584', 'centerline_recall': '0.7452397338839183', 'cldice': '0.8444606263109563', 'foreground_fraction': '0.10375595092773438', 'thin_clean_centerline_recall': '0.6562368752624947'}, {'experiment': 'L2_soft_none_t0.5', 'id': '4', 'fold': '2', 'target': 'clean', 'dice': '0.7627724344838904', 'precision': '0.6767261625176139', 'recall': '0.873887990295188', 'centerline_precision': '0.8192143871273071', 'centerline_recall': '0.8441905327199881', 'cldice': '0.831514950604292', 'foreground_fraction': '0.0974578857421875', 'thin_clean_centerline_recall': '0.7537544696066746'}, {'experiment': 'L2_soft_none_t0.5', 'id': '4', 'fold': '2', 'target': 'proxy', 'dice': '0.7470139588434541', 'precision': '0.8127446375450134', 'recall': '0.6911196911196911', 'centerline_precision': '0.9015617605300521', 'centerline_recall': '0.7369489559164734', 'cldice': '0.8109864542817876', 'foreground_fraction': '0.0974578857421875', 'thin_clean_centerline_recall': '0.7537544696066746'}, {'experiment': 'L2_soft_none_t0.5', 'id': '5', 'fold': '2', 'target': 'clean', 'dice': '0.7653713208343842', 'precision': '0.7026975044477569', 'recall': '0.8403197055948479', 'centerline_precision': '0.7202326849663697', 'centerline_recall': '0.8636924342105263', 'cldice': '0.7854658194951122', 'foreground_fraction': '0.07933425903320312', 'thin_clean_centerline_recall': '0.7140896614821592'}, {'experiment': 'L2_soft_none_t0.5', 'id': '5', 'fold': '2', 'target': 'proxy', 'dice': '0.7244326944266921', 'precision': '0.7905467134682886', 'recall': '0.668523563615663', 'centerline_precision': '0.7624068351208871', 'centerline_recall': '0.7742663656884876', 'cldice': '0.7682908364565656', 'foreground_fraction': '0.07933425903320312', 'thin_clean_centerline_recall': '0.7140896614821592'}, {'experiment': 'L2_soft_none_t0.5', 'id': '6', 'fold': '0', 'target': 'clean', 'dice': '0.810398848336336', 'precision': '0.8400042131872761', 'recall': '0.7828092791937964', 'centerline_precision': '0.9113491001041202', 'centerline_recall': '0.6987110633727175', 'cldice': '0.7909887012699567', 'foreground_fraction': '0.10865020751953125', 'thin_clean_centerline_recall': '0.44442174090723335'}, {'experiment': 'L2_soft_none_t0.5', 'id': '6', 'fold': '0', 'target': 'proxy', 'dice': '0.7267206199594296', 'precision': '0.9284811459869391', 'recall': '0.5969930243583087', 'centerline_precision': '0.9711438345976499', 'centerline_recall': '0.6146169040440794', 'cldice': '0.7528013557873267', 'foreground_fraction': '0.10865020751953125', 'thin_clean_centerline_recall': '0.44442174090723335'}, {'experiment': 'L2_soft_none_t0.5', 'id': '7', 'fold': '4', 'target': 'clean', 'dice': '0.7564310818401121', 'precision': '0.7115630382841585', 'recall': '0.807338292775186', 'centerline_precision': '0.8348924228250701', 'centerline_recall': '0.738998752895065', 'cldice': '0.7840242944204379', 'foreground_fraction': '0.06865310668945312', 'thin_clean_centerline_recall': '0.5875142531356898'}, {'experiment': 'L2_soft_none_t0.5', 'id': '7', 'fold': '4', 'target': 'proxy', 'dice': '0.732587240875165', 'precision': '0.8620325609823859', 'recall': '0.636942152153385', 'centerline_precision': '0.9490177736202058', 'centerline_recall': '0.6743535988819007', 'cldice': '0.7884499651577344', 'foreground_fraction': '0.06865310668945312', 'thin_clean_centerline_recall': '0.5875142531356898'}, {'experiment': 'L2_soft_none_t0.5', 'id': '8', 'fold': '4', 'target': 'clean', 'dice': '0.7472680125955101', 'precision': '0.7127147462792033', 'recall': '0.7853423524259088', 'centerline_precision': '0.7589224433768016', 'centerline_recall': '0.748662367951708', 'cldice': '0.7537574924882471', 'foreground_fraction': '0.08637619018554688', 'thin_clean_centerline_recall': '0.6036906854130053'}, {'experiment': 'L2_soft_none_t0.5', 'id': '8', 'fold': '4', 'target': 'proxy', 'dice': '0.7471023291758709', 'precision': '0.8967009671863269', 'recall': '0.6402825517959068', 'centerline_precision': '0.9401166781056967', 'centerline_recall': '0.7059369202226345', 'cldice': '0.8063687270765182', 'foreground_fraction': '0.08637619018554688', 'thin_clean_centerline_recall': '0.6036906854130053'}, {'experiment': 'L2_soft_none_t0.5', 'id': '9', 'fold': '0', 'target': 'clean', 'dice': '0.7918068697640256', 'precision': '0.7610592576514', 'recall': '0.8251435564341523', 'centerline_precision': '0.8482370975983649', 'centerline_recall': '0.7662018047579984', 'cldice': '0.8051352009592411', 'foreground_fraction': '0.08787155151367188', 'thin_clean_centerline_recall': '0.6228888888888889'}, {'experiment': 'L2_soft_none_t0.5', 'id': '9', 'fold': '0', 'target': 'proxy', 'dice': '0.7458591569719367', 'precision': '0.8933796396787498', 'recall': '0.6401530469406166', 'centerline_precision': '0.9560551865099642', 'centerline_recall': '0.6833797585886723', 'cldice': '0.7970414007678953', 'foreground_fraction': '0.08787155151367188', 'thin_clean_centerline_recall': '0.6228888888888889'}, {'experiment': 'L2_soft_none_t0.5', 'id': '10', 'fold': '0', 'target': 'clean', 'dice': '0.784716033819938', 'precision': '0.7310573846839186', 'recall': '0.8468755864139613', 'centerline_precision': '0.8120531154239019', 'centerline_recall': '0.7667410714285714', 'cldice': '0.7887468562553208', 'foreground_fraction': '0.09419631958007812', 'thin_clean_centerline_recall': '0.5970802919708029'}, {'experiment': 'L2_soft_none_t0.5', 'id': '10', 'fold': '0', 'target': 'proxy', 'dice': '0.7654177864436503', 'precision': '0.8823553233709959', 'recall': '0.6758483776909238', 'centerline_precision': '0.9356486210418795', 'centerline_recall': '0.6944370152447178', 'cldice': '0.7971961978565912', 'foreground_fraction': '0.09419631958007812', 'thin_clean_centerline_recall': '0.5970802919708029'}, {'experiment': 'L2_soft_none_t0.5', 'id': '11', 'fold': '2', 'target': 'clean', 'dice': '0.7907603017996733', 'precision': '0.7259281360555816', 'recall': '0.8683083511777302', 'centerline_precision': '0.7926569414850024', 'centerline_recall': '0.8521895082947243', 'cldice': '0.821345882226228', 'foreground_fraction': '0.08949661254882812', 'thin_clean_centerline_recall': '0.7585696670776819'}, {'experiment': 'L2_soft_none_t0.5', 'id': '11', 'fold': '2', 'target': 'proxy', 'dice': '0.7871399410301143', 'precision': '0.887558075103363', 'recall': '0.7071348524467688', 'centerline_precision': '0.94279626290772', 'centerline_recall': '0.7777615855435733', 'cldice': '0.8523639201596096', 'foreground_fraction': '0.08949661254882812', 'thin_clean_centerline_recall': '0.7585696670776819'}, {'experiment': 'L2_soft_none_t0.5', 'id': '12', 'fold': '4', 'target': 'clean', 'dice': '0.6371856337798865', 'precision': '0.6942038946162657', 'recall': '0.5888228207220862', 'centerline_precision': '0.8128481247679168', 'centerline_recall': '0.5953915950334289', 'cldice': '0.6873303383230408', 'foreground_fraction': '0.08325576782226562', 'thin_clean_centerline_recall': '0.5054716981132076'}, {'experiment': 'L2_soft_none_t0.5', 'id': '12', 'fold': '4', 'target': 'proxy', 'dice': '0.6029419181312886', 'precision': '0.8216723940435281', 'recall': '0.47618162506638345', 'centerline_precision': '0.896212402525065', 'centerline_recall': '0.5232224084271008', 'cldice': '0.6607114438694783', 'foreground_fraction': '0.08325576782226562', 'thin_clean_centerline_recall': '0.5054716981132076'}, {'experiment': 'L2_soft_none_t0.5', 'id': '13', 'fold': '4', 'target': 'clean', 'dice': '0.7878077873396357', 'precision': '0.7782266548836426', 'recall': '0.7976277765796851', 'centerline_precision': '0.8459672528805336', 'centerline_recall': '0.7509829619921363', 'cldice': '0.7956503432603613', 'foreground_fraction': '0.09064865112304688', 'thin_clean_centerline_recall': '0.6186063750926613'}, {'experiment': 'L2_soft_none_t0.5', 'id': '13', 'fold': '4', 'target': 'proxy', 'dice': '0.7260315431958065', 'precision': '0.908555317089593', 'recall': '0.604575620957128', 'centerline_precision': '0.9569436021831413', 'centerline_recall': '0.6477299185098953', 'cldice': '0.7725446870874405', 'foreground_fraction': '0.09064865112304688', 'thin_clean_centerline_recall': '0.6186063750926613'}, {'experiment': 'L2_soft_none_t0.5', 'id': '14', 'fold': '1', 'target': 'clean', 'dice': '0.7986147549589703', 'precision': '0.7604828716472799', 'recall': '0.8407724942549054', 'centerline_precision': '0.8694227769110764', 'centerline_recall': '0.8318249358368229', 'cldice': '0.8502083972838794', 'foreground_fraction': '0.09543228149414062', 'thin_clean_centerline_recall': '0.72'}, {'experiment': 'L2_soft_none_t0.5', 'id': '14', 'fold': '1', 'target': 'proxy', 'dice': '0.7727473874123076', 'precision': '0.9074229523923731', 'recall': '0.672881406171266', 'centerline_precision': '0.9630265210608424', 'centerline_recall': '0.7918618658814082', 'cldice': '0.8690968423231293', 'foreground_fraction': '0.09543228149414062', 'thin_clean_centerline_recall': '0.72'}, {'experiment': 'L2_soft_none_t0.5', 'id': '15', 'fold': '3', 'target': 'clean', 'dice': '0.7666480802935983', 'precision': '0.6976803189561436', 'recall': '0.8507469283125608', 'centerline_precision': '0.8563837638376384', 'centerline_recall': '0.8194048104362006', 'cldice': '0.837486287280715', 'foreground_fraction': '0.10524749755859375', 'thin_clean_centerline_recall': '0.709149573634478'}, {'experiment': 'L2_soft_none_t0.5', 'id': '15', 'fold': '3', 'target': 'proxy', 'dice': '0.7422380547752431', 'precision': '0.8231605654222545', 'recall': '0.6758019401297387', 'centerline_precision': '0.91690036900369', 'centerline_recall': '0.7243744955609362', 'cldice': '0.8093455357373893', 'foreground_fraction': '0.10524749755859375', 'thin_clean_centerline_recall': '0.709149573634478'}, {'experiment': 'L2_soft_none_t0.5', 'id': '16', 'fold': '3', 'target': 'clean', 'dice': '0.7740907304724838', 'precision': '0.7465470776486441', 'recall': '0.8037446731344637', 'centerline_precision': '0.7889829045069936', 'centerline_recall': '0.710220597833181', 'cldice': '0.74753281891943', 'foreground_fraction': '0.0905914306640625', 'thin_clean_centerline_recall': '0.5413549459684123'}, {'experiment': 'L2_soft_none_t0.5', 'id': '16', 'fold': '3', 'target': 'proxy', 'dice': '0.7440373825286292', 'precision': '0.9018022570321711', 'recall': '0.6332534965551909', 'centerline_precision': '0.9473320670005181', 'centerline_recall': '0.6495865697820096', 'cldice': '0.7707019927918477', 'foreground_fraction': '0.0905914306640625', 'thin_clean_centerline_recall': '0.5413549459684123'}, {'experiment': 'L2_soft_none_t0.5', 'id': '17', 'fold': '1', 'target': 'clean', 'dice': '0.7778552856875772', 'precision': '0.7783324167688989', 'recall': '0.7773787392259591', 'centerline_precision': '0.8241688986922956', 'centerline_recall': '0.7170038535645472', 'cldice': '0.7668605294053935', 'foreground_fraction': '0.09017562866210938', 'thin_clean_centerline_recall': '0.5556826849733029'}, {'experiment': 'L2_soft_none_t0.5', 'id': '17', 'fold': '1', 'target': 'proxy', 'dice': '0.7295454164646545', 'precision': '0.9209357417826474', 'recall': '0.6040175350979413', 'centerline_precision': '0.9569875531747282', 'centerline_recall': '0.6653859587844918', 'cldice': '0.7849808640490145', 'foreground_fraction': '0.09017562866210938', 'thin_clean_centerline_recall': '0.5556826849733029'}, {'experiment': 'L2_soft_none_t0.5', 'id': '18', 'fold': '3', 'target': 'clean', 'dice': '0.7704548250835204', 'precision': '0.7142694194137105', 'recall': '0.8362341419986646', 'centerline_precision': '0.8459471562919839', 'centerline_recall': '0.8220875420875421', 'cldice': '0.8338467049008468', 'foreground_fraction': '0.10033035278320312', 'thin_clean_centerline_recall': '0.7130906624463034'}, {'experiment': 'L2_soft_none_t0.5', 'id': '18', 'fold': '3', 'target': 'proxy', 'dice': '0.7395299964979243', 'precision': '0.8429337287555606', 'recall': '0.6587235559781317', 'centerline_precision': '0.9299895506792059', 'centerline_recall': '0.7183711122825513', 'cldice': '0.8105964221836941', 'foreground_fraction': '0.10033035278320312', 'thin_clean_centerline_recall': '0.7130906624463034'}, {'experiment': 'L2_soft_none_t0.5', 'id': '19', 'fold': '3', 'target': 'clean', 'dice': '0.7594465141037798', 'precision': '0.8563284492673902', 'recall': '0.6822582573010877', 'centerline_precision': '0.8833081570996979', 'centerline_recall': '0.6488830486202365', 'cldice': '0.7481620932299516', 'foreground_fraction': '0.07628250122070312', 'thin_clean_centerline_recall': '0.3763549281572977'}, {'experiment': 'L2_soft_none_t0.5', 'id': '19', 'fold': '3', 'target': 'proxy', 'dice': '0.6751429937495792', 'precision': '0.9532929939490924', 'recall': '0.5226462685748753', 'centerline_precision': '0.9784743202416919', 'centerline_recall': '0.6080061388924415', 'cldice': '0.7499851511316452', 'foreground_fraction': '0.07628250122070312', 'thin_clean_centerline_recall': '0.3763549281572977'}, {'experiment': 'L2_soft_none_t0.5', 'id': '20', 'fold': '1', 'target': 'clean', 'dice': '0.7940681362729578', 'precision': '0.8192870730295261', 'recall': '0.7703553931098841', 'centerline_precision': '0.8646370023419204', 'centerline_recall': '0.7458840372226199', 'cldice': '0.800882350736916', 'foreground_fraction': '0.09224700927734375', 'thin_clean_centerline_recall': '0.586839481555334'}, {'experiment': 'L2_soft_none_t0.5', 'id': '20', 'fold': '1', 'target': 'proxy', 'dice': '0.7275146695742978', 'precision': '0.9382598627077992', 'recall': '0.5940772936740679', 'centerline_precision': '0.9734582357533177', 'centerline_recall': '0.6751020408163265', 'cldice': '0.7972819082769673', 'foreground_fraction': '0.09224700927734375', 'thin_clean_centerline_recall': '0.586839481555334'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '1', 'fold': '2', 'target': 'clean', 'dice': '0.7256247829183678', 'precision': '0.5835128880376665', 'recall': '0.9592443589295084', 'centerline_precision': '0.7588873643507547', 'centerline_recall': '0.9398672933370197', 'cldice': '0.8397368152630574', 'foreground_fraction': '0.14340591430664062', 'thin_clean_centerline_recall': '0.8944174757281553'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '1', 'fold': '2', 'target': 'proxy', 'dice': '0.8062940097716653', 'precision': '0.7660468704280052', 'recall': '0.8510047281323877', 'centerline_precision': '0.8358488212548335', 'centerline_recall': '0.9074149938549775', 'cldice': '0.8701629052684355', 'foreground_fraction': '0.14340591430664062', 'thin_clean_centerline_recall': '0.8944174757281553'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '2', 'fold': '0', 'target': 'clean', 'dice': '0.7413187382128226', 'precision': '0.6215621221047409', 'recall': '0.9182355366222344', 'centerline_precision': '0.7357570034623859', 'centerline_recall': '0.8684129901960784', 'cldice': '0.7966000385997877', 'foreground_fraction': '0.12302780151367188', 'thin_clean_centerline_recall': '0.7560975609756098'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '2', 'fold': '0', 'target': 'proxy', 'dice': '0.8179377516938502', 'precision': '0.8124089175529441', 'recall': '0.8235423542354235', 'centerline_precision': '0.8836953100409191', 'centerline_recall': '0.8520596486047541', 'cldice': '0.8675891854396711', 'foreground_fraction': '0.12302780151367188', 'thin_clean_centerline_recall': '0.7560975609756098'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '3', 'fold': '1', 'target': 'clean', 'dice': '0.7793060608752931', 'precision': '0.6735523456474498', 'recall': '0.9244534895961262', 'centerline_precision': '0.871465935982065', 'centerline_recall': '0.8955431754874652', 'cldice': '0.8833405176835639', 'foreground_fraction': '0.14921188354492188', 'thin_clean_centerline_recall': '0.8168836623267535'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '3', 'fold': '1', 'target': 'proxy', 'dice': '0.8289811057730341', 'precision': '0.8513613703182922', 'recall': '0.8077473500375967', 'centerline_precision': '0.9384730352472288', 'centerline_recall': '0.8706125258086718', 'cldice': '0.903270024600759', 'foreground_fraction': '0.14921188354492188', 'thin_clean_centerline_recall': '0.8168836623267535'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '4', 'fold': '2', 'target': 'clean', 'dice': '0.6882589632094545', 'precision': '0.5351741167887308', 'recall': '0.9640113222806308', 'centerline_precision': '0.7487108013937283', 'centerline_recall': '0.9436118118415195', 'cldice': '0.8349381498812879', 'foreground_fraction': '0.13594436645507812', 'thin_clean_centerline_recall': '0.9108462455303933'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '4', 'fold': '2', 'target': 'proxy', 'dice': '0.7793425800463157', 'precision': '0.7181861548390718', 'recall': '0.8518839036080416', 'centerline_precision': '0.8342857142857143', 'centerline_recall': '0.8900812064965197', 'cldice': '0.8612807705652087', 'foreground_fraction': '0.13594436645507812', 'thin_clean_centerline_recall': '0.9108462455303933'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '5', 'fold': '2', 'target': 'clean', 'dice': '0.6545632589117615', 'precision': '0.4963319373904779', 'recall': '0.9608993157380255', 'centerline_precision': '0.5552218430034129', 'centerline_recall': '0.9496299342105263', 'cldice': '0.7007404851788539', 'foreground_fraction': '0.12843704223632812', 'thin_clean_centerline_recall': '0.8879231473010064'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '5', 'fold': '2', 'target': 'proxy', 'dice': '0.7219456935914971', 'precision': '0.6246398764442068', 'recall': '0.8551620379782865', 'centerline_precision': '0.6061433447098976', 'centerline_recall': '0.9193515288323415', 'cldice': '0.7305941439929244', 'foreground_fraction': '0.12843704223632812', 'thin_clean_centerline_recall': '0.8879231473010064'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '6', 'fold': '0', 'target': 'clean', 'dice': '0.7757328068264933', 'precision': '0.6710300019109497', 'recall': '0.9191506069430357', 'centerline_precision': '0.8330555199384536', 'centerline_recall': '0.8237379162191192', 'cldice': '0.8283705175467299', 'foreground_fraction': '0.159698486328125', 'thin_clean_centerline_recall': '0.6646914589293011'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '6', 'fold': '0', 'target': 'proxy', 'dice': '0.8174464084680801', 'precision': '0.8412000764379897', 'recall': '0.7949974038873965', 'centerline_precision': '0.9322990126939351', 'centerline_recall': '0.7784592993034619', 'cldice': '0.8484621481285363', 'foreground_fraction': '0.159698486328125', 'thin_clean_centerline_recall': '0.6646914589293011'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '7', 'fold': '4', 'target': 'clean', 'dice': '0.7011394778602309', 'precision': '0.5666381722101337', 'recall': '0.9193670407262641', 'centerline_precision': '0.7779128672745694', 'centerline_recall': '0.839836094779975', 'cldice': '0.807689350888225', 'foreground_fraction': '0.098175048828125', 'thin_clean_centerline_recall': '0.7442987457240593'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '7', 'fold': '4', 'target': 'proxy', 'dice': '0.7826642445056542', 'precision': '0.7616956792042275', 'recall': '0.8048199696185901', 'centerline_precision': '0.8988855116514691', 'centerline_recall': '0.8034591194968553', 'cldice': '0.8484977113392067', 'foreground_fraction': '0.098175048828125', 'thin_clean_centerline_recall': '0.7442987457240593'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '8', 'fold': '4', 'target': 'clean', 'dice': '0.6902244750186636', 'precision': '0.5495373439797065', 'recall': '0.9277337096695703', 'centerline_precision': '0.7305551376560855', 'centerline_recall': '0.8717245163945672', 'cldice': '0.794920939628539', 'foreground_fraction': '0.13233566284179688', 'thin_clean_centerline_recall': '0.7954745166959578'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '8', 'fold': '4', 'target': 'proxy', 'dice': '0.79395198939822', 'precision': '0.7598512582514196', 'recall': '0.8312572924221879', 'centerline_precision': '0.8975477659094329', 'centerline_recall': '0.8348794063079777', 'cldice': '0.8650801118253072', 'foreground_fraction': '0.13233566284179688', 'thin_clean_centerline_recall': '0.7954745166959578'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '9', 'fold': '0', 'target': 'clean', 'dice': '0.7260266501850133', 'precision': '0.5914462577377603', 'recall': '0.9398945683893439', 'centerline_precision': '0.7678806494076349', 'centerline_recall': '0.8858353841946951', 'cldice': '0.8226513334359509', 'foreground_fraction': '0.12879562377929688', 'thin_clean_centerline_recall': '0.8148888888888889'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '9', 'fold': '0', 'target': 'proxy', 'dice': '0.8129874070705311', 'precision': '0.7935313805052868', 'recall': '0.8334214701216288', 'centerline_precision': '0.911657159572912', 'centerline_recall': '0.8526329751956493', 'cldice': '0.8811577427167713', 'foreground_fraction': '0.12879562377929688', 'thin_clean_centerline_recall': '0.8148888888888889'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '10', 'fold': '0', 'target': 'clean', 'dice': '0.7062706270632192', 'precision': '0.5625156462963478', 'recall': '0.9487239632201163', 'centerline_precision': '0.7366942637492608', 'centerline_recall': '0.892578125', 'cldice': '0.8071789458610489', 'foreground_fraction': '0.13714218139648438', 'thin_clean_centerline_recall': '0.8128953771289538'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '10', 'fold': '0', 'target': 'proxy', 'dice': '0.812037132089029', 'precision': '0.7701037523295596', 'recall': '0.8588001737080464', 'centerline_precision': '0.8994677705499704', 'centerline_recall': '0.8562449852901846', 'cldice': '0.8773243406721282', 'foreground_fraction': '0.13714218139648438', 'thin_clean_centerline_recall': '0.8128953771289538'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '11', 'fold': '2', 'target': 'clean', 'dice': '0.7085443097107518', 'precision': '0.5592060636446752', 'recall': '0.9667074538594881', 'centerline_precision': '0.7151515151515152', 'centerline_recall': '0.9429083843969511', 'cldice': '0.8133872122885099', 'foreground_fraction': '0.12934494018554688', 'thin_clean_centerline_recall': '0.906288532675709'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '11', 'fold': '2', 'target': 'proxy', 'dice': '0.8204691100801521', 'precision': '0.7665083905978116', 'recall': '0.8826026420348423', 'centerline_precision': '0.8662438336856941', 'centerline_recall': '0.9122704750801516', 'cldice': '0.8886615865801752', 'foreground_fraction': '0.12934494018554688', 'thin_clean_centerline_recall': '0.906288532675709'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '12', 'fold': '4', 'target': 'clean', 'dice': '0.6480408068635553', 'precision': '0.5646301694034804', 'recall': '0.7603668726438926', 'centerline_precision': '0.7212447288061655', 'centerline_recall': '0.7609837631327603', 'cldice': '0.7405815376664588', 'foreground_fraction': '0.13218307495117188', 'thin_clean_centerline_recall': '0.7120754716981132'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '12', 'fold': '4', 'target': 'proxy', 'dice': '0.6982616752642085', 'precision': '0.7285792617817668', 'recall': '0.6703664365374402', 'centerline_precision': '0.8252144830594736', 'centerline_recall': '0.7261192243236773', 'cldice': '0.772501876917984', 'foreground_fraction': '0.13218307495117188', 'thin_clean_centerline_recall': '0.7120754716981132'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '13', 'fold': '4', 'target': 'clean', 'dice': '0.7349298665758897', 'precision': '0.6089865343727853', 'recall': '0.926547336640069', 'centerline_precision': '0.7757470651013874', 'centerline_recall': '0.8701298701298701', 'cldice': '0.820232276862736', 'foreground_fraction': '0.13456344604492188', 'thin_clean_centerline_recall': '0.7979985174203114'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '13', 'fold': '4', 'target': 'proxy', 'dice': '0.8140478404194995', 'precision': '0.819078667611623', 'recall': '0.8090784352160398', 'centerline_precision': '0.9204909284951974', 'centerline_recall': '0.8334109429569266', 'cldice': '0.8747892059267012', 'foreground_fraction': '0.13456344604492188', 'thin_clean_centerline_recall': '0.7979985174203114'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '14', 'fold': '1', 'target': 'clean', 'dice': '0.7231921867688086', 'precision': '0.5817369780456306', 'recall': '0.9555418066112781', 'centerline_precision': '0.80323951141795', 'centerline_recall': '0.9324598135890855', 'cldice': '0.8630395302840068', 'foreground_fraction': '0.14178466796875', 'thin_clean_centerline_recall': '0.8870056497175142'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '14', 'fold': '1', 'target': 'proxy', 'dice': '0.8235244341022163', 'precision': '0.7855144210073182', 'recall': '0.8654000059282094', 'centerline_precision': '0.894583112055231', 'centerline_recall': '0.9190202114844064', 'cldice': '0.906637024933097', 'foreground_fraction': '0.14178466796875', 'thin_clean_centerline_recall': '0.8870056497175142'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '15', 'fold': '3', 'target': 'clean', 'dice': '0.6943804403486713', 'precision': '0.544218710810133', 'recall': '0.958985238221515', 'centerline_precision': '0.7779910428662828', 'centerline_recall': '0.9456447886941161', 'cldice': '0.8536642855366641', 'foreground_fraction': '0.15209197998046875', 'thin_clean_centerline_recall': '0.9128831528001844'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '15', 'fold': '3', 'target': 'proxy', 'dice': '0.78267733681776', 'precision': '0.7211938801103587', 'recall': '0.8556210200559424', 'centerline_precision': '0.8502879078694817', 'centerline_recall': '0.9019370460048426', 'cldice': '0.8753512637538241', 'foreground_fraction': '0.15209197998046875', 'thin_clean_centerline_recall': '0.9128831528001844'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '16', 'fold': '3', 'target': 'clean', 'dice': '0.7137517630470491', 'precision': '0.5839824591771969', 'recall': '0.9176715930728081', 'centerline_precision': '0.7155312546179992', 'centerline_recall': '0.8257407649131967', 'cldice': '0.7666957136966399', 'foreground_fraction': '0.13222503662109375', 'thin_clean_centerline_recall': '0.7227763923524522'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '16', 'fold': '3', 'target': 'proxy', 'dice': '0.8003095749188821', 'precision': '0.790577577750851', 'recall': '0.810284159791833', 'centerline_precision': '0.8921235407122802', 'centerline_recall': '0.7930343272362816', 'cldice': '0.8396656543302661', 'foreground_fraction': '0.13222503662109375', 'thin_clean_centerline_recall': '0.7227763923524522'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '17', 'fold': '1', 'target': 'clean', 'dice': '0.7230433322141031', 'precision': '0.600050178412132', 'recall': '0.9094558053067433', 'centerline_precision': '0.763256061817213', 'centerline_recall': '0.8429672447013488', 'cldice': '0.8011337611906619', 'foreground_fraction': '0.1368408203125', 'thin_clean_centerline_recall': '0.7513348588863463'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '17', 'fold': '1', 'target': 'proxy', 'dice': '0.8039324749008483', 'precision': '0.8058374219446922', 'recall': '0.8020365129571057', 'centerline_precision': '0.9092725819344525', 'centerline_recall': '0.8123180812667365', 'cldice': '0.8580652473242756', 'foreground_fraction': '0.1368408203125', 'thin_clean_centerline_recall': '0.7513348588863463'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '18', 'fold': '3', 'target': 'clean', 'dice': '0.6981223227806079', 'precision': '0.5469179723967401', 'recall': '0.9648787002003116', 'centerline_precision': '0.7699296225207933', 'centerline_recall': '0.9524579124579124', 'cldice': '0.8515221413451877', 'foreground_fraction': '0.15118789672851562', 'thin_clean_centerline_recall': '0.9210942799005201'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '18', 'fold': '3', 'target': 'proxy', 'dice': '0.8050321330625264', 'precision': '0.7443292205989958', 'recall': '0.8765153315902068', 'centerline_precision': '0.8830454254638516', 'centerline_recall': '0.9281760674749605', 'cldice': '0.9050484809330607', 'foreground_fraction': '0.15118789672851562', 'thin_clean_centerline_recall': '0.9210942799005201'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '19', 'fold': '3', 'target': 'clean', 'dice': '0.7783383057593738', 'precision': '0.712540554856651', 'recall': '0.8575242041515598', 'centerline_precision': '0.8273012725169393', 'centerline_recall': '0.7797634691195795', 'cldice': '0.8028292744546541', 'foreground_fraction': '0.11522674560546875', 'thin_clean_centerline_recall': '0.590622636753214'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '19', 'fold': '3', 'target': 'proxy', 'dice': '0.7961607678467364', 'precision': '0.8787658081175925', 'recall': '0.727751274880737', 'centerline_precision': '0.9436456784002644', 'centerline_recall': '0.7553395574881698', 'cldice': '0.8390572137912146', 'foreground_fraction': '0.11522674560546875', 'thin_clean_centerline_recall': '0.590622636753214'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '20', 'fold': '1', 'target': 'clean', 'dice': '0.7665348757752615', 'precision': '0.659038133340794', 'recall': '0.9159343650361614', 'centerline_precision': '0.7913955649980082', 'centerline_recall': '0.8617275113338105', 'cldice': '0.8250654055590262', 'foreground_fraction': '0.13634872436523438', 'thin_clean_centerline_recall': '0.7702891326021934'}, {'experiment': 'L2_soft_d4_t0.1', 'id': '20', 'fold': '1', 'target': 'proxy', 'dice': '0.825157232704639', 'precision': '0.8534258456201215', 'recall': '0.7987012987012987', 'centerline_precision': '0.9260390386402868', 'centerline_recall': '0.8368513119533527', 'cldice': '0.8791890932356454', 'foreground_fraction': '0.13634872436523438', 'thin_clean_centerline_recall': '0.7702891326021934'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '1', 'fold': '2', 'target': 'clean', 'dice': '0.8082730668365885', 'precision': '0.7406422953684824', 'recall': '0.889496239286339', 'centerline_precision': '0.8720211827007943', 'centerline_recall': '0.872684545203207', 'cldice': '0.8723527378419632', 'foreground_fraction': '0.104766845703125', 'thin_clean_centerline_recall': '0.7815533980582524'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '1', 'fold': '2', 'target': 'proxy', 'dice': '0.7878441863503396', 'precision': '0.8792965336440431', 'recall': '0.7136229314420803', 'centerline_precision': '0.9345395704619005', 'centerline_recall': '0.7981701488461014', 'cldice': '0.8609885195958248', 'foreground_fraction': '0.104766845703125', 'thin_clean_centerline_recall': '0.7815533980582524'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '2', 'fold': '0', 'target': 'clean', 'dice': '0.7977381662363932', 'precision': '0.7703267639279925', 'recall': '0.8271723695662132', 'centerline_precision': '0.8154848046309696', 'centerline_recall': '0.7884497549019608', 'cldice': '0.8017394357096407', 'foreground_fraction': '0.08942413330078125', 'thin_clean_centerline_recall': '0.6114982578397212'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '2', 'fold': '0', 'target': 'proxy', 'dice': '0.7755397506202372', 'precision': '0.9140431703779541', 'recall': '0.6734873487348735', 'centerline_precision': '0.9493487698986975', 'centerline_recall': '0.7574191643289532', 'cldice': '0.8425925253614271', 'foreground_fraction': '0.08942413330078125', 'thin_clean_centerline_recall': '0.6114982578397212'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '3', 'fold': '1', 'target': 'clean', 'dice': '0.8243163417772954', 'precision': '0.8163735847758009', 'recall': '0.8324151724621917', 'centerline_precision': '0.9241897343163166', 'centerline_recall': '0.8184772516248839', 'cldice': '0.8681271635092227', 'foreground_fraction': '0.11085128784179688', 'thin_clean_centerline_recall': '0.6881562368752625'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '3', 'fold': '1', 'target': 'proxy', 'dice': '0.7661554221327631', 'precision': '0.9265631990089129', 'recall': '0.6530914206709195', 'centerline_precision': '0.9744053414939491', 'centerline_recall': '0.7692131222757513', 'cldice': '0.8597355335091572', 'foreground_fraction': '0.11085128784179688', 'thin_clean_centerline_recall': '0.6881562368752625'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '4', 'fold': '2', 'target': 'clean', 'dice': '0.7627648191744858', 'precision': '0.6613968678096934', 'recall': '0.9008289526890416', 'centerline_precision': '0.8294802172226532', 'centerline_recall': '0.8697136073601425', 'cldice': '0.8491205906209278', 'foreground_fraction': '0.10279083251953125', 'thin_clean_centerline_recall': '0.7938021454112039'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '4', 'fold': '2', 'target': 'proxy', 'dice': '0.7634321810848159', 'precision': '0.8073183403844727', 'recall': '0.7240713620023965', 'centerline_precision': '0.9070597362296354', 'centerline_recall': '0.7692865429234339', 'cldice': '0.8325115847325744', 'foreground_fraction': '0.10279083251953125', 'thin_clean_centerline_recall': '0.7938021454112039'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '5', 'fold': '2', 'target': 'clean', 'dice': '0.7652428261642916', 'precision': '0.6814568644181973', 'recall': '0.8725202691047094', 'centerline_precision': '0.6975082767032584', 'centerline_recall': '0.8828125', 'cldice': '0.7792962474513111', 'foreground_fraction': '0.08494186401367188', 'thin_clean_centerline_recall': '0.7465690759377859'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '5', 'fold': '2', 'target': 'proxy', 'dice': '0.7342296201456803', 'precision': '0.7725782548165446', 'recall': '0.6995079900784776', 'centerline_precision': '0.7377591914967765', 'centerline_recall': '0.8144879950749025', 'cldice': '0.7742272106254421', 'foreground_fraction': '0.08494186401367188', 'thin_clean_centerline_recall': '0.7465690759377859'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '6', 'fold': '0', 'target': 'clean', 'dice': '0.8223226477667236', 'precision': '0.8171421186960877', 'recall': '0.8275692831201126', 'centerline_precision': '0.8968299711815562', 'centerline_recall': '0.7322234156820623', 'cldice': '0.8062104165277191', 'foreground_fraction': '0.11807632446289062', 'thin_clean_centerline_recall': '0.4991826726604005'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '6', 'fold': '0', 'target': 'proxy', 'dice': '0.7613289036548022', 'precision': '0.9254353374470972', 'recall': '0.6466577872090661', 'centerline_precision': '0.9695965417867435', 'centerline_recall': '0.6625428838756627', 'cldice': '0.7871867793776746', 'foreground_fraction': '0.11807632446289062', 'thin_clean_centerline_recall': '0.4991826726604005'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '7', 'fold': '4', 'target': 'clean', 'dice': '0.760718190712549', 'precision': '0.6997141647258098', 'recall': '0.8333753625015761', 'centerline_precision': '0.8472255017709563', 'centerline_recall': '0.7475503295920185', 'cldice': '0.7942730139650469', 'foreground_fraction': '0.0720672607421875', 'thin_clean_centerline_recall': '0.5983466362599772'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '7', 'fold': '4', 'target': 'proxy', 'dice': '0.7463293948998906', 'precision': '0.8542769426212153', 'recall': '0.6626021266986903', 'centerline_precision': '0.9544273907910271', 'centerline_recall': '0.6933962264150944', 'cldice': '0.8032368807576333', 'foreground_fraction': '0.0720672607421875', 'thin_clean_centerline_recall': '0.5983466362599772'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '8', 'fold': '4', 'target': 'clean', 'dice': '0.7612725121603758', 'precision': '0.6923444785398318', 'recall': '0.8454426006131686', 'centerline_precision': '0.8035410282601294', 'centerline_recall': '0.7863904513650707', 'cldice': '0.7948732382514943', 'foreground_fraction': '0.09572219848632812', 'thin_clean_centerline_recall': '0.6610281195079086'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '8', 'fold': '4', 'target': 'proxy', 'dice': '0.7692416027044412', 'precision': '0.8706810664328697', 'recall': '0.6889722809119864', 'centerline_precision': '0.9458631256384066', 'centerline_recall': '0.7215743440233237', 'cldice': '0.818634073944426', 'foreground_fraction': '0.09572219848632812', 'thin_clean_centerline_recall': '0.6610281195079086'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '9', 'fold': '0', 'target': 'clean', 'dice': '0.7929759304517788', 'precision': '0.7335067013402681', 'recall': '0.8629389061470395', 'centerline_precision': '0.8447563996696945', 'centerline_recall': '0.7940935192780968', 'cldice': '0.8186418714620228', 'foreground_fraction': '0.09534835815429688', 'thin_clean_centerline_recall': '0.6671111111111111'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '9', 'fold': '0', 'target': 'proxy', 'dice': '0.7772566588502725', 'precision': '0.8884576915383077', 'recall': '0.6907954085917815', 'centerline_precision': '0.9580511973575557', 'centerline_recall': '0.7361719060883406', 'cldice': '0.8325826446994283', 'foreground_fraction': '0.09534835815429688', 'thin_clean_centerline_recall': '0.6671111111111111'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '10', 'fold': '0', 'target': 'clean', 'dice': '0.7811473023008837', 'precision': '0.7016665421119498', 'recall': '0.8809345092887971', 'centerline_precision': '0.7945499587118084', 'centerline_recall': '0.8037109375', 'cldice': '0.7991041934648263', 'foreground_fraction': '0.10208892822265625', 'thin_clean_centerline_recall': '0.6593673965936739'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '10', 'fold': '0', 'target': 'proxy', 'dice': '0.794338983051196', 'precision': '0.8756072042448247', 'recall': '0.7268751163223525', 'centerline_precision': '0.9430222956234517', 'centerline_recall': '0.7445841133993046', 'cldice': '0.8321364699120394', 'foreground_fraction': '0.10208892822265625', 'thin_clean_centerline_recall': '0.6593673965936739'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '11', 'fold': '2', 'target': 'clean', 'dice': '0.796088195052383', 'precision': '0.7107212944050623', 'recall': '0.9047619047619048', 'centerline_precision': '0.7984955185659411', 'centerline_recall': '0.877895680765207', 'cldice': '0.8363152552209763', 'foreground_fraction': '0.09524917602539062', 'thin_clean_centerline_recall': '0.7997533908754624'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '11', 'fold': '2', 'target': 'proxy', 'dice': '0.8079608938551015', 'precision': '0.880411710521046', 'recall': '0.7465276598634836', 'centerline_precision': '0.9406209987195903', 'centerline_recall': '0.8090935587292335', 'cldice': '0.8699137673965869', 'foreground_fraction': '0.09524917602539062', 'thin_clean_centerline_recall': '0.7997533908754624'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '12', 'fold': '4', 'target': 'clean', 'dice': '0.6525704140108697', 'precision': '0.6785549448244032', 'recall': '0.6285025844312309', 'centerline_precision': '0.8137709596860506', 'centerline_recall': '0.6370582617000955', 'cldice': '0.7146527039265275', 'foreground_fraction': '0.09091567993164062', 'thin_clean_centerline_recall': '0.5577358490566038'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '12', 'fold': '4', 'target': 'proxy', 'dice': '0.6367554030545969', 'precision': '0.8214660344899929', 'recall': '0.519861922464153', 'centerline_precision': '0.9001070281840885', 'centerline_recall': '0.5744553507301892', 'cldice': '0.7013217019017116', 'foreground_fraction': '0.09091567993164062', 'thin_clean_centerline_recall': '0.5577358490566038'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '13', 'fold': '4', 'target': 'clean', 'dice': '0.7926513467777406', 'precision': '0.751682264676309', 'recall': '0.8383437567392711', 'centerline_precision': '0.8474094040131775', 'centerline_recall': '0.7805313952102942', 'cldice': '0.8125966801056053', 'foreground_fraction': '0.09864044189453125', 'thin_clean_centerline_recall': '0.660303928836175'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '13', 'fold': '4', 'target': 'proxy', 'dice': '0.7587584661115842', 'precision': '0.9033181220512028', 'recall': '0.6540841757441684', 'centerline_precision': '0.9646600778676251', 'centerline_recall': '0.7023282887077997', 'cldice': '0.8128527772096451', 'foreground_fraction': '0.09864044189453125', 'thin_clean_centerline_recall': '0.660303928836175'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '14', 'fold': '1', 'target': 'clean', 'dice': '0.8009964241235064', 'precision': '0.7342909760589319', 'recall': '0.88103234930175', 'centerline_precision': '0.890018484288355', 'centerline_recall': '0.8587059300283669', 'cldice': '0.8740818668010532', 'foreground_fraction': '0.10356903076171875', 'thin_clean_centerline_recall': '0.7642937853107344'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '14', 'fold': '1', 'target': 'proxy', 'dice': '0.7933384794787174', 'precision': '0.8895764272559853', 'recall': '0.7158905652547648', 'centerline_precision': '0.9653419593345656', 'centerline_recall': '0.8199705528041762', 'cldice': '0.8867377276064444', 'foreground_fraction': '0.10356903076171875', 'thin_clean_centerline_recall': '0.7642937853107344'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '15', 'fold': '3', 'target': 'clean', 'dice': '0.7668196746504181', 'precision': '0.6780680004075949', 'recall': '0.882303544594714', 'centerline_precision': '0.8645758057522763', 'centerline_recall': '0.8604429949721429', 'cldice': '0.8625044496553138', 'foreground_fraction': '0.11230850219726562', 'thin_clean_centerline_recall': '0.7750633786586771'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '15', 'fold': '3', 'target': 'proxy', 'dice': '0.7635573461072477', 'precision': '0.8175673380659624', 'recall': '0.7162411474141522', 'centerline_precision': '0.9254227489521607', 'centerline_recall': '0.7668818940005381', 'cldice': '0.8387259982095954', 'foreground_fraction': '0.11230850219726562', 'thin_clean_centerline_recall': '0.7750633786586771'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '16', 'fold': '3', 'target': 'clean', 'dice': '0.7778174261670036', 'precision': '0.7242309346050111', 'recall': '0.8399673587813945', 'centerline_precision': '0.789553752535497', 'centerline_recall': '0.7390680067876256', 'cldice': '0.7634771840438673', 'foreground_fraction': '0.09759140014648438', 'thin_clean_centerline_recall': '0.5860349127182045'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '16', 'fold': '3', 'target': 'proxy', 'dice': '0.7685263122457686', 'precision': '0.8922331235586132', 'recall': '0.6749460362518112', 'centerline_precision': '0.9523326572008114', 'centerline_recall': '0.6817840140315711', 'cldice': '0.794661964044535', 'foreground_fraction': '0.09759140014648438', 'thin_clean_centerline_recall': '0.5860349127182045'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '17', 'fold': '1', 'target': 'clean', 'dice': '0.7811895895655919', 'precision': '0.7534838076545632', 'recall': '0.8110106472874767', 'centerline_precision': '0.827538247566064', 'centerline_recall': '0.7396435452793835', 'cldice': '0.7811261285427838', 'foreground_fraction': '0.09717941284179688', 'thin_clean_centerline_recall': '0.5901983218916858'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '17', 'fold': '1', 'target': 'proxy', 'dice': '0.7539704471938586', 'precision': '0.9103434739941119', 'recall': '0.6434437600577104', 'centerline_precision': '0.9596662030598053', 'centerline_recall': '0.6951915240423798', 'cldice': '0.8062950661569643', 'foreground_fraction': '0.09717941284179688', 'thin_clean_centerline_recall': '0.5901983218916858'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '18', 'fold': '3', 'target': 'clean', 'dice': '0.7778147184796945', 'precision': '0.6936654109111204', 'recall': '0.8851991987536167', 'centerline_precision': '0.8408795023868074', 'centerline_recall': '0.8725925925925926', 'cldice': '0.8564425731771194', 'foreground_fraction': '0.1093597412109375', 'thin_clean_centerline_recall': '0.7935790187655437'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '18', 'fold': '3', 'target': 'proxy', 'dice': '0.7807907066302214', 'precision': '0.8487163387749407', 'recall': '0.7229320180651295', 'centerline_precision': '0.943729205844062', 'centerline_recall': '0.8031101739588824', 'cldice': '0.8677598357795221', 'foreground_fraction': '0.1093597412109375', 'thin_clean_centerline_recall': '0.7935790187655437'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '19', 'fold': '3', 'target': 'clean', 'dice': '0.7830011121571927', 'precision': '0.8452232534515399', 'recall': '0.7293119247778796', 'centerline_precision': '0.8891752577319587', 'centerline_recall': '0.688042049934297', 'cldice': '0.7757839888100068', 'foreground_fraction': '0.08261489868164062', 'thin_clean_centerline_recall': '0.43609780690698263'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '19', 'fold': '3', 'target': 'proxy', 'dice': '0.7095009547406556', 'precision': '0.9522094472918686', 'recall': '0.5653890442506991', 'centerline_precision': '0.9795655375552282', 'centerline_recall': '0.6461184294666837', 'cldice': '0.778645000533909', 'foreground_fraction': '0.08261489868164062', 'thin_clean_centerline_recall': '0.43609780690698263'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '20', 'fold': '1', 'target': 'clean', 'dice': '0.8082627524627136', 'precision': '0.7968112437660572', 'recall': '0.8200482152577961', 'centerline_precision': '0.8548044862079418', 'centerline_recall': '0.7817943211644', 'cldice': '0.8166708786696327', 'foreground_fraction': '0.1009674072265625', 'thin_clean_centerline_recall': '0.6400797607178464'}, {'experiment': 'L2_soft_d4_t0.4', 'id': '20', 'fold': '1', 'target': 'proxy', 'dice': '0.7671821837306415', 'precision': '0.937093849176364', 'recall': '0.6494291998324256', 'centerline_precision': '0.9777205213701121', 'centerline_recall': '0.7332944606413995', 'cldice': '0.8380488188750419', 'foreground_fraction': '0.1009674072265625', 'thin_clean_centerline_recall': '0.6400797607178464'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '1', 'fold': '2', 'target': 'clean', 'dice': '0.8149572914731548', 'precision': '0.77314287956677', 'recall': '0.8615532622004548', 'centerline_precision': '0.8867407520170498', 'centerline_recall': '0.8472491014653027', 'cldice': '0.8665452152102324', 'foreground_fraction': '0.09720993041992188', 'thin_clean_centerline_recall': '0.7393203883495145'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '1', 'fold': '2', 'target': 'proxy', 'dice': '0.7712354398803647', 'precision': '0.8976965035513872', 'recall': '0.6760047281323877', 'centerline_precision': '0.9450449079007459', 'centerline_recall': '0.7641676908370886', 'cldice': '0.8450356445314638', 'foreground_fraction': '0.09720993041992188', 'thin_clean_centerline_recall': '0.7393203883495145'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '2', 'fold': '0', 'target': 'clean', 'dice': '0.7971998986810114', 'precision': '0.8015373217262456', 'recall': '0.7929091658650543', 'centerline_precision': '0.8348247576435496', 'centerline_recall': '0.7594975490196079', 'cldice': '0.795381654815074', 'foreground_fraction': '0.0823822021484375', 'thin_clean_centerline_recall': '0.5609756097560976'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '2', 'fold': '0', 'target': 'proxy', 'dice': '0.7514931381180815', 'precision': '0.9292924615669569', 'recall': '0.6308030803080308', 'centerline_precision': '0.9574944071588367', 'centerline_recall': '0.722574929868596', 'cldice': '0.8236105961275245', 'foreground_fraction': '0.0823822021484375', 'thin_clean_centerline_recall': '0.5609756097560976'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '3', 'fold': '1', 'target': 'clean', 'dice': '0.8219271979511853', 'precision': '0.845725528044842', 'recall': '0.7994315590020703', 'centerline_precision': '0.9314481576692374', 'centerline_recall': '0.7930594243268339', 'cldice': '0.8567010634495873', 'foreground_fraction': '0.10276412963867188', 'thin_clean_centerline_recall': '0.6442671146577068'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '3', 'fold': '1', 'target': 'proxy', 'dice': '0.7429216911660609', 'precision': '0.9399383793013846', 'recall': '0.6141848788415359', 'centerline_precision': '0.9787203656098258', 'centerline_recall': '0.738701537049782', 'cldice': '0.8419389985632408', 'foreground_fraction': '0.10276412963867188', 'thin_clean_centerline_recall': '0.6442671146577068'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '4', 'fold': '2', 'target': 'clean', 'dice': '0.7725384803327798', 'precision': '0.6890526566028765', 'recall': '0.8790436716538617', 'centerline_precision': '0.8405015076971909', 'centerline_recall': '0.8477518919721027', 'cldice': '0.8441111310603904', 'foreground_fraction': '0.09627914428710938', 'thin_clean_centerline_recall': '0.7594755661501787'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '4', 'fold': '2', 'target': 'proxy', 'dice': '0.7531067416751025', 'precision': '0.8247949601806728', 'recall': '0.6928837704699774', 'centerline_precision': '0.9154102523408982', 'centerline_recall': '0.7392691415313225', 'cldice': '0.8179645602685091', 'foreground_fraction': '0.09627914428710938', 'thin_clean_centerline_recall': '0.7594755661501787'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '5', 'fold': '2', 'target': 'clean', 'dice': '0.7784166533573487', 'precision': '0.7247087980173482', 'recall': '0.8407222126387212', 'centerline_precision': '0.7346560356017059', 'centerline_recall': '0.8587582236842105', 'cldice': '0.7918743145112023', 'foreground_fraction': '0.07696151733398438', 'thin_clean_centerline_recall': '0.6980786825251601'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '5', 'fold': '2', 'target': 'proxy', 'dice': '0.7248927805581556', 'precision': '0.8042627013630731', 'recall': '0.659781238563819', 'centerline_precision': '0.7691451882069349', 'centerline_recall': '0.7839113482454341', 'cldice': '0.7764580713347112', 'foreground_fraction': '0.07696151733398438', 'thin_clean_centerline_recall': '0.6980786825251601'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '6', 'fold': '0', 'target': 'clean', 'dice': '0.8203034035878228', 'precision': '0.8469579761655864', 'recall': '0.7952753329188889', 'centerline_precision': '0.9138755980861244', 'centerline_recall': '0.7022556390977444', 'cldice': '0.7942106153558486', 'foreground_fraction': '0.10947418212890625', 'thin_clean_centerline_recall': '0.44871270944013075'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '6', 'fold': '0', 'target': 'proxy', 'dice': '0.7385711350096054', 'precision': '0.9392989058470974', 'recall': '0.6085287942750074', 'centerline_precision': '0.9763755980861244', 'centerline_recall': '0.6297951970059258', 'cldice': '0.7656927445417637', 'foreground_fraction': '0.10947418212890625', 'thin_clean_centerline_recall': '0.44871270944013075'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '7', 'fold': '4', 'target': 'clean', 'dice': '0.7642169467207911', 'precision': '0.7267970882620565', 'recall': '0.8056991552137183', 'centerline_precision': '0.8631578947368421', 'centerline_recall': '0.7277748084803135', 'cldice': '0.7897060262323758', 'foreground_fraction': '0.06707763671875', 'thin_clean_centerline_recall': '0.5678449258836944'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '7', 'fold': '4', 'target': 'proxy', 'dice': '0.730502372380891', 'precision': '0.8711897179253867', 'recall': '0.6289362400952498', 'centerline_precision': '0.9649938800489596', 'centerline_recall': '0.6643955276030747', 'cldice': '0.7869667190150108', 'foreground_fraction': '0.06707763671875', 'thin_clean_centerline_recall': '0.5678449258836944'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '8', 'fold': '4', 'target': 'clean', 'dice': '0.7696838617648297', 'precision': '0.7299786147601798', 'recall': '0.8139568835466446', 'centerline_precision': '0.8086074177746676', 'centerline_recall': '0.760323775552202', 'cldice': '0.7837226354309146', 'foreground_fraction': '0.08740615844726562', 'thin_clean_centerline_recall': '0.6208260105448155'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '8', 'fold': '4', 'target': 'proxy', 'dice': '0.7495606326893864', 'precision': '0.8934665910181993', 'recall': '0.645580397969159', 'centerline_precision': '0.9504898530440867', 'centerline_recall': '0.6881791677710045', 'cldice': '0.7983397595657288', 'foreground_fraction': '0.08740615844726562', 'thin_clean_centerline_recall': '0.6208260105448155'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '9', 'fold': '0', 'target': 'clean', 'dice': '0.7997390150070877', 'precision': '0.766044567044524', 'recall': '0.8365339357996799', 'centerline_precision': '0.8565314158534497', 'centerline_recall': '0.7687995624829095', 'cldice': '0.8102977012534657', 'foreground_fraction': '0.08850479125976562', 'thin_clean_centerline_recall': '0.6264444444444445'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '9', 'fold': '0', 'target': 'proxy', 'dice': '0.7584375225848117', 'precision': '0.9046592819275031', 'recall': '0.6529069586586618', 'centerline_precision': '0.963020030816641', 'centerline_recall': '0.7020825043109166', 'cldice': '0.8121055618782399', 'foreground_fraction': '0.08850479125976562', 'thin_clean_centerline_recall': '0.6264444444444445'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '10', 'fold': '0', 'target': 'clean', 'dice': '0.7910101644962376', 'precision': '0.7351057401812688', 'recall': '0.8561174704447363', 'centerline_precision': '0.8155487804878049', 'centerline_recall': '0.7769252232142857', 'cldice': '0.7957686176975228', 'foreground_fraction': '0.09469985961914062', 'thin_clean_centerline_recall': '0.6138686131386861'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '10', 'fold': '0', 'target': 'proxy', 'dice': '0.7786131118241617', 'precision': '0.8948640483383686', 'recall': '0.6890936162292947', 'centerline_precision': '0.9544376693766937', 'centerline_recall': '0.7115538914148168', 'cldice': '0.8152908498950848', 'foreground_fraction': '0.09469985961914062', 'thin_clean_centerline_recall': '0.6138686131386861'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '11', 'fold': '2', 'target': 'clean', 'dice': '0.806382182295044', 'precision': '0.7450929528750541', 'recall': '0.8786581013561742', 'centerline_precision': '0.8155435317265125', 'centerline_recall': '0.8535345987146914', 'cldice': '0.8341066944571918', 'foreground_fraction': '0.08823394775390625', 'thin_clean_centerline_recall': '0.7595561035758323'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '11', 'fold': '2', 'target': 'proxy', 'dice': '0.7907259828446075', 'precision': '0.8987029831387808', 'recall': '0.7059123170441811', 'centerline_precision': '0.9509755697655353', 'centerline_recall': '0.7752841737102886', 'cldice': '0.8541893091242899', 'foreground_fraction': '0.08823394775390625', 'thin_clean_centerline_recall': '0.7595561035758323'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '12', 'fold': '4', 'target': 'clean', 'dice': '0.6424625764291747', 'precision': '0.702152172911194', 'recall': '0.5921262290622207', 'centerline_precision': '0.8322653599698454', 'centerline_recall': '0.6027936962750716', 'cldice': '0.6991828112365924', 'foreground_fraction': '0.08277511596679688', 'thin_clean_centerline_recall': '0.5154716981132076'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '12', 'fold': '4', 'target': 'proxy', 'dice': '0.612847251470522', 'precision': '0.8382413936126089', 'recall': '0.48297928836962295', 'centerline_precision': '0.9165096117602713', 'centerline_recall': '0.536150347139095', 'cldice': '0.6765340277898136', 'foreground_fraction': '0.08277511596679688', 'thin_clean_centerline_recall': '0.5154716981132076'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '13', 'fold': '4', 'target': 'clean', 'dice': '0.7948076188864489', 'precision': '0.7819373148032219', 'recall': '0.8081086909639853', 'centerline_precision': '0.8616119586993374', 'centerline_recall': '0.7543190754199929', 'cldice': '0.804403557248469', 'foreground_fraction': '0.09140396118164062', 'thin_clean_centerline_recall': '0.6217568569310601'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '13', 'fold': '4', 'target': 'proxy', 'dice': '0.7365933771287428', 'precision': '0.917198781353032', 'recall': '0.6154126179608524', 'centerline_precision': '0.9676375404530745', 'centerline_recall': '0.6649592549476135', 'cldice': '0.7882405989913737', 'foreground_fraction': '0.09140396118164062', 'thin_clean_centerline_recall': '0.6217568569310601'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '14', 'fold': '1', 'target': 'clean', 'dice': '0.8083706900167406', 'precision': '0.7684189565909996', 'recall': '0.8527046137528725', 'centerline_precision': '0.8953525134366108', 'centerline_recall': '0.8347967040389032', 'cldice': '0.8640148718044415', 'foreground_fraction': '0.09578704833984375', 'thin_clean_centerline_recall': '0.7249717514124294'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '14', 'fold': '1', 'target': 'proxy', 'dice': '0.7749927778820076', 'precision': '0.9081242532855436', 'recall': '0.6759047929572872', 'centerline_precision': '0.9702813784381916', 'centerline_recall': '0.7878463391781555', 'cldice': '0.8695985215586889', 'foreground_fraction': '0.09578704833984375', 'thin_clean_centerline_recall': '0.7249717514124294'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '15', 'fold': '3', 'target': 'clean', 'dice': '0.7736447173071432', 'precision': '0.7084068195179306', 'recall': '0.8521170335012818', 'centerline_precision': '0.8785660941000747', 'centerline_recall': '0.8275580921320832', 'cldice': '0.8522995998914467', 'foreground_fraction': '0.10382080078125', 'thin_clean_centerline_recall': '0.7238995160175156'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '15', 'fold': '3', 'target': 'proxy', 'dice': '0.7483805202069179', 'precision': '0.8362360376249265', 'recall': '0.6772302565018151', 'centerline_precision': '0.9357729648991785', 'centerline_recall': '0.725181598062954', 'cldice': '0.8171269091184237', 'foreground_fraction': '0.10382080078125', 'thin_clean_centerline_recall': '0.7238995160175156'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '16', 'fold': '3', 'target': 'clean', 'dice': '0.7816137132879537', 'precision': '0.7539705944306357', 'recall': '0.8113609574757458', 'centerline_precision': '0.7976211299632674', 'centerline_recall': '0.7112648479310795', 'cldice': '0.7519718256002816', 'foreground_fraction': '0.09054946899414062', 'thin_clean_centerline_recall': '0.541978387364921'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '16', 'fold': '3', 'target': 'proxy', 'dice': '0.7500173743836437', 'precision': '0.9092977208577326', 'recall': '0.6382211183062776', 'centerline_precision': '0.9557460206401959', 'centerline_recall': '0.6553495364570283', 'cldice': '0.777542596822398', 'foreground_fraction': '0.09054946899414062', 'thin_clean_centerline_recall': '0.541978387364921'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '17', 'fold': '1', 'target': 'clean', 'dice': '0.7833538893382738', 'precision': '0.7885521020635329', 'recall': '0.7782237620415752', 'centerline_precision': '0.8464763747782615', 'centerline_recall': '0.7129094412331407', 'cldice': '0.7739726668846557', 'foreground_fraction': '0.08910369873046875', 'thin_clean_centerline_recall': '0.5482456140350878'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '17', 'fold': '1', 'target': 'proxy', 'dice': '0.7303703703708243', 'precision': '0.9286754002911208', 'recall': '0.6018533932634149', 'centerline_precision': '0.9687147234317046', 'centerline_recall': '0.6636395389451624', 'cldice': '0.7876689604026453', 'foreground_fraction': '0.08910369873046875', 'thin_clean_centerline_recall': '0.5482456140350878'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '18', 'fold': '3', 'target': 'clean', 'dice': '0.7873227004951304', 'precision': '0.7305348163949413', 'recall': '0.8536835076786111', 'centerline_precision': '0.8619341254323959', 'centerline_recall': '0.8416161616161616', 'cldice': '0.8516539790195452', 'foreground_fraction': '0.1001434326171875', 'thin_clean_centerline_recall': '0.7433868415102871'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '18', 'fold': '3', 'target': 'proxy', 'dice': '0.7624357347937464', 'precision': '0.8699527655035807', 'recall': '0.6785714285714286', 'centerline_precision': '0.9560836215972327', 'centerline_recall': '0.7602793885081708', 'cldice': '0.8470127437038886', 'foreground_fraction': '0.1001434326171875', 'thin_clean_centerline_recall': '0.7433868415102871'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '19', 'fold': '3', 'target': 'clean', 'dice': '0.7688575307712862', 'precision': '0.8710035300050428', 'recall': '0.6881549065699829', 'centerline_precision': '0.9037728658536586', 'centerline_recall': '0.6551905387647832', 'cldice': '0.7596630288375225', 'foreground_fraction': '0.07564544677734375', 'thin_clean_centerline_recall': '0.38492563650113437'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '19', 'fold': '3', 'target': 'proxy', 'dice': '0.6772875816999195', 'precision': '0.9615229450327786', 'recall': '0.5227559357350441', 'centerline_precision': '0.9830411585365854', 'centerline_recall': '0.6077503517073795', 'cldice': '0.7511274808749956', 'foreground_fraction': '0.07564544677734375', 'thin_clean_centerline_recall': '0.38492563650113437'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '20', 'fold': '1', 'target': 'clean', 'dice': '0.8042825018411489', 'precision': '0.823706843842987', 'recall': '0.7857531689867019', 'centerline_precision': '0.860049704877291', 'centerline_recall': '0.7568599379623002', 'cldice': '0.8051620808380553', 'foreground_fraction': '0.09358596801757812', 'thin_clean_centerline_recall': '0.6001994017946162'}, {'experiment': 'L2_soft_d4_t0.5', 'id': '20', 'fold': '1', 'target': 'proxy', 'dice': '0.7426703866086207', 'precision': '0.9494150735743692', 'recall': '0.6098659405111018', 'centerline_precision': '0.9793414103758932', 'centerline_recall': '0.6995918367346938', 'cldice': '0.816160210364919', 'foreground_fraction': '0.09358596801757812', 'thin_clean_centerline_recall': '0.6001994017946162'}]
for name,source in SOURCES.items():
    (RUNTIME_DIR/name).write_text(source,encoding='utf-8',newline='\n')
def load_module(name):
    spec=importlib.util.spec_from_file_location(name,RUNTIME_DIR/(name+'.py'))
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
    return module
BASE=load_module('task4_pipeline')
V2=load_module('task4_kaggle_v2')
ABL=load_module('task4_controlled_ablation')
STRUCT=load_module('task4_cldice_ablation')
print('Embedded sources loaded:',RUNTIME_DIR)


## 3. 数据与已有L2结果检查


In [ ]:
import torch
CFG = STRUCT.StructureConfig()
ROOT = BASE.find_data(DATA_DIR)
AUDIT_DIR = RUNTIME_DIR/'preflight_audit'
TRAIN, TEST, LABELS, SPLIT, IDS, AUDIT = V2.audit_v2(ROOT,AUDIT_DIR,CFG)
if AUDIT['data_fingerprint'] != EXPECTED_CONFIG['data_fingerprint']:
    raise ValueError('Dataset differs from the returned label ablation.')
BASELINE_RESOLVED = STRUCT.materialize_reference(BASELINE_RUN,RUNTIME_DIR/'preflight_reference')
STRUCT.verify_reference(BASELINE_RESOLVED,EXPECTED_L2,EXPECTED_CONFIG,SPLIT)
print('Data:',ROOT)
print('Read-only L2 baseline:',BASELINE_RESOLVED)
print('New output:',OUTPUT_DIR)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Train only: T1_L2_cldice; fresh ImageNet initialization; 5 folds x 80 epochs')
print('Loss: original L2 + 0.1 * soft-clDice(main logits, clean C)')
print('Soft skeleton: 10 iterations; smooth=1; per-image mean; float32')
if RUN_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('Enable GPU. This notebook does not launch the formal experiment on CPU.')


## 4. 新增组训练与配对汇总

同一会话中断后保持输出目录和配置再次运行即可续接。新会话把本轮完整输出另行挂载，填写`RESUME_FROM`，使用新的空输出目录；还需挂载原L2对照输出。轻量分析ZIP不能恢复训练。完整五折后才生成完整20图汇总，部分折不会冒充完整OOF。


In [ ]:
RESULT_DIR = None
if RUN_TRAINING:
    RESULT_DIR = STRUCT.run_structure(
        data_dir=ROOT,output_dir=OUTPUT_DIR,cfg=CFG,baseline_run=BASELINE_RESOLVED,
        expected=EXPECTED_L2,expected_config=EXPECTED_CONFIG,expected_rows=EXPECTED_L2_ROWS,
        folds_to_run=tuple(FOLDS_TO_RUN),weights_path=WEIGHTS_PATH,resume_from=RESUME_FROM)
else:
    print('Preflight only; no training or competition submission.')


## 5. 查看与下载


In [ ]:
from IPython.display import display, Image as DisplayImage, FileLink
import pandas as pd
if RESULT_DIR is not None:
    for name in ['structure_summary.csv','structure_paired_summary.csv','structure_calibration_development.csv']:
        if (RESULT_DIR/name).exists():
            print(name)
            display(pd.read_csv(RESULT_DIR/name))
    preview=RESULT_DIR/'oof_error_previews/oof_12.png'
    if preview.exists():display(DisplayImage(filename=str(preview)))
    display(FileLink(str(RESULT_DIR/'structure_ablation_analysis_results.zip')))
    print('Preserve the entire output with Kaggle Save Version for checkpoints/resume:',RESULT_DIR)


## 运行后

下载`structure_ablation_analysis_results.zip`供分析，另用Save Version保留完整输出。不要重复提交阈值变体。

优先核对：固定0.5的干净/代理Dice是否牺牲、最差折是否退化、clDice/细中心线收益能否在多数折重复、是否新增伪分支。仅clDice提高不足以证明比赛方案变好；若像素Dice下降而结构指标提高，应报告取舍。仍没有独立私榜或盲测结论。
